# Street Fighter II — Student Guide
**SYE6308 | Train, evaluate, and test your own reinforcement-learning agent**

Start with **PvE**: your agent controls P1 and plays against the game's built-in CPU.
After that, you can run a match between two saved agents or use the optional PvP training section.

| Your goal | Sections to run | Does the model learn? |
|---|---|---|
| First PvE experiment | 1 → 2 → 3 → 4 → 5 → 6 | Only Section 5 |
| Evaluate an existing agent against the CPU | 1 → 2 → 6; enter its package path | No |
| Watch two saved agents compete | 1 → 8; enter both agent paths | No |
| Train against a fixed saved opponent | 1 → 9.1 → all of 9.2 → 9.3 → 9.4 → 9.5 | Only Section 9.4 |
| Package a model for the practice arena | 1 → 7 (or 7.1) → 7.2 | No |

**For your first attempt, follow the first row. Do not enable every `RUN_*` switch.**
A switch set to `False` skips that optional task; it is not an error. Set it to `True` only when you are ready to run that task.
`TRUST_*` switches are separate: enable them only for your own or reviewed agent packages.

**Course rules:** the bundled arena rules are a practice draft. Your instructor must confirm the allowed characters, decision interval, submission format and grading rules. A practice ZIP is not automatically the official course deliverable.
No game ROM or pretrained game policy is included.


## 1. Shared setup — required for every route
**Run the next three code cells in order.** They create the project folders, write the shared framework, and prepare the dependencies/game resource. They do not train a model.

In Kaggle, make an editable copy, start a session, and select an available GPU under **Session options / Accelerator** for training. Enable Internet when installing missing dependencies. Attach the matching course-authorized game resource; `ROM_FILE` is a ROM path, not a `.state` file. The installer can scan `/kaggle/input`; the optional legacy external source is not guaranteed to remain available.

**Normally edit:** `ROM_FILE` when needed. **Leave unchanged:** `ROOT`, the shared framework cell, and `COMMON_RULES`, unless your instructor changes the published protocol. `speed_stars` controls game speed, not CPU difficulty.

**Expected output:** `Setup finished`, dependency versions, and the actual training device. CPU fallback is possible, but game workers remain on CPU even with GPU training. This code uses one learner GPU; selecting a two-GPU session does not enable multi-GPU training.

**Updating an existing RC3 project:** R2 changes `setup_sf2.py`; the R1 model-picker callback fix in `sf2_submission.py` is retained. Setup may report these files as different. Stop active tasks and preserve your files first; then explicitly enable `REPLACE_CHANGED_PUBLIC_FILES` for the update. The installer backs up changed public files. Run the relevant environment check again before training. The eight student-source cells are unchanged.

**R2 installation fix:** if the emulator is missing, Python 3.13/3.14 now selects
`stable-retro==1.0.1`; the earlier Python 3.8–3.12 installation branch retains
`stable-retro==0.9.5`. An already installed emulator is not automatically upgraded.
The package installer uses this notebook's Python and requires an emulator wheel;
it does not downgrade Python or change the system interpreter.
The printed versions identify the actual environment. Installation success does
not replace the real-game check. After any backend/runtime change, rerun the
relevant environment/self-check and keep new results separate from older runs.

**R3 runtime fix:** Setup and game launch resolve both the legacy game ID and its `-v0` integration name, with the same required ROM SHA1. This does not change characters, models, rewards or practice rules. A successful ROM lookup is not a completed game test. Re-run the environment/self-check before training or matches.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, importlib.util

ROOT = Path('/kaggle/working/sf2_classroom_rc3')
CODE_DIR = ROOT / 'code'
PVE_STUDENT_DIR = ROOT / 'student' / 'pve'
PVP_STUDENT_DIR = ROOT / 'student' / 'pvp'
for directory in (ROOT, CODE_DIR, PVE_STUDENT_DIR, PVP_STUDENT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

ROM_FILE = ''  # Exact matching ROM path; not a .state file.
ALLOW_ORIGINAL_ROM_SOURCE = True  # Optional fallback inherited from the supplied course.
REPLACE_CHANGED_PUBLIC_FILES = False  # For an R1/R2 -> R3 update: back up/stop tasks first, then enable once.

# Shared arena rules, intentionally identical for all participants.
# This is game speed, NOT CPU difficulty. Do not change rules when using incompatible packages.
COMMON_RULES = {
    'ruleset': 'hyper', 'speed_stars': 2,
    'decision_frames': 6, 'position_scale': 1024.0,
    'max_round_frames': 7200,
}
print('Current Python:', sys.version.split()[0], '|', sys.executable)
print('Project:', ROOT)
print('Student files:', PVE_STUDENT_DIR, '|', PVP_STUDENT_DIR)
print('Shared arena rules:', COMMON_RULES)
print('Existing RC1/RC2 files and model packages will not be overwritten.')


In [ ]:
# Public core: shared by the training and practice-arena notebooks. Run unchanged.
CORE_FILES = {'input_contract_smoke.py': '"""Installed-SB3 shape/export test on synthetic images, NOT a game or training run."""\nfrom pathlib import Path\nimport argparse,json,tempfile\nimport numpy as np\n\ndef main():\n    import gymnasium as gym\n    import torch\n    from sf2_common import source_module\n    from sf2_bundle import make_spec,export_agent,LoadedAgent\n    from sf2_actions import library\n    from sf2_observation import Side,observation_space,image_shape,description\n    p=argparse.ArgumentParser();p.add_argument(\'--job\',required=True);args=p.parse_args()\n    job=json.loads(Path(args.job).read_text());source=Path(job[\'sources\'])\n    torch.set_num_threads(2)\n    actions=source_module(source/\'student_actions.py\',\'actions\')\n    spec=make_spec(job[\'agent\'],library(actions.CUSTOM_ACTIONS),source/\'student_observation.py\')\n    cfg=dict(job[\'train\'],n_steps=8,batch_size=8,n_epochs=1)\n    common=job[\'common\']\n    obs_space=observation_space(spec[\'use_position\'],common[\'position_scale\'],spec=spec)\n    act_space=gym.spaces.Discrete(len(spec[\'actions\']))\n    raw={\'p1_x\':200,\'p1_y\':192,\'p2_x\':390,\'p2_y\':180}\n    rgb=np.zeros((224,320,3),np.uint8)\n    x0,y0,x1,y1=spec[\'observation\'][\'config\'][\'crop_xyxy\']\n    rgb[y0:y1,x0:x1]=[120,160,200]\n    side=Side(spec,1,common[\'position_scale\']);side.reset(rgb)\n    observed=side.observe(raw)\n    if not obs_space.contains(observed):raise RuntimeError(\'Synthetic observation is outside declared space.\')\n    class ShapeOnly(gym.Env):\n        observation_space=obs_space\n        action_space=act_space\n    module=source_module(source/\'student_model.py\',\'model\')\n    model=module.build_model(ShapeOnly(),cfg,int(job[\'agent\'][\'seed\']),\'cpu\')\n    from stable_baselines3 import PPO\n    if not isinstance(model,PPO):raise TypeError(\'Expected ordinary stable_baselines3.PPO.\')\n    model.policy.set_training_mode(False)\n    def probs(m):\n        obs,_=m.policy.obs_to_tensor(observed)\n        with torch.no_grad():return m.policy.get_distribution(obs).distribution.probs.cpu().numpy()\n    before=probs(model)\n    with tempfile.TemporaryDirectory() as td:\n        package=export_agent(Path(td)/\'synthetic_shape_only.zip\',model,spec,source/\'student_model.py\',source/\'student_reward.py\',common,{\'scope\':\'Synthetic contract only; never trained\'})\n        loaded=LoadedAgent(package,trusted=True,device=\'cpu\')\n        try:\n            error=float(np.max(np.abs(before-probs(loaded.model))))\n            if error>1e-6:raise RuntimeError(\'CPU export/reload prediction mismatch.\')\n            index=loaded.act(side,raw,deterministic=True)\n            if not 0<=index<len(spec[\'actions\']):raise RuntimeError(\'Invalid action index.\')\n        finally:loaded.close()\n    info={\'status\':\'SYNTHETIC_INPUT_CONTRACT_PASSED\',\'scope\':\'Installed PPO on synthetic input; no emulator, no learning, no useful model saved\',\n          \'image_shape\':list(image_shape(spec)),\'source_sha256\':spec[\'observation\'][\'source_sha256\'],\n          \'parameter_count\':sum(p.numel() for p in model.policy.parameters()),\'reload_probability_error\':error,\n          \'input_definition\':description(spec)}\n    print(json.dumps(info,indent=2),flush=True)\n    Path(job[\'report\']).write_text(json.dumps(info,indent=2),encoding=\'utf-8\')\n\nif __name__==\'__main__\':main()\n', 'lab_notebook.py': '"""Notebook task launcher with separate PvE/PvP student sources and check gates.\n\nEach run snapshots only the selected profile\'s four student files. Game-engine\ncode is shared. No task reads the other profile\'s mutable files or parameters.\nExternal packages are copied once into the run, preventing mid-run replacement.\n"""\nfrom pathlib import Path\nimport copy, datetime, hashlib, json, os, shutil, signal, subprocess, sys\n\nSTUDENT_FILES = (\'student_observation.py\', \'student_actions.py\',\n                 \'student_reward.py\', \'student_model.py\')\n\n\ndef profile_directory(root, profile):\n    if profile not in (\'pve\', \'pvp\'):\n        raise ValueError(\'profile must be pve or pvp\')\n    return Path(root).resolve()/\'student\'/profile\n\n\ndef last_training(root, kind=\'pve\'):\n    if kind == \'frozen\':\n        kind = \'pvp\'  # compatibility name, not a shared latest-run pointer\n    if kind not in (\'pve\', \'pvp\'):\n        raise ValueError(\'Choose last_training(..., kind="pve" or "pvp").\')\n    p = Path(root)/(\'last_training_\'+kind+\'.json\')\n    if not p.is_file():\n        raise FileNotFoundError(f\'No completed {kind.upper()} training in this project. Supply an existing v3 package path explicitly.\')\n    return json.loads(p.read_text(encoding=\'utf-8\'))\n\n\ndef input_check_fingerprint(root, agent, common, pve=None, profile=\'pve\'):\n    root = Path(root).resolve()\n    source = profile_directory(root, profile)\n    missing = [str(source/n) for n in STUDENT_FILES if not (source/n).is_file()]\n    if missing:\n        raise FileNotFoundError(\'Run the four student editing cells for \'+profile+\': \'+\', \'.join(missing))\n    return {\n        \'profile\': profile,\n        \'source_hashes\': {n:hashlib.sha256((source/n).read_bytes()).hexdigest() for n in STUDENT_FILES},\n        \'framework_hashes\': {p.name:hashlib.sha256(p.read_bytes()).hexdigest()\n                             for p in sorted((root/\'code\').glob(\'*.py\'))},\n        \'agent\': copy.deepcopy(agent), \'common\':copy.deepcopy(common),\n        # There is deliberately no dependency on CPU opponent/difficulty for PvP.\n        \'pve\':copy.deepcopy(pve or {}) if profile == \'pve\' else None,\n    }\n\n\ndef assert_no_running_task(root):\n    script = str(Path(root).resolve()/\'code\'/\'run_lab.py\').encode()\n    procdir = Path(\'/proc\')\n    if not procdir.is_dir():\n        return\n    for p in procdir.iterdir():\n        if p.name.isdigit() and int(p.name) != os.getpid():\n            try:\n                args=(p/\'cmdline\').read_bytes().split(b\'\\0\')\n            except (OSError, PermissionError):\n                continue\n            if script in args:\n                raise RuntimeError(\'A task from this project is still running. Finish/stop it first; do not overwrite its sources.\')\n\n\ndef _profile_for_task(kind):\n    if kind in (\'check_frozen\', \'train_frozen\', \'evaluate_frozen\', \'pvp\'):\n        return \'pvp\'\n    if kind in (\'check\', \'actions\', \'train\', \'evaluate\'):\n        return \'pve\'\n    raise ValueError(\'Unknown task: \'+str(kind))\n\n\ndef _marker(root, profile):\n    return Path(root)/\'checks\'/(profile+\'_check.json\')\n\n\ndef _snapshot_packages(job, out):\n    """Copy only package bytes, never execute a model here. Preserve exact identity."""\n    refs=[]\n    if job.get(\'initialization_bundle\'):\n        refs.append((job, \'initialization_bundle\', \'initial_learner\'))\n    for section, keys in ((\'frozen\', (\'bundle\',)),\n                          (\'evaluation\', (\'bundle\',\'initial_bundle\')),\n                          (\'pvp\', (\'agent_a\',\'agent_b\'))):\n        for key in keys:\n            if job.get(section, {}).get(key):\n                refs.append((job[section],key,section+\'_\'+key))\n    if refs and job.get(\'trust_packages\') is not True:\n        raise ValueError(\'Review these Python/model packages, then explicitly enable the trust switch.\')\n    records=[];copies={}\n    folder=Path(out)/\'package_inputs\'\n    for obj, key, label in refs:\n        original=Path(obj[key]).expanduser().resolve()\n        if not original.is_file():\n            raise FileNotFoundError(\'Package does not exist: \'+str(original))\n        if original.stat().st_size > 512*1024**2:\n            raise ValueError(\'Package exceeds the supported 512 MiB limit.\')\n        if str(original) not in copies:\n            folder.mkdir(parents=True,exist_ok=True)\n            target=folder/(label+\'.zip\')\n            shutil.copy2(original,target)\n            digest=hashlib.sha256(target.read_bytes()).hexdigest()\n            copies[str(original)]=(str(target),digest)\n        target,digest=copies[str(original)]\n        records.append({\'role\':label,\'original_path\':str(original),\'snapshot\':target,\'sha256\':digest})\n        obj[key]=target\n    job[\'package_inputs\']=records\n    return records\n\n\ndef run_task(root, kind, agent, common, train, pve=None, **extra):\n    root=Path(root).resolve();code=root/\'code\'\n    assert_no_running_task(root)\n    profile=_profile_for_task(kind)\n    source=profile_directory(root,profile)\n    fingerprint=None\n    if kind in (\'check\',\'train\'):\n        fingerprint=input_check_fingerprint(root,agent,common,pve,profile=\'pve\')\n    elif kind in (\'check_frozen\',\'train_frozen\'):\n        from sf2_release import frozen_check_fingerprint\n        fingerprint=frozen_check_fingerprint(root,agent,common,{},extra[\'frozen\'],\n            initialization_bundle=extra.get(\'initialization_bundle\',\'\'),train=train)\n    if kind in (\'train\',\'train_frozen\'):\n        marker=_marker(root,profile)\n        if not marker.is_file() or json.loads(marker.read_text()).get(\'fingerprint\') != fingerprint:\n            required=\'4 (PvE check)\' if profile==\'pve\' else \'9.3 (PvP check)\'\n            raise RuntimeError(\'Run Section \'+required+\' with the current \'+profile.upper()+\' files, packages and settings before training.\')\n    # Do not require mutable student files for a saved-package-only match/evaluation.\n    uses_student=kind in (\'check\',\'actions\',\'train\',\'check_frozen\',\'train_frozen\')\n    if uses_student:\n        for name in STUDENT_FILES:\n            p=source/name\n            if not p.is_file():raise FileNotFoundError(\'Run the editing cell: \'+str(p))\n            compile(p.read_text(encoding=\'utf-8\'),str(p),\'exec\')\n    stamp=datetime.datetime.now(datetime.timezone.utc).strftime(\'%Y%m%dT%H%M%S_%f\')\n    out=root/\'runs\'/profile/f\'{stamp}_{kind}\';out.mkdir(parents=True)\n    sources=out/\'sources\';sources.mkdir()\n    if uses_student:\n        for name in STUDENT_FILES:shutil.copy2(source/name,sources/name)\n    framework=out/\'framework_sources\';framework.mkdir()\n    for p in code.glob(\'*.py\'):shutil.copy2(p,framework/p.name)\n    job={\'kind\':kind,\'profile\':profile,\'out\':str(out),\'sources\':str(sources),\n         \'agent\':copy.deepcopy(agent),\'common\':copy.deepcopy(common),\'train\':copy.deepcopy(train),\n         \'pve\':copy.deepcopy(pve or {}) if profile==\'pve\' else {}, **copy.deepcopy(extra)}\n    if kind in (\'check_frozen\',\'train_frozen\',\'evaluate_frozen\'):\n        from sf2_release import validate_frozen_config\n        job[\'frozen\']=validate_frozen_config(job[\'frozen\'])\n    package_inputs=_snapshot_packages(job,out)\n    # Freeze known package bytes between the successful check and a training launch.\n    if fingerprint is not None and profile==\'pvp\':\n        expected={\'frozen_bundle\':fingerprint[\'opponent_sha256\'],\n                  \'initial_learner\':fingerprint.get(\'initialization_sha256\')}\n        for item in package_inputs:\n            if item[\'role\'] in expected and expected[item[\'role\']] != item[\'sha256\']:\n                raise RuntimeError(\'A selected package changed while starting the task. Repeat the check.\')\n    (out/\'job.json\').write_text(json.dumps(job,indent=2),encoding=\'utf-8\')\n    env=os.environ.copy();env.update(OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\',\n        SF2_START_CACHE_DIR=str(root/\'cache\'/\'match_starts_v2\'))\n    print(\'Task:\',kind,\'| profile:\',profile,\'| output:\',out,flush=True)\n    with (out/\'console.log\').open(\'w\',encoding=\'utf-8\') as log:\n        proc=subprocess.Popen([sys.executable,\'-u\',str(code/\'run_lab.py\'),\'--job\',str(out/\'job.json\')],\n            cwd=code,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,\n            bufsize=1,start_new_session=True)\n        try:\n            for line in proc.stdout:print(line,end=\'\');log.write(line);log.flush()\n            proc.wait()\n        except KeyboardInterrupt:\n            os.killpg(proc.pid,signal.SIGINT)\n            try:proc.wait(timeout=20)\n            except subprocess.TimeoutExpired:os.killpg(proc.pid,signal.SIGKILL);proc.wait()\n            print(\'Interrupted. Existing packages were not overwritten. Results:\',out);raise\n    p=out/\'report.json\'\n    if not p.is_file():raise RuntimeError(\'No report. Inspect console.log in \'+str(out))\n    report=json.loads(p.read_text(encoding=\'utf-8\'))\n    if proc.returncode or report.get(\'status\')==\'ERROR\':\n        raise RuntimeError(report.get(\'error\',\'Task failed\')+\' | \'+str(out))\n    print(\'STATUS:\',report[\'status\'])\n    if kind in (\'check\',\'check_frozen\'):\n        expected=\'ENVIRONMENT_CHECK_PASSED\' if kind==\'check\' else \'FROZEN_ENVIRONMENT_CHECK_PASSED\'\n        if report.get(\'status\')==expected:\n            marker=_marker(root,profile);marker.parent.mkdir(exist_ok=True)\n            marker.write_text(json.dumps({\'fingerprint\':fingerprint,\'directory\':str(out)},indent=2),encoding=\'utf-8\')\n    if kind in (\'train\',\'train_frozen\'):\n        saved=json.dumps({\'directory\':str(out),**report},indent=2)\n        (root/(\'last_training_\'+profile+\'.json\')).write_text(saved,encoding=\'utf-8\')\n        print(\'BEST:\',report.get(\'best_agent\'));print(\'LAST:\',report[\'last_agent\'])\n    return out,report\n\n\ndef run_pvp_task(root, kind, agent, common, train, opponent, **extra):\n    """PvP does not accept or read PVE_CONFIG/PVE_TRAIN_CONFIG."""\n    kinds={\'check\':\'check_frozen\',\'train\':\'train_frozen\',\'evaluate\':\'evaluate_frozen\'}\n    if kind not in kinds:raise ValueError(\'PvP kind must be check, train or evaluate\')\n    return run_task(root,kinds[kind],agent,common,train,None,frozen=opponent,**extra)\n\n\ndef show_result(out,report=None):\n    from IPython.display import display,Image,Video,FileLink\n    out=Path(out);report=report or json.loads((out/\'report.json\').read_text())\n    folders=[out]+sorted(out.glob(\'learner_P*\'))\n    for folder in folders:\n        for name in (\'actual_game.png\',\'actual_model_images.png\',\'motion_game.png\',\'motion_model_images.png\'):\n            p=folder/name\n            if p.is_file():\n                print(folder.name,\':\',\'RESET input\' if name.startswith(\'actual_\') else \'STEPPED history\')\n                display(Image(filename=str(p)))\n        for name in (\'actual_input.json\',\'motion_input.json\'):\n            p=folder/name\n            if p.is_file():\n                d=json.loads(p.read_text());print(name,\': shape=\',d[\'image_shape\'],\n                    \'learner_port=\',d[\'player\'],\'frames=\',d[\'frame_ids\'])\n    videos=list(report.get(\'videos\',[]))\n    if report.get(\'video\'):videos=[report[\'video\']]+videos\n    for video in videos:\n        if video and Path(video).is_file():display(Video(video,embed=True,width=800))\n    if report.get(\'summary\'):print(json.dumps(report[\'summary\'],indent=2))\n    if report.get(\'validation_target\'):print(\'Validation target:\',report[\'validation_target\'])\n    print(\'Full results:\',out);display(FileLink(str(out/\'report.json\')))\n', 'lab_review.py': '"""Read-only run review and handoff files. Does not load or train a policy."""\nfrom pathlib import Path\nimport json\nimport zipfile\n\n\ndef show_training_review(run_directory):\n    import pandas as pd\n    import matplotlib.pyplot as plt\n    from IPython.display import display\n    run = Path(run_directory)\n    if not run.is_dir():\n        raise FileNotFoundError(run)\n    validation = run/\'validation_history.jsonl\'\n    if validation.is_file():\n        records = [json.loads(s) for s in validation.read_text().splitlines() if s.strip()]\n    else:\n        records = []\n        for path in sorted(run.glob(\'validation_*/report.json\')):\n            data = json.loads(path.read_text())\n            step = path.parent.name.rsplit(\'_\', 1)[-1]\n            if step.isdigit():\n                records.append({\'steps\': int(step), \'win_fraction\': data[\'win_fraction\'],\n                    \'mean_hp_margin\': data[\'mean_hp_margin\'], \'report\': str(path)})\n    if records:\n        table = pd.DataFrame(records).sort_values(\'steps\').drop_duplicates(\'steps\', keep=\'last\')\n        display(table[[c for c in (\'steps\',\'target\',\'learner_sides\',\'win_fraction\',\'mean_hp_margin\') if c in table]])\n        fig, ax = plt.subplots(figsize=(7, 3))\n        ax.plot(table[\'steps\'], table[\'win_fraction\'], marker=\'o\')\n        ax.set(xlabel=\'Total transitions in this phase\', ylabel=\'Validation win fraction\', ylim=(-0.03, 1.03))\n        target=str(table[\'target\'].iloc[-1]) if \'target\' in table else \'recorded validation\'\n        ax.set_title(\'Validation target: \'+target)\n        fig.tight_layout(); fig.savefig(run/\'validation_curve.png\', dpi=150); plt.show()\n    path = run/\'logs/progress.csv\'\n    if path.is_file():\n        logs = pd.read_csv(path)\n        if {\'time/total_timesteps\', \'rollout/ep_rew_mean\'} <= set(logs.columns):\n            fig, ax = plt.subplots(figsize=(7, 3))\n            ax.plot(logs[\'time/total_timesteps\'], logs[\'rollout/ep_rew_mean\'])\n            ax.set(xlabel=\'Total transitions in this phase\', ylabel=\'Mean training episode return\')\n            ax.set_title(\'Training return is not a validation win rate\')\n            fig.tight_layout(); fig.savefig(run/\'reward_curve.png\', dpi=150); plt.show()\n    selection = run/\'best_selection.json\'\n    if selection.is_file():\n        print(\'Best checkpoint selection:\', json.loads(selection.read_text()))\n    path = run/\'episodes.jsonl\'\n    if path.is_file():\n        rows = [json.loads(s) for s in path.read_text().splitlines() if s.strip()][-100:]\n        groups = []\n        for result in (\'win\', \'loss\', \'draw\', \'cutoff\'):\n            values = [float(r[\'episode\'][\'r\']) for r in rows if r[\'result\'] == result]\n            if values:\n                groups.append({\'result\': result, \'episodes\': len(values),\n                    \'mean_episode_return\': sum(values)/len(values),\n                    \'positive_return_episodes\': sum(v>0 for v in values)})\n        if groups:\n            print(\'Most recent completed training episodes (not the validation games):\')\n            display(pd.DataFrame(groups))\n        ports=[]\n        for player in (1,2):\n            subset=[r for r in rows if r.get(\'self_player\')==player]\n            if subset:ports.append({\'learner_player\':player,\'episodes\':len(subset),\n                \'win_fraction\':sum(r[\'result\']==\'win\' for r in subset)/len(subset)})\n        if ports:display(pd.DataFrame(ports))\n\n\ndef make_handoff(run_directory, destination=None):\n    """Archive model packages and reports, not ROMs, caches or emulator states."""\n    run = Path(run_directory).resolve()\n    if not (run/\'report.json\').is_file():\n        raise FileNotFoundError(\'A completed run report is required.\')\n    dest = Path(destination) if destination else run.parent/(run.name+\'_handoff.zip\')\n    paths = []\n    for name in (\'best_agent.zip\', \'last_agent.zip\', \'initial_agent.zip\', \'report.json\',\n                 \'job.json\', \'best_selection.json\', \'episodes.jsonl\', \'console.log\',\n                 \'validation_history.jsonl\', \'performance_rollouts.json\',\n                 \'validation_curve.png\', \'reward_curve.png\'):\n        p = run/name\n        if p.is_file(): paths.append(p)\n    for folder in (\'sources\', \'framework_sources\', \'logs\'):\n        d = run/folder\n        if d.is_dir(): paths.extend(p for p in d.rglob(\'*\') if p.is_file() and \'__pycache__\' not in p.parts)\n    paths.extend(run.glob(\'validation_*/report.json\'))\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(dest, \'w\', zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(set(paths)):\n            z.write(p, p.relative_to(run))\n        z.writestr(\'HANDOFF_README.txt\',\n            \'Agent files are trusted-Python packages, not raw .pth weights.\\n\'\n            \'Use best_agent.zip for the validation-selected model; last_agent.zip is the final checkpoint.\\n\'\n            \'A completed run does not certify a winning model. Check the recorded evaluation results.\\n\'\n            \'No ROM, match-start state, game cache, credentials or pretrained external weights are included.\\n\')\n    print(\'Handoff archive:\', dest)\n    return dest\n', 'run_lab.py': '"""Explicit task entry points. Ordinary PPO and standard SB3 vector environments.\nTraining, evaluation and PvP are separate tasks; no overloaded global MODE.\n"""\nfrom __future__ import annotations\nimport argparse, hashlib, json, math, os, shutil, subprocess, sys, time, traceback\nfrom pathlib import Path\nimport numpy as np\nfrom sf2_common import (json_write,file_hash,source_module,parameter_hash,\n                         rules_signature,runtime_versions)\nfrom sf2_actions import library,compiled,action_index,QUANTUM\nfrom sf2_bundle import make_spec,LoadedAgent,export_agent\nfrom sf2_env import BattleEnv\nfrom sf2_observation import description as input_description,image_shape,display_frames\nfrom sf2_release import (initialise_policy, validate_frozen_config, validate_pvp_evaluation,\n                         validation_job, validation_target)\nfrom sf2_pvp_eval import evaluate_frozen\n\n\ndef student(job):\n    folder=Path(job[\'sources\'])\n    actions=source_module(folder/\'student_actions.py\',\'actions\')\n    reward=source_module(folder/\'student_reward.py\',\'reward\').compute_reward\n    module=source_module(folder/\'student_model.py\',\'model\')\n    return make_spec(job[\'agent\'],library(actions.CUSTOM_ACTIONS),folder/\'student_observation.py\'),reward,module\n\n\ndef schedule(settings):\n    from sf2_boot import normalize_character\n    chars=settings[\'opponent_characters\'];levels=settings[\'difficulty_stars\']\n    if not chars or not levels:raise ValueError(\'Choose opponent characters and CPU difficulties.\')\n    if any(type(x) is not int or x not in range(1,9) for x in levels):\n        raise ValueError(\'CPU difficulty must be 1..8 stars.\')\n    jitter=settings.get(\'neutral_jitter_max\',0)\n    if type(jitter) is not int or not 0<=jitter<=60:raise ValueError(\'neutral_jitter_max must be 0..60.\')\n    return [{\'character\':normalize_character(c),\'difficulty_stars\':n,\'neutral_jitter_max\':jitter}\n             for c in chars for n in levels]\n\n\ndef show_input(folder,env,obs,prefix=\'actual\'):\n    from PIL import Image,ImageDraw\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    Image.fromarray(env.render()).save(folder/(prefix+\'_game.png\'))\n    values=obs if isinstance(obs,dict) else {\'image\':obs}\n    np.savez_compressed(folder/(prefix+\'_input.npz\'),**values)\n    images=values[\'image\'];frames=display_frames(images,env.spec)\n    side=env.sides[env.player]\n    h,w=frames[0].shape[:2];zoom=2;top=30\n    board=Image.new(\'RGB\',(4*w*zoom,h*zoom+top),\'white\');draw=ImageDraw.Draw(board)\n    for i,img in enumerate(frames):\n        im=Image.fromarray(img).convert(\'RGB\').resize((w*zoom,h*zoom),Image.Resampling.NEAREST)\n        board.paste(im,(i*w*zoom,top))\n        draw.text((i*w*zoom+5,5),f\'Game frame {side.frame_ids[i]}\',fill=\'black\')\n    board.save(folder/(prefix+\'_model_images.png\'))\n    meta={\n        \'keys\':list(values),\'image_shape\':list(images.shape),\'image_dtype\':str(images.dtype),\n        \'position_order\':[\'self_x\',\'self_y\',\'opponent_x\',\'opponent_y\',\'dx\',\'dy\'],\n        \'position\':values.get(\'position\',np.array([])).tolist(),\n        \'scale\':env.common[\'position_scale\'],\'player\':env.player,\n        \'raw\':dict(env.game.raw),\'image_mirrored\':False,\'control_input\':False,\n        \'frame_ids\':list(side.frame_ids),\'observed_frames_checked\':side.observed_frames_checked,\n        \'input_definition\':input_description(env.spec),\n        \'source\':\'Actual env observation returned to the policy, not a candidate resize\'}\n    json_write(folder/(prefix+\'_input.json\'),meta)\n    print(\'ACTUAL MODEL INPUT:\',json.dumps({k:meta[k] for k in\n          (\'keys\',\'image_shape\',\'image_dtype\',\'frame_ids\')}),flush=True)\n    return meta\n\n\ndef check(job,out):\n    """One selected real match: cache roundtrip, fixed input replay and a real preview."""\n    import gymnasium as gym\n    import torch\n    from sf2_observation import observation_space\n    from sf2_video import Video\n    spec,reward,module=student(job)\n    # Installed ordinary PPO, synthetic shape contract only. This model is never trained/exported.\n    shape_space=observation_space(spec[\'use_position\'],job[\'common\'][\'position_scale\'],spec=spec)\n    shape_actions=gym.spaces.Discrete(len(spec[\'actions\']))\n    class ShapeOnly(gym.Env):\n        observation_space=shape_space\n        action_space=shape_actions\n    shape=ShapeOnly()\n    model=module.build_model(shape,job[\'train\'],job[\'agent\'][\'seed\'],\'cpu\')\n    data={\'image\':np.zeros(image_shape(spec),np.uint8),\'position\':np.zeros(6,np.float32)}\n    if not spec[\'use_position\']:data=data[\'image\']\n    action,_=model.predict(data,deterministic=True)\n    action_index(action,len(spec[\'actions\']));del model\n    pair=schedule(job[\'pve\'])[0]\n    env=BattleEnv(spec,job[\'common\'],[pair],out/\'game\',reward)\n    traces=[];resets=[];rec=None;motion_input=None\n    try:\n        for repeat in range(2):\n            t=time.perf_counter();obs,info=env.reset(seed=int(job[\'agent\'][\'seed\']),options={\'offset\':0})\n            resets.append(time.perf_counter()-t)\n            if repeat==0:reset_input=show_input(out,env,obs)\n            h=hashlib.sha256()\n            for i in range(60):\n                values=obs if isinstance(obs,dict) else {\'image\':obs}\n                for key in sorted(values):h.update(key.encode()+np.ascontiguousarray(values[key]).tobytes())\n                h.update(json.dumps(env.game.raw,sort_keys=True).encode())\n                # Uses only the 15 known built-in actions, not student-provided scripts.\n                obs,r,term,trunc,info=env.step((2,2,11,0,3,0,14,0)[i%8])\n                h.update(json.dumps([r,term,trunc,info[\'result\'],info[\'end_reason\']]).encode())\n                if repeat==0 and i==11:\n                    motion_input=show_input(out,env,obs,prefix=\'motion\')\n                if term or trunc:break\n            traces.append(h.hexdigest())\n        if traces[0]!=traces[1]:raise RuntimeError(\'Same-start fixed-input replay differs after warm reset.\')\n        if motion_input is None:raise RuntimeError(\'Round ended before a stepped four-frame input could be checked.\')\n        obs,_=env.reset(seed=int(job[\'agent\'][\'seed\']),options={\'offset\':0})\n        rec=Video(out/\'preview.mp4\',env.game.fps)\n        env.game.on_frame=lambda rgb,raw,k1,k2,frame:rec.write(rgb,\n            [\'RANDOM PREVIEW | no model learning\',\n             f\'P1 HP={raw["p1_hp"]} | P2 HP={raw["p2_hp"]} | frame={frame}\',\n             \'Input display is not proof of an executed attack.\'],k1,k2)\n        rng=np.random.default_rng(int(job[\'agent\'][\'seed\']))\n        for _ in range(120):\n            obs,r,t,tr,info=env.step(int(rng.integers(env.action_space.n)))\n            if t or tr:break\n        video=rec.finish();rec=None\n        print(\'ENVIRONMENT_CHECK_PASSED | ordinary PPO contract + one cached match replay\',flush=True)\n        print(\'Action count:\',len(spec[\'actions\']),\'| input:\',list(obs) if isinstance(obs,dict) else [\'image\'],flush=True)\n        return {\'status\':\'ENVIRONMENT_CHECK_PASSED\',\'training_performed\':False,\n                \'game_config\':pair,\'trace_sha256\':traces,\'reset_seconds\':resets,\n                \'action_names\':[a[\'name\'] for a in spec[\'actions\']],\n                \'input_keys\':list(obs) if isinstance(obs,dict) else [\'image\'],\n                \'input_definition\':input_description(spec),\'actual_image_shape\':list(image_shape(spec)),\n                \'motion_frame_ids\':None if motion_input is None else motion_input[\'frame_ids\'],\n                \'video\':video,\'scope\':\'First PvE condition; real reset and stepped history, fixed 60-decision replay, per-observed-frame padding guard. Not an all-match crop certification or a winning model.\'}\n    finally:\n        if rec:rec.close()\n        env.close()\n\n\ndef actions_preview(job,out):\n    """C hold/release or a named action test, always actual emulator output."""\n    from sf2_game import Game\n    from sf2_video import Video\n    spec,_,_=student(job);p=schedule(job[\'pve\'])[0]\n    game=Game(job[\'common\'],out/\'game\');videos=[];rows=[];rec=None\n    try:\n        tests=job.get(\'actions\',{}).get(\'names\',[\'C_hold\',\'C_tap\',\'NO_OP\'])\n        cmds=compiled(spec[\'actions\']);names=[a[\'name\'] for a in spec[\'actions\']]\n        for name in tests:\n            game.open(spec[\'character\'],p[\'character\'],\'pve\',p[\'difficulty_stars\'])\n            rec=Video(out/(name+\'.mp4\'),game.fps);trace=[];previous=();presses=0\n            for frame in range(240):\n                if name==\'C_hold\':keys=(\'C\',)\n                elif name==\'C_tap\':keys=(\'C\',) if frame%30<6 else ()\n                elif name in names:\n                    keys=cmds[names.index(name)][frame%6] if frame%60<6 else ()\n                else:raise ValueError(\'Unknown preview name: \'+name)\n                presses+=int(\'C\' in keys and \'C\' not in previous);previous=keys\n                raw,winner,reason=game.step_keys(keys,())\n                rec.write(game.rgb,[f\'ACTION CHECK | {name} | frame={frame}\',\n                    f\'P1 HP={raw["p1_hp"]} | P2 HP={raw["p2_hp"]} | C press edges={presses}\',\n                    \'No model. Native CPU active. Buttons are not an animation detector.\'],keys,())\n                trace.append({\'frame\':frame,\'keys\':list(keys),\'raw\':dict(raw)})\n                if reason:break\n            videos.append(rec.finish());rec=None\n            json_write(out/(name+\'_trace.json\'),trace)\n            rows.append({\'name\':name,\'frames\':len(trace),\'C_press_edges\':presses})\n        return {\'status\':\'ACTIONS_RECORDED\',\'training_performed\':False,\'videos\':videos,\'cases\':rows}\n    finally:\n        if rec:rec.close()\n        game.close()\n\n\ndef summary(rows):\n    n=len(rows);count={k:sum(r[\'result\']==k for r in rows) for k in (\'win\',\'loss\',\'draw\',\'cutoff\')}\n    return {**count,\'episodes\':n,\'win_fraction\':count[\'win\']/n if n else None,\n            \'mean_hp_margin\':float(np.mean([max(0,r[\'self_hp\'])-max(0,r[\'opponent_hp\']) for r in rows])) if n else None}\n\n\ndef evaluate(job,out):\n    import torch\n    from sf2_video import Video\n    cfg=job[\'evaluation\'];rows=[];videos=[]\n    offsets=cfg.get(\'offsets\',[1,7,13,19,25,31,37,43,49,55])\n    if not offsets or any(type(x) is not int or not 0<=x<=60 for x in offsets):\n        raise ValueError(\'Evaluation offsets must be a nonempty list of integers in 0..60.\')\n    actors=[]\n    trained=LoadedAgent(cfg[\'bundle\'],trusted=job.get(\'trust_packages\',False),device=\'cpu\')\n    actors.append((\'trained\',trained,bool(cfg.get(\'deterministic\',True))))\n    try:\n        if rules_signature(trained.common)!=rules_signature(job[\'common\']):raise ValueError(\'Evaluation arena rules differ from the package.\')\n        if cfg.get(\'compare_sampling\',False):actors.append((\'trained_sampled\',trained,False))\n        if cfg.get(\'initial_bundle\'):\n            initial=LoadedAgent(cfg[\'initial_bundle\'],trusted=job.get(\'trust_packages\',False),device=\'cpu\')\n            if initial.spec!=trained.spec or rules_signature(initial.common)!=rules_signature(trained.common):\n                initial.close();raise ValueError(\'Initial and trained packages differ in actions, character or arena rules.\')\n            actors.insert(0,(\'initial\',initial,bool(cfg.get(\'deterministic\',True))))\n        if cfg.get(\'include_random\',True):actors.insert(0,(\'random\',None,False))\n        for label,actor,det in actors:\n            # A separate evaluation process: these seeds do not alter training RNGs.\n            torch.manual_seed(int(job[\'agent\'][\'seed\']))\n            rng=np.random.default_rng(int(job[\'agent\'][\'seed\']))\n            spec=trained.spec if actor is None else actor.spec\n            reward_module=source_module(trained.directory/\'reward.py\',\'reward\')\n            env=BattleEnv(spec,job[\'common\'],schedule(job[\'pve\']),out/label,reward_module.compute_reward)\n            rec=None\n            try:\n                for ci,pair in enumerate(schedule(job[\'pve\'])):\n                    env.opponents=[pair]\n                    for j,offset in enumerate(offsets):\n                        obs,reset=env.reset(seed=int(job[\'agent\'][\'seed\'])+ci*1000+j,options={\'offset\':offset})\n                        record=cfg.get(\'record_video\',True) and ci==0 and j==0\n                        if record:\n                            rec=Video(out/f\'{label}.mp4\',env.game.fps)\n                            def frame(rgb,raw,k1,k2,number):\n                                rec.write(rgb,[f\'{label} | CPU stars={pair["difficulty_stars"]} | offset={offset}\',\n                                    f\'P1 HP={raw["p1_hp"]} | P2 HP={raw["p2_hp"]} | frame={number}\',\n                                    \'NO LEARNING | input highlight does not guarantee an executed move.\'],k1,k2)\n                            env.game.on_frame=frame\n                        counts={};total_reward=0.;first_damage=None;decision=0\n                        while True:\n                            if actor is None:a=int(rng.integers(env.action_space.n))\n                            else:a=actor.act(env.sides[env.player],env.game.raw,deterministic=det)\n                            obs,r,t,tr,info=env.step(a);total_reward+=r;decision+=1\n                            name=info[\'action_name\'];counts[name]=counts.get(name,0)+1\n                            if first_damage is None and info[\'damage_dealt\']>0:first_damage=decision\n                            if t or tr:break\n                        row={k:info[k] for k in (\'result\',\'end_reason\',\'self_hp\',\'opponent_hp\',\'native_frames\')}\n                        row.update(controller=label,deterministic=det,condition=pair,offset=offset,\n                                   initial_readings=reset[\'raw\'],action_counts=counts,\n                                   first_damage_decision=first_damage,episode_reward=total_reward)\n                        rows.append(row)\n                        print(f\'EVAL {label}: offset={offset}, {row["result"]}, HP={row["self_hp"]}/{row["opponent_hp"]}, reward={total_reward:+.4f}, reason={row["end_reason"]}\',flush=True)\n                        if rec:videos.append(rec.finish());rec=None\n                        env.game.on_frame=None\n            finally:\n                if rec:rec.close()\n                env.close()\n        groups={label:summary([r for r in rows if r[\'controller\']==label]) for label,_,_ in actors}\n        print(\'EVALUATION SUMMARY:\',json.dumps(groups),flush=True)\n        return {\'status\':\'EVALUATION_COMPLETE\',\'training_performed\':False,\'episodes\':rows,\n                \'summary\':groups,\'videos\':videos,\'win_fraction\':groups[\'trained\'][\'win_fraction\'],\n                \'mean_hp_margin\':groups[\'trained\'][\'mean_hp_margin\'],\n                \'note\':\'Offsets are start-time scenarios, not independently seeded native CPUs. Selection validation is not a held-out task.\'}\n    finally:\n        closed=set()\n        for _,actor,_ in actors:\n            if actor is not None and id(actor) not in closed:actor.close();closed.add(id(actor))\n\n\nclass WorkerFactory:\n    """Pickleable factory for spawn, one emulator in each worker process."""\n    def __init__(self,job,rank):self.job=job;self.rank=rank\n    def __call__(self):\n        import torch,cv2\n        torch.set_num_threads(1);cv2.setNumThreads(1)\n        spec,reward,_=student(self.job)\n        frozen=None\n        if self.job[\'kind\']==\'train_frozen\':\n            frozen=LoadedAgent(self.job[\'frozen\'][\'bundle\'],trusted=self.job.get(\'trust_packages\',False),device=\'cpu\')\n            opponents=[{\'character\':frozen.spec[\'character\'],\'difficulty_stars\':1,\n                        \'neutral_jitter_max\':self.job[\'frozen\'][\'neutral_jitter_max\']}]\n            sides=tuple(self.job[\'frozen\'][\'learner_sides\'])\n        else:opponents=schedule(self.job[\'pve\']);sides=(1,)\n        return BattleEnv(spec,self.job[\'common\'],opponents,\n                         Path(self.job[\'out\'])/f\'worker_{self.rank:02d}\',reward,\n                         opponent_agent=frozen,learner_sides=sides,\n                         opponent_deterministic=bool(self.job.get(\'frozen\',{}).get(\'opponent_deterministic\',True)))\n\n\ndef check_frozen(job, out):\n    """Real two-controller smoke test. No learner update, no strength claim."""\n    from sf2_bundle import LoadedAgent\n    from sf2_video import Video\n    from sf2_observation import position\n    spec, reward, module = student(job)\n    cfg = validate_frozen_config(job[\'frozen\'])\n    sides = list(cfg[\'learner_sides\'])\n    if not sides or any(type(p) is not int or p not in (1,2) for p in sides):\n        raise ValueError(\'Use learner_sides=[1], [2], or [1,2].\')\n    rows=[]; videos=[]\n    for player in sides:\n        opponent=LoadedAgent(cfg[\'bundle\'],trusted=job.get(\'trust_packages\',False),device=\'cpu\')\n        pair={\'character\':opponent.spec[\'character\'],\'difficulty_stars\':1,\'neutral_jitter_max\':cfg[\'neutral_jitter_max\']}\n        env=None;rec=None\n        try:\n            env=BattleEnv(spec,job[\'common\'],[pair],out/f\'learner_P{player}\',reward,\n                opponent_agent=opponent,learner_sides=(player,),\n                opponent_deterministic=bool(cfg.get(\'opponent_deterministic\',True)))\n            obs,_=env.reset(seed=int(job[\'agent\'][\'seed\']),options={\'offset\':0})\n            learner_start=show_input(out/f\'learner_P{player}\',env,obs)\n            if env.player!=player:raise RuntimeError(\'Requested learner port was not selected.\')\n            learner=module.build_model(env,dict(job[\'train\'],n_steps=8,batch_size=8,n_epochs=1),\n                                       int(job[\'agent\'][\'seed\']),\'cpu\')\n            initialization=initialise_policy(learner,spec,job[\'sources\'],job[\'common\'],\n                job.get(\'initialization_bundle\',\'\'),trusted=job.get(\'trust_packages\',False))\n            before=parameter_hash(learner)\n            rec=Video(out/f\'frozen_check_learner_P{player}.mp4\',env.game.fps)\n            def on_frame(rgb,raw,k1,k2,number):\n                rec.write(rgb,[f\'FROZEN ENV CHECK | learner=P{player} | opponent=P{3-player}\',\n                    f\'P1 HP={raw["p1_hp"]} | P2 HP={raw["p2_hp"]} | frame={number}\',\n                    \'No optimiser updates. This is a wiring check, not a winning-model test.\'],k1,k2)\n            env.game.on_frame=on_frame\n            transitions=[]\n            for decision in range(60):\n                pre_elapsed=env.game.elapsed\n                pre_raw=dict(env.game.raw)\n                if spec[\'use_position\']:\n                    np.testing.assert_array_equal(obs[\'position\'],position(pre_raw,player,job[\'common\'][\'position_scale\']))\n                action,_=learner.predict(obs,deterministic=True)\n                obs,r,terminated,truncated,info=env.step(action)\n                advanced=env.game.elapsed-pre_elapsed\n                if not (1<=advanced<=6):raise RuntimeError(\'Frozen mode advanced outside the six-frame protocol.\')\n                if not(terminated or truncated) and advanced!=6:raise RuntimeError(\'Nonterminal step did not run six frames.\')\n                if env.sides[1].frame_ids[-1]!=env.sides[2].frame_ids[-1]:\n                    raise RuntimeError(\'P1/P2 observations do not share the same game clock.\')\n                if decision==11:show_input(out/f\'learner_P{player}\',env,obs,prefix=\'motion\')\n                transitions.append({\'decision\':decision,\'learner_player\':player,\'action\':int(action),\n                    \'reward\':r,\'result\':info[\'result\'],\'end_reason\':info[\'end_reason\'],\n                    \'frames\':advanced,\'raw_before\':pre_raw,\'raw_after\':dict(env.game.raw)})\n                if terminated or truncated:break\n            stats=env.performance_stats()\n            if not stats[\'frozen_opponent_unchanged\'] or parameter_hash(learner)!=before:\n                raise RuntimeError(\'A model changed during the no-learning frozen check.\')\n            videos.append(rec.finish());rec=None\n            json_write(out/f\'frozen_check_P{player}_transitions.json\',transitions)\n            rows.append({\'learner_player\':player,\'decisions\':len(transitions),\n                \'initialization\':initialization,\'input_shape\':learner_start[\'image_shape\'],\n                \'learner_unchanged\':True,\'frozen_opponent_unchanged\':True})\n        finally:\n            if rec:rec.close()\n            if env:env.close()\n            else:opponent.close()\n    return {\'status\':\'FROZEN_ENVIRONMENT_CHECK_PASSED\',\'training_performed\':False,\n            \'cases\':rows,\'videos\':videos,\n            \'scope\':\'Requested ports; 60 decisions each; self-first positions, shared frame clock, unchanged model weights. Not proof of learning quality.\'}\n\n\ndef train(job,out):\n    import torch\n    from stable_baselines3 import PPO\n    from stable_baselines3.common.vec_env import SubprocVecEnv,DummyVecEnv\n    from stable_baselines3.common.callbacks import BaseCallback\n    from stable_baselines3.common.logger import configure\n    cfg=dict(job[\'train\']);spec,reward,module=student(job)\n    if job[\'kind\']==\'train_frozen\':\n        job=dict(job,frozen=validate_frozen_config(job[\'frozen\']),\n                 pvp_validation=validate_pvp_evaluation(job.get(\'pvp_validation\',{})))\n    n=int(cfg[\'n_envs\']);steps=int(cfg[\'n_steps\']);batch=int(cfg[\'batch_size\'])\n    if n not in (1,2,4,8):raise ValueError(\'n_envs must be 1,2,4 or 8.\')\n    if steps<2 or batch<2 or steps*n%batch:raise ValueError(\'n_envs*n_steps must divide evenly into batch_size.\')\n    if int(cfg[\'total_timesteps\'])<=0:raise ValueError(\'total_timesteps must be positive.\')\n    if cfg.get(\'resume_from\'):raise ValueError(\'Exact optimiser resume is not supported here. Use initialization_bundle for a new phase from saved policy weights.\')\n    device=cfg.get(\'device\',\'auto\')\n    if device==\'auto\':device=\'cuda:0\' if torch.cuda.is_available() else \'cpu\'\n    if str(device).startswith(\'cuda\') and not torch.cuda.is_available():raise RuntimeError(\'CUDA requested but unavailable.\')\n    torch.set_num_threads(int(cfg.get(\'torch_threads\',2)))\n    vec=None;model=None;completed=False;episodes=[];rollouts=[]\n    source=Path(job[\'sources\']);start=time.perf_counter()\n    try:\n        # Game engines run in CPU processes; all learner parameters stay in this one model.\n        t=time.perf_counter()\n        vec=(SubprocVecEnv([WorkerFactory(job,i) for i in range(n)],start_method=\'spawn\')\n             if n>1 else DummyVecEnv([WorkerFactory(job,0)]))\n        torch.set_num_threads(int(cfg.get(\'torch_threads\',2)))\n        worker_start=time.perf_counter()-t\n        model=module.build_model(vec,cfg,int(job[\'agent\'][\'seed\']),device)\n        if not isinstance(model,PPO):raise TypeError(\'This classroom bundle supports ordinary stable_baselines3.PPO.\')\n        model.set_logger(configure(str(out/\'logs\'),[\'stdout\',\'csv\']))\n        devices=sorted({str(p.device) for p in model.policy.parameters()})\n        print(\'ACTUAL TRAINING CONFIG:\',json.dumps({\'algorithm\':\'PPO\',\'n_envs\':model.n_envs,\n            \'n_steps_per_env\':steps,\'rollout_total\':n*steps,\'devices\':devices,\n            \'image_shape\':list(image_shape(spec)),\'observation_sha256\':spec[\'observation\'][\'source_sha256\'],\n            \'action_count\':len(spec[\'actions\']),\'input\':\'image+position6\' if spec[\'use_position\'] else \'image_only\',\n            \'requested_steps\':cfg[\'total_timesteps\'],\'effective_step_budget\':math.ceil(cfg[\'total_timesteps\']/(n*steps))*n*steps,\n            \'learning_rate\':cfg[\'learning_rate\'],\'gamma\':cfg[\'gamma\'],\'clip_range\':cfg[\'clip_range\'],\n            \'batch_size\':batch,\'n_epochs\':cfg[\'n_epochs\'],\n            \'profile\':job.get(\'profile\'), \'validation_target\':validation_target(job),\n            \'learner_sides\':job.get(\'frozen\',{}).get(\'learner_sides\',[1])}),flush=True)\n        initialization = initialise_policy(model, spec, source, job[\'common\'],\n            job.get(\'initialization_bundle\', \'\'), trusted=job.get(\'trust_packages\', False))\n        print(\'TRAINING INITIALIZATION:\', json.dumps(initialization), flush=True)\n        before=parameter_hash(model)\n        def pack(name,label):\n            return export_agent(out/name,model,spec,source/\'student_model.py\',source/\'student_reward.py\',job[\'common\'],\n                {\'label\':label,\'train\':cfg,\'initialization\':initialization,\'steps\':int(model.num_timesteps),\'sources\':{p.name:file_hash(p) for p in source.glob(\'*.py\')},\n                 \'profile\':job.get(\'profile\'), \'validation_target\':validation_target(job),\n                 \'opponent_sha256\':file_hash(job[\'frozen\'][\'bundle\']) if job[\'kind\']==\'train_frozen\' else None,\n                 \'learner_sides\':job.get(\'frozen\',{}).get(\'learner_sides\',[1])})\n        initial=pack(\'initial_agent.zip\',\'untrained_control\' if initialization[\'mode\']==\'from_scratch\' else \'phase_start_control\')\n        class Audit(BaseCallback):\n            def __init__(self):\n                super().__init__();self.end=None;self.begin=None;self.best=None;self.saved=0;self.evaluated=0\n                self.sampling_seconds=0.;self.update_seconds=0.;self.evaluation_seconds=0.;self.counts={}\n            def _on_step(self):\n                for rank,info in enumerate(self.locals[\'infos\']):\n                    name=info.get(\'action_name\',\'unknown\');self.counts[name]=self.counts.get(name,0)+1\n                    if \'episode\' in info:\n                        row={k:info.get(k) for k in (\'result\',\'end_reason\',\'self_hp\',\'opponent_hp\',\'self_player\',\'native_frames\',\'neutral_offset_frames\')}\n                        row.update(steps=int(self.num_timesteps),worker=rank,episode=info[\'episode\'])\n                        episodes.append(row)\n                        with (out/\'episodes.jsonl\').open(\'a\',encoding=\'utf-8\') as f:f.write(json.dumps(row)+\'\\n\')\n                return True\n            def update_timing(self):\n                if self.end is not None:\n                    if model.device.type==\'cuda\':torch.cuda.synchronize(model.device)\n                    seconds=time.perf_counter()-self.end;self.update_seconds+=seconds\n                    rollouts[-1][\'update_and_logging_seconds\']=seconds\n                    print(f\'PERF envs={n} sample={rollouts[-1]["sample_seconds"]:.3f}s update+log={seconds:.3f}s\',flush=True)\n                    json_write(out/\'performance_rollouts.json\',rollouts);self.end=None\n            def validate(self,package):\n                t=time.perf_counter();dest=out/f\'validation_{int(model.num_timesteps)}\'\n                ej=validation_job(job,package,dest)\n                path=out/\'evaluation_job.json\';json_write(path,ej)\n                subprocess.run([sys.executable,\'-u\',__file__,\'--job\',str(path)],check=True)\n                data=json.loads((dest/\'report.json\').read_text())\n                self.evaluation_seconds+=time.perf_counter()-t\n                with (out/\'validation_history.jsonl\').open(\'a\',encoding=\'utf-8\') as history:\n                    history.write(json.dumps({\'steps\':int(model.num_timesteps),\n                        \'target\':validation_target(job),\'learner_sides\':data.get(\'learner_sides\',[1]),\n                        \'opponent_sha256\':data.get(\'opponent_sha256\'),\n                        \'per_port\':data.get(\'per_port\'), \'win_fraction\':data[\'win_fraction\'],\n                        \'mean_hp_margin\':data[\'mean_hp_margin\'], \'report\':str(dest/\'report.json\')})+\'\\n\')\n                score=(data[\'win_fraction\'],data[\'mean_hp_margin\'])\n                if self.best is None or score>self.best:\n                    self.best=score;shutil.copy2(package,out/\'best_agent.zip\')\n                    json_write(out/\'best_selection.json\',{\'steps\':int(model.num_timesteps),\'score\':score,\n                        \'target\':validation_target(job),\'learner_sides\':data.get(\'learner_sides\',[1]),\n                        \'opponent_sha256\':data.get(\'opponent_sha256\'),\'validation\':str(dest)})\n            def _on_rollout_start(self):\n                self.update_timing();now=int(model.num_timesteps);package=None\n                if cfg.get(\'checkpoint_every\',0) and now-self.saved>=cfg[\'checkpoint_every\']:\n                    package=pack(f\'checkpoint_{now}_agent.zip\',\'checkpoint\');self.saved=now\n                if cfg.get(\'evaluate_every\',0) and now-self.evaluated>=cfg[\'evaluate_every\']:\n                    package=package or pack(f\'evaluation_{now}_agent.zip\',\'validation_checkpoint\')\n                    self.validate(package);self.evaluated=now\n                if model.device.type==\'cuda\':torch.cuda.synchronize(model.device)\n                self.begin=time.perf_counter()\n            def _on_rollout_end(self):\n                if model.device.type==\'cuda\':torch.cuda.synchronize(model.device)\n                seconds=time.perf_counter()-self.begin;self.sampling_seconds+=seconds\n                rollouts.append({\'steps\':int(model.num_timesteps),\'sample_seconds\':seconds})\n                recent = episodes[-100:]\n                if recent:\n                    for outcome in (\'win\', \'loss\', \'draw\', \'cutoff\'):\n                        self.logger.record(\'audit/train_\'+outcome+\'_fraction_100\',\n                            sum(row[\'result\']==outcome for row in recent)/len(recent))\n                    self.logger.record(\'audit/completed_training_episodes\', len(episodes))\n                self.end=time.perf_counter()\n        audit=Audit()\n        # PvP can select the starting policy if subsequent training regresses.\n        # The evaluation subprocess cannot advance the learner\'s RNG or emulator.\n        if job[\'kind\']==\'train_frozen\' and cfg.get(\'evaluate_initial\',True):audit.validate(initial)\n        model.learn(total_timesteps=int(cfg[\'total_timesteps\']),callback=audit,progress_bar=False)\n        audit.update_timing()\n        if before==parameter_hash(model):raise RuntimeError(\'PPO returned without changing any policy parameters.\')\n        latest=pack(\'last_agent.zip\',\'last_trained\')\n        if cfg.get(\'evaluate_final\',True):audit.validate(latest)\n        # Same-device export/reload check: no emulator is created by the agent loader.\n        held=model._last_obs\n        def probs(m):\n            m.policy.set_training_mode(False)\n            data,_=m.policy.obs_to_tensor(held)\n            with torch.no_grad():return m.policy.get_distribution(data).distribution.probs.cpu().numpy()\n        expected=probs(model);loaded=LoadedAgent(latest,trusted=True,device=device)\n        try:error=float(np.max(np.abs(expected-probs(loaded.model))))\n        finally:loaded.close()\n        if error>1e-5:raise RuntimeError(f\'Export/reload probability mismatch: {error}\')\n        active=audit.sampling_seconds+audit.update_seconds\n        worker_stats = vec.env_method(\'performance_stats\')\n        if job[\'kind\']==\'train_frozen\' and not all(row.get(\'frozen_opponent_unchanged\') is True for row in worker_stats):\n            raise RuntimeError(\'A frozen opponent changed parameters during training.\')\n        completed=True\n        return {\'status\':\'TRAINING_COMPLETE\' ,\'training_performed\':True,\'algorithm\':\'PPO\',\n                \'n_envs\':n,\'devices\':devices,\'actual_timesteps\':int(model.num_timesteps),\n                \'initial_agent\':initial,\'last_agent\':latest,\n                \'best_agent\':str(out/\'best_agent.zip\') if (out/\'best_agent.zip\').exists() else None,\n                \'sampling_seconds\':audit.sampling_seconds,\'update_and_logging_seconds\':audit.update_seconds,\n                \'evaluation_seconds\':audit.evaluation_seconds,\'worker_start_seconds\':worker_start,\n                \'active_steps_per_second\':model.num_timesteps/max(active,1e-9),\n                \'worker_stats\':worker_stats,\'action_counts\':audit.counts,\n                \'initialization\':initialization,\n                \'validation_target\':validation_target(job),\n                \'learner_sides\':job.get(\'frozen\',{}).get(\'learner_sides\',[1]),\n                \'opponent_sha256\':file_hash(job[\'frozen\'][\'bundle\']) if job[\'kind\']==\'train_frozen\' else None,\n                \'learner_parameters_changed\':True,\n                \'frozen_opponent_unchanged\':(all(row.get(\'frozen_opponent_unchanged\') is True for row in worker_stats) if job[\'kind\']==\'train_frozen\' else None),\n                \'export_reload_probability_error\':error,\'episodes\':episodes,\n                \'note\':\'PPO step budget rounds up to a full rollout. Speed excludes independent evaluation/startup/export. Best uses the explicitly reported target: CPU PvE or the fixed agent on the configured learner ports. A PvP best may remain at step zero. A complete run is not proof of a winning baseline.\'}\n    finally:\n        if model is not None and not completed:\n            try:model.save(out/\'interrupted_policy.zip\')\n            except Exception:pass\n        if vec is not None:vec.close()\n\n\ndef pvp(job,out):\n    from sf2_game import Game\n    from sf2_observation import Side\n    from sf2_video import Video\n    cfg=job[\'pvp\'];actors=[];game=None;rec=None;rows=[];videos=[]\n    try:\n        for key in (\'agent_a\',\'agent_b\'):\n            actors.append(LoadedAgent(cfg[key],trusted=job.get(\'trust_packages\',False),device=\'cpu\'))\n        for a in actors:\n            if rules_signature(a.common)!=rules_signature(job[\'common\']):raise ValueError(\'Both agents must use the same arena rules.\')\n        before=[parameter_hash(a.model) for a in actors]\n        order=[(0,1),(1,0)] if cfg.get(\'swap_sides\',True) else [(0,1)]\n        rounds=int(cfg.get(\'rounds_per_side\',1))\n        if not 1<=rounds<=20:raise ValueError(\'rounds_per_side must be 1..20.\')\n        game=Game(job[\'common\'],out/\'game\')\n        for pair in order:\n            a,b=(actors[i] for i in pair)\n            for rep in range(rounds):\n                game.open(a.spec[\'character\'],b.spec[\'character\'],\'pvp\',1)\n                sides=[Side(a.spec,1,job[\'common\'][\'position_scale\']),Side(b.spec,2,job[\'common\'][\'position_scale\'])]\n                for s in sides:s.reset(game.rgb,frame_id=game.elapsed)\n                commands=[compiled(a.spec[\'actions\']),compiled(b.spec[\'actions\'])]\n                rec=Video(out/f\'pvp_{len(rows):02d}.mp4\',game.fps)\n                game.on_frame=lambda rgb,raw,k1,k2,frame:rec.write(rgb,\n                    [f\'PvP | P1={a.spec["team_name"]} | P2={b.spec["team_name"]}\',\n                     f\'P1 HP={raw["p1_hp"]} | P2 HP={raw["p2_hp"]} | frame={frame}\',\n                     \'NO LEARNING | synchronous decisions; no wall-time tournament deadline.\'],k1,k2)\n                winner=reason=None\n                while reason is None:\n                    # Both act on the same pre-step game state, even for identical characters.\n                    raw=dict(game.raw)\n                    ids=[a.act(sides[0],raw,bool(cfg.get(\'deterministic\',True))),\n                         b.act(sides[1],raw,bool(cfg.get(\'deterministic\',True)))]\n                    for k1,k2 in zip(commands[0][ids[0]],commands[1][ids[1]]):\n                        raw,winner,reason=game.step_keys(k1,k2)\n                        if reason:break\n                    for s in sides:s.update(game.rgb,frame_id=game.elapsed)\n                result=((\'A\' if pair[int(winner[-1])-1]==0 else \'B\') if winner in (\'p1\',\'p2\') else (\'cutoff\' if reason==\'frame_cap\' else winner))\n                rows.append({\'p1_agent\':\'A\' if pair[0]==0 else \'B\',\'p2_agent\':\'A\' if pair[1]==0 else \'B\',\n                             \'winner\':result,\'end_reason\':reason,\'p1_hp\':raw[\'p1_hp\'],\'p2_hp\':raw[\'p2_hp\'],\n                             \'native_frames\':game.elapsed})\n                videos.append(rec.finish());rec=None;game.on_frame=None\n        if before!=[parameter_hash(a.model) for a in actors]:raise RuntimeError(\'PvP altered policy parameters.\')\n        return {\'status\':\'PVP_COMPLETE\',\'training_performed\':False,\'rounds\':rows,\'videos\':videos,\n                \'note\':\'Port reassignment is checked by the interface; P1 training does not guarantee P2 skill. No latency deadline is enforced.\'}\n    finally:\n        if rec:rec.close()\n        if game:game.close()\n        for a in actors:a.close()\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--job\',required=True);args=p.parse_args()\n    job=json.loads(Path(args.job).read_text());out=Path(job[\'out\']);out.mkdir(parents=True,exist_ok=True)\n    json_write(out/\'job.json\',job);t=time.perf_counter();code=1\n    try:\n        import torch,cv2\n        torch.set_num_threads(int(job[\'train\'].get(\'torch_threads\',2)));cv2.setNumThreads(1)\n        fn={\'check\':check,\'actions\':actions_preview,\'train\':train,\'train_frozen\':train,\n            \'evaluate\':evaluate,\'pvp\':pvp,\'check_frozen\':check_frozen,\'evaluate_frozen\':evaluate_frozen}[job[\'kind\']]\n        report=fn(job,out);code=0\n    except BaseException as exc:\n        report={\'status\':\'ERROR\',\'error\':repr(exc),\'traceback\':traceback.format_exc()}\n        print(report[\'traceback\'],flush=True)\n    report.update(wall_seconds=time.perf_counter()-t,runtime_versions=runtime_versions())\n    json_write(out/\'report.json\',report);print(\'REPORT=\'+str(out/\'report.json\'),flush=True)\n    raise SystemExit(code)\n\nif __name__==\'__main__\':main()\n', 'setup_sf2.py': '"""Install only missing dependencies and locate/import the original course ROM.\nNo external state download. Match-start snapshots are generated at runtime.\nROM is not included in this package.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport importlib\nimport importlib.metadata\nimport importlib.util\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport urllib.request\n\nROM_SHA1=\'a5aad1d108046d9388e33247610dafb4c6516e0b\'\nGAME=\'StreetFighterIISpecialChampionEdition-Genesis\'\n# Same external ROM URL that was already present in the user\'s original v8.0.\nLEGACY_URL=\'https://archive.org/download/street-fighter-ii-special-champion-edition-usa/Street%20Fighter%20II%27%20-%20Special%20Champion%20Edition%20%28USA%29.md\'\n\n\ndef retro_requirement_for_python(version):\n    """Select a published wheel recipe; never alter the notebook interpreter.\n\n    Preserve the original 0.9.5 recipe on older runtimes. Version 1.0.1\n    publishes CPython 3.13/3.14 wheels; 0.9.5 explicitly excludes Python 3.13.\n    This selects an installer, not a certificate of game/model compatibility.\n    """\n    version = tuple(version[:2])\n    if (3, 8) <= version < (3, 13):\n        return \'stable-retro==0.9.5\'\n    if (3, 13) <= version < (3, 15):\n        return \'stable-retro==1.0.1\'\n    raise RuntimeError(\n        \'No installation recipe is defined for Python \' + \'.\'.join(map(str, version))\n        + \'. Keep the full version report; do not remove this check or replace system Python.\'\n    )\n\n\ndef setup(directory, rom_file=\'\', allow_legacy_fetch=False, install_sb3=False):\n    folder=Path(directory).resolve();folder.mkdir(parents=True,exist_ok=True)\n    print(\'Setup Python:\', sys.version, \'| executable:\', sys.executable, flush=True)\n    if not (importlib.util.find_spec(\'retro\') or importlib.util.find_spec(\'stable_retro\')):\n        requirement = retro_requirement_for_python(sys.version_info)\n        print(\'Installing emulator dependency:\', requirement, flush=True)\n        # Require a published emulator wheel instead of silently building C++ from source.\n        # Use the SAME interpreter that runs the notebook and its worker processes.\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'install\',\n                        \'--only-binary=stable-retro\', requirement], check=True)\n        importlib.invalidate_caches()\n    for module,package in [(\'cv2\',\'opencv-python-headless\'),(\'PIL\',\'pillow\')]:\n        if importlib.util.find_spec(module) is None:\n            subprocess.run([sys.executable,\'-m\',\'pip\',\'install\',package],check=True)\n    if install_sb3 and importlib.util.find_spec(\'stable_baselines3\') is None:\n        import gymnasium\n        from packaging.version import Version\n        version=Version(gymnasium.__version__)\n        if version < Version(\'0.29\'):\n            raise RuntimeError(\'Gymnasium is older than 0.29; inspect compatibility before installing SB3.\')\n        package=\'stable-baselines3==2.3.2\' if version < Version(\'1.0\') else \'stable-baselines3==2.7.0\'\n        subprocess.run([sys.executable,\'-m\',\'pip\',\'install\',package],check=True)\n    try:\n        import stable_retro as backend\n        module_name=\'stable_retro\'\n    except ModuleNotFoundError as exc:\n        if exc.name!=\'stable_retro\':raise\n        import retro as backend\n        module_name=\'retro\'\n    print(\'Emulator import:\', module_name, \'| module:\', getattr(backend, \'__file__\', \'unknown\'), flush=True)\n    try:\n        print(\'stable-retro version:\', importlib.metadata.version(\'stable-retro\'), flush=True)\n    except importlib.metadata.PackageNotFoundError:\n        print(\'WARNING: stable-retro distribution metadata was not found. \'\n              \'Review the backend before starting any training or arena test.\', flush=True)\n    def installed():\n        # The course identity stays unchanged; only the backend integration ID\n        # differs. Check these two known aliases, never an arbitrary game/version.\n        for game_id in (GAME, GAME+\'-v0\'):\n            for method in (\'get_original_romfile_path\',\'get_romfile_path\'):\n                fn=getattr(backend.data,method,None)\n                if fn:\n                    try:\n                        p=Path(fn(game_id))\n                        if p.is_file() and hashlib.sha1(p.read_bytes()).hexdigest()==ROM_SHA1:\n                            print(\'Resolved game ID:\', game_id, flush=True)\n                            return p\n                    except (OSError,RuntimeError,ValueError,TypeError):pass\n        return None\n    path=installed()\n    if path is None:\n        if rom_file:\n            candidate=Path(rom_file).expanduser().resolve()\n            if not candidate.is_file():raise FileNotFoundError(candidate)\n        else:\n            candidate=None\n            for p in Path(\'/kaggle/input\').rglob(\'*\'):\n                if p.is_file() and p.suffix.lower() in (\'.md\',\'.bin\',\'.gen\') and p.stat().st_size<8*1024**2:\n                    if hashlib.sha1(p.read_bytes()).hexdigest()==ROM_SHA1:\n                        candidate=p;break\n            if candidate is None and allow_legacy_fetch:\n                target=folder/\'StreetFighterIISpecialChampionEdition-Genesis.md\'\n                print(\'Trying the original notebook ROM source; no state download.\',flush=True)\n                try:\n                    req=urllib.request.Request(LEGACY_URL,headers={\'User-Agent\':\'Mozilla/5.0\'})\n                    with urllib.request.urlopen(req,timeout=90) as response:\n                        data=response.read(8*1024**2+1)\n                    if len(data)>8*1024**2 or hashlib.sha1(data).hexdigest()!=ROM_SHA1:\n                        raise RuntimeError(\'Downloaded bytes are not the expected ROM; file not imported.\')\n                    target.write_bytes(data);candidate=target\n                except Exception as exc:\n                    raise RuntimeError(\'Original ROM source unavailable. Attach your matching game ROM and set ROM_FILE. \'\n                                       \'A .state file is not required and cannot substitute for the ROM.\') from exc\n            if candidate is None:\n                raise FileNotFoundError(\'Provide ROM_FILE or attach your matching game ROM; no .state is needed.\')\n        if hashlib.sha1(candidate.read_bytes()).hexdigest()!=ROM_SHA1:\n            raise RuntimeError(\'Provided ROM does not match the required SHA1.\')\n        # Dedicated folder avoids importing every file from the user\'s input directory.\n        import shutil\n        import_dir=folder/\'rom_import\';import_dir.mkdir(exist_ok=True)\n        local=import_dir/\'StreetFighterIISpecialChampionEdition-Genesis.md\'\n        if candidate.resolve()!=local.resolve():shutil.copy2(candidate,local)\n        subprocess.run([sys.executable,\'-m\',module_name+\'.import\',str(import_dir)],check=True)\n        path=installed()\n    if path is None:raise RuntimeError(\'ROM import finished, but neither \'+GAME+\' nor \'+GAME+\'-v0 resolves to the required SHA1. Keep the full setup log.\')\n    print(\'ROM verified:\',path,flush=True)\n    print(\'SHA1:\',ROM_SHA1,flush=True)\n    print(\'Resources ready. Match-start caches will be generated separately; no PPO training yet.\',flush=True)\n    return path\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--directory\',required=True)\n    p.add_argument(\'--rom-file\',default=\'\');p.add_argument(\'--legacy-fetch\',action=\'store_true\')\n    p.add_argument(\'--sb3\',action=\'store_true\');a=p.parse_args()\n    setup(a.directory,a.rom_file,a.legacy_fetch,a.sb3)\n', 'sf2_actions.py': '"""Fixed six-frame actions. Ordinary PPO: no persistent queues or action masks."""\nfrom __future__ import annotations\nimport hashlib, json, re\nimport numpy as np\n\nQUANTUM = 6\nKEYS = frozenset((\'LEFT\',\'RIGHT\',\'UP\',\'DOWN\',\'A\',\'B\',\'C\',\'X\',\'Y\',\'Z\'))\nBASIC_NAMES = (\'NO_OP\',\'LEFT\',\'RIGHT\',\'DOWN\',\'UP\',\'UP_LEFT\',\'UP_RIGHT\',\n               \'DOWN_LEFT\',\'DOWN_RIGHT\',\'A\',\'B\',\'C\',\'X\',\'Y\',\'Z\')\nBASIC_KEYS = ((),(\'LEFT\',),(\'RIGHT\',),(\'DOWN\',),(\'UP\',),(\'UP\',\'LEFT\'),\n              (\'UP\',\'RIGHT\'),(\'DOWN\',\'LEFT\'),(\'DOWN\',\'RIGHT\'),\n              (\'A\',),(\'B\',),(\'C\',),(\'X\',),(\'Y\',),(\'Z\',))\n\ndef validate_library(actions):\n    if not isinstance(actions, list) or not 1 <= len(actions) <= 64:\n        raise ValueError(\'Provide 1..64 actions.\')\n    names = set()\n    for action in actions:\n        if set(action) != {\'name\',\'segments\'}:\n            raise ValueError(\'An action needs exactly name and segments.\')\n        name = action[\'name\']\n        if not isinstance(name, str) or not re.fullmatch(r\'[A-Za-z][A-Za-z0-9_]{0,47}\',name) or name in names:\n            raise ValueError(\'Action names must be unique short English identifiers.\')\n        names.add(name)\n        if not isinstance(action[\'segments\'],list) or not action[\'segments\']:\n            raise ValueError(\'Empty action sequence.\')\n        total = 0\n        for part in action[\'segments\']:\n            if set(part) != {\'keys\',\'frames\'}:\n                raise ValueError(\'Each segment needs keys and frames.\')\n            k = part[\'keys\']; duration = part[\'frames\']\n            if not isinstance(k,list) or any(not isinstance(x,str) for x in k):\n                raise ValueError(\'keys must be a list of controller button names.\')\n            if len(set(k)) != len(k) or set(k)-KEYS:\n                raise ValueError(\'Only combat keys are supported; no START, MODE, FWD/BACK or RAM writes.\')\n            if {\'LEFT\',\'RIGHT\'} <= set(k) or {\'UP\',\'DOWN\'} <= set(k):\n                raise ValueError(\'Conflicting directions.\')\n            if type(duration) is not int or not 1 <= duration <= QUANTUM:\n                raise ValueError(\'Each segment lasts 1..6 native frames.\')\n            total += duration\n        if total != QUANTUM:\n            raise ValueError(\'Every action must finish in exactly 6 frames. This demo has no long-macro queue.\')\n    return actions\n\ndef library(custom=()):\n    """The 15 course basics, optionally followed by student-defined short actions."""\n    if not isinstance(custom,(list,tuple)):\n        raise ValueError(\'CUSTOM_ACTIONS must be a list.\')\n    basic = [{\'name\':n,\'segments\':[{\'keys\':list(k),\'frames\':QUANTUM}]}\n             for n,k in zip(BASIC_NAMES,BASIC_KEYS)]\n    return validate_library(basic + json.loads(json.dumps(list(custom))))\n\ndef compiled(actions):\n    result = []\n    for a in validate_library(actions):\n        result.append(tuple(tuple(p[\'keys\']) for p in a[\'segments\'] for _ in range(p[\'frames\'])))\n    return tuple(result)\n\ndef action_index(action,n):\n    a = np.asarray(action)\n    if a.size != 1 or a.dtype.kind not in (\'i\',\'u\'):\n        raise ValueError(\'The policy must return one integer action ID.\')\n    value = int(a.item())\n    if not 0 <= value < n:\n        raise ValueError(\'Action ID is outside this agent action table.\')\n    return value\n\ndef signature(actions):\n    data = {\'protocol\':\'fixed-sixframe-v2\',\'actions\':validate_library(actions)}\n    return hashlib.sha256(json.dumps(data,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest()\n', 'sf2_arena.py': '"""Public practice arena using the classroom game engine and isolated actor RPC.\nNo student reward is called. No learner is optimized. No arbitrary student game\nwrapper is accepted. This is a reviewed-code practice runner, NOT a secure sandbox.\n"""\nfrom __future__ import annotations\nimport argparse\nimport base64\nfrom collections import Counter\nimport datetime as dt\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport sys\nimport time\nimport traceback\nimport zipfile\nimport numpy as np\n\nfrom sf2_actions import action_index,compiled\nfrom sf2_common import json_write,runtime_versions\nfrom sf2_submission import inspect_submission,load_rules,rules_hash,_public_check,digest\nfrom sf2_arena_rpc import ActorProcess\n\n\ndef prepare_frame(rgb,raw,frame):\n    rgb=np.asarray(rgb)\n    if rgb.shape!=(224,320,3) or rgb.dtype!=np.uint8:\n        raise ValueError(\'Arena game must provide raw 224x320 RGB uint8.\')\n    # Do not expose RAM, opponent actions/weights or result functions to actor RPC.\n    coordinates={k:int(raw[k]) for k in (\'p1_x\',\'p1_y\',\'p2_x\',\'p2_y\')}\n    return {\'rgb_b64\':base64.b64encode(rgb.tobytes()).decode(\'ascii\'),\n            \'coordinates\':coordinates,\'frame\':int(frame)}\n\n\ndef latency_summary(values):\n    if not values:return {\'requests\':0}\n    return {\'requests\':len(values),\'mean_ms\':float(np.mean(values))*1000,\n            \'p95_ms\':float(np.percentile(values,95))*1000,\'max_ms\':float(max(values))*1000}\n\n\ndef play_matches(infos,rule,out,*,offsets,swap_sides,seed=42,record_video=True,\n                 max_video_rounds=2,game_factory=None,actor_factory=ActorProcess):\n    """Both policies receive the same frame before either set of buttons is applied."""\n    if not offsets or len(offsets)>20 or any(type(x)is not int or not 0<=x<=60 for x in offsets):\n        raise ValueError(\'Use 1..20 startup offsets, each an integer from 0 to 60.\')\n    if len(infos)!=2:raise ValueError(\'Exactly two submissions are required.\')\n    if game_factory is None:\n        from sf2_game import Game\n        game_factory=Game\n    from sf2_video import Video\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    actors=[];game=None;rec=None;rows=[];videos=[];counts=[Counter(),Counter()]\n    traces=out/\'decisions.jsonl\'\n    commands=[compiled(x[\'spec\'][\'actions\']) for x in infos]\n    initial_hashes=[]\n    try:\n        for i,info in enumerate(infos):\n            a=actor_factory(info[\'agent_path\'],out/f\'actor_{"AB"[i]}\',rule[\'inference\'])\n            actors.append(a);initial_hashes.append(a.ready[\'parameter_hash\'])\n            if a.ready.get(\'action_count\')!=len(commands[i]):raise ValueError(\'Actor loaded wrong action count.\')\n        game=game_factory(rule[\'common\'],out/\'game\')\n        orders=[(0,1),(1,0)] if swap_sides else [(0,1)]\n        with traces.open(\'w\',encoding=\'utf-8\') as trace:\n            for order in orders:\n                for index,offset in enumerate(offsets):\n                    p1,p2=(infos[i] for i in order)\n                    game.open(p1[\'spec\'][\'character\'],p2[\'spec\'][\'character\'],\'pvp\',1)\n                    for _ in range(offset):\n                        raw,winner,reason=game.step_keys((),())\n                        if reason:raise RuntimeError(\'Round ended during the configured start offset.\')\n                    rid=len(rows);payload=prepare_frame(game.rgb,dict(game.raw),game.elapsed)\n                    for port,idx in enumerate(order,1):\n                        res=actors[idx].request(\'reset\',player=port,seed=int(seed)+index*100+idx,**payload)\n                        if res.get(\'status\')!=\'reset\' or res.get(\'player\')!=port:\n                            raise ValueError(\'Actor reset returned a wrong controller port.\')\n                    if record_video and index == 0 and len(videos)<max_video_rounds:\n                        rec=Video(out/f\'round_{rid:02d}.mp4\',game.fps)\n                        def on_frame(rgb,raw,k1,k2,frame):\n                            rec.write(rgb,[f\'P1 {p1["metadata"]["team_id"]} | P2 {p2["metadata"]["team_id"]}\',\n                                f\'HP {raw["p1_hp"]}/{raw["p2_hp"]} | native frame={frame}\',\n                                \'PRACTICE | CPU inference | NO TRAINING | six-frame decisions\'],k1,k2)\n                        game.on_frame=on_frame\n                    winner=reason=None;decisions=0\n                    while reason is None:\n                        frame=game.elapsed\n                        payload=prepare_frame(game.rgb,dict(game.raw),frame)\n                        ids=[]\n                        # Independent JSON processes; neither receives the other\'s chosen action.\n                        for port,idx in enumerate(order,1):\n                            res=actors[idx].request(\'act\',round_id=rid,**payload)\n                            if res.get(\'status\')!=\'action\' or res.get(\'frame\')!=frame or res.get(\'player\')!=port:\n                                raise ValueError(\'Actor action has wrong frame/controller identity.\')\n                            a=action_index(res[\'action\'],len(commands[idx]));ids.append(a);counts[idx][a]+=1\n                        trace.write(json.dumps({\'round\':rid,\'frame\':frame,\'offset\':offset,\n                            \'p1_agent\':\'AB\'[order[0]],\'p2_agent\':\'AB\'[order[1]],\n                            \'p1_action\':ids[0],\'p2_action\':ids[1]},separators=(\',\',\':\'))+\'\\n\')\n                        for k1,k2 in zip(commands[order[0]][ids[0]],commands[order[1]][ids[1]]):\n                            raw,winner,reason=game.step_keys(k1,k2)\n                            if reason:break\n                        decisions+=1\n                    end_hashes=[]\n                    for a in actors:\n                        result=a.request(\'finish\')\n                        if result.get(\'status\')!=\'unchanged\':raise RuntimeError(\'Actor did not confirm unchanged model.\')\n                        end_hashes.append(result[\'parameter_hash\'])\n                    if end_hashes!=initial_hashes:raise RuntimeError(\'An actor changed policy parameters during inference.\')\n                    if reason==\'frame_cap\':outcome=\'cutoff\'\n                    elif winner in (\'p1\',\'p2\'):outcome=\'AB\'[order[int(winner[-1])-1]]\n                    elif winner==\'draw\':outcome=\'draw\'\n                    else:raise RuntimeError(\'Unexpected winner/result from the public game engine.\')\n                    row={\'round\':rid,\'offset\':offset,\'p1_agent\':\'AB\'[order[0]],\'p2_agent\':\'AB\'[order[1]],\n                         \'winner\':outcome,\'end_reason\':reason,\'p1_hp\':raw[\'p1_hp\'],\'p2_hp\':raw[\'p2_hp\'],\n                         \'native_frames\':game.elapsed,\'decisions\':decisions,\'weights_unchanged\':True}\n                    rows.append(row)\n                    json_write(out/\'rounds_partial.json\',rows)\n                    print(f\'ARENA round={rid} P1={row["p1_agent"]} P2={row["p2_agent"]} offset={offset}: \'\n                          f\'{outcome}, HP={row["p1_hp"]}/{row["p2_hp"]}, reason={reason}\',flush=True)\n                    if rec is not None:\n                        videos.append(rec.finish());rec=None;game.on_frame=None\n        totals=Counter(row[\'winner\'] for row in rows)\n        return {\'rounds\':rows,\'videos\':videos,\'summary\':{k:totals[k] for k in (\'A\',\'B\',\'draw\',\'cutoff\')},\n            \'training_performed\':False,\'policy_weights_unchanged\':True,\n            \'actor_initial_hashes\':initial_hashes,\n            \'actor_runtime_versions\':[a.ready.get(\'runtime_versions\',{}) for a in actors],\n            \'latency\':{name:latency_summary(actors[i].times) for i,name in enumerate(\'AB\')},\n            \'action_counts\':{name:{infos[i][\'spec\'][\'actions\'][k][\'name\']:v for k,v in counts[i].items()} for i,name in enumerate(\'AB\')},\n            \'same_package_interface_test\':infos[0][\'agent_sha256\']==infos[1][\'agent_sha256\'],\n            \'warnings\':[\'One or more rounds reached the safety frame cap; not scored as wins.\'] if totals[\'cutoff\'] else [],\n            \'coverage\':\'Only these submitted agents, character matchup, offsets and ports were tested.\',\n            \'note\':\'Self-match is an interface check, not a skill benchmark. Rewards are never used for match scoring. Cutoffs are not wins.\'}\n    finally:\n        if rec is not None:rec.close()\n        if game is not None:game.close()\n        for a in actors:a.close()\n\n\ndef framework_hashes(folder):\n    return {p.name:digest(p.read_bytes()) for p in sorted(Path(folder).glob(\'*.py\'))}\n\n\ndef run_job(job):\n    out=Path(job[\'out\']);out.mkdir(parents=True,exist_ok=True)\n    rule=load_rules(job[\'rules_path\'])\n    if job.get(\'trust_submissions\') is not True:\n        raise ValueError(\'Review custom Python/serialized weights, then set TRUST_SUBMISSIONS=True. Process separation is not a sandbox.\')\n    infos=[]\n    for i,key in enumerate((\'submission_a\',\'submission_b\')):\n        infos.append(inspect_submission(job[key],rule,out/\'inputs\'/f\'agent_{"AB"[i]}.zip\'))\n    for i,info in enumerate(infos):\n        json_write(out/f\'inspection_{"AB"[i]}.json\',{k:v for k,v in info.items() if k not in (\'spec\',\'manifest\')})\n    result=play_matches(infos,rule,out,offsets=job[\'offsets\'],swap_sides=job[\'swap_sides\'],\n                        seed=int(job.get(\'seed\',42)),record_video=job.get(\'record_video\',True),max_video_rounds=2)\n    success=\'ARENA_SELF_CHECK_PASSED\' if job[\'mode\']==\'self_check\' else \'ARENA_MATCH_COMPLETE\'\n    result.update(status=success,rules_id=rule[\'rules_id\'],rules_sha256=rules_hash(rule),\n        submissions={name:{\'team_id\':x[\'metadata\'][\'team_id\'],\'submission_sha256\':x[\'submission_sha256\'],\n                          \'agent_sha256\':x[\'agent_sha256\'],\'character\':x[\'spec\'][\'character\']}\n                     for name,x in zip(\'AB\',infos)},\n        framework_hashes=framework_hashes(Path(__file__).parent),\n        official_score=False,security_sandbox=False,\n        rule_status=rule[\'status\'],timing_scope=\'CPU actor RPC including preprocessing, inference and transport; early diagnostic snapshots may add cost.\')\n    # Receipt is OUTSIDE submission.zip: no circular checksum or post-check model mutation.\n    json_write(out/\'receipt.json\',{k:v for k,v in result.items() if k not in (\'videos\',\'action_counts\')})\n    return result\n\n\ndef make_diagnostic_zip(out):\n    out=Path(out);target=out/\'arena_diagnostics.zip\'\n    with zipfile.ZipFile(target,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.rglob(\'*\')):\n            rel=p.relative_to(out)\n            if not p.is_file() or p==target or rel.parts[0] in (\'inputs\',\'submission_inputs\') or \'__pycache__\' in rel.parts:\n                continue\n            if p.suffix.lower() not in (\'.json\',\'.jsonl\',\'.log\',\'.png\',\'.npz\',\'.mp4\'):continue\n            z.write(p,rel.as_posix())\n    return target\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--job\',required=True);args=p.parse_args()\n    job=json.loads(Path(args.job).read_text());out=Path(job[\'out\']);out.mkdir(parents=True,exist_ok=True)\n    start=time.perf_counter();code=1\n    try:report=run_job(job);code=0\n    except BaseException as exc:\n        report={\'status\':\'ARENA_CHECK_FAILED\',\'error\':type(exc).__name__+\': \'+str(exc),\n                \'traceback\':traceback.format_exc(),\'official_score\':False,\'security_sandbox\':False}\n        print(report[\'traceback\'],flush=True)\n    report.update(wall_seconds=time.perf_counter()-start,runtime_versions=runtime_versions())\n    json_write(out/\'report.json\',report)\n    print(\'STATUS:\',report[\'status\'],flush=True);print(\'REPORT=\'+str(out/\'report.json\'),flush=True)\n    print(\'DIAGNOSTICS=\'+str(make_diagnostic_zip(out)),flush=True)\n    return code\n\nif __name__==\'__main__\':raise SystemExit(main())\n', 'sf2_arena_actor.py': '"""One reviewed agent in a separate CPU process. JSON/base64 RPC, no pickle IPC.\nThis is process separation for compatibility/diagnostics, NOT a security sandbox.\nThe actor receives RGB + four approved coordinates, never an emulator handle.\n"""\nfrom __future__ import annotations\nimport argparse\nimport base64\nfrom contextlib import redirect_stdout\nimport json\nfrom pathlib import Path\nimport sys\nimport traceback\n\nMAX_REQUEST = 512 * 1024\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--agent\',required=True);p.add_argument(\'--work\',required=True)\n    args=p.parse_args();work=Path(args.work);work.mkdir(parents=True,exist_ok=True)\n    wire=sys.stdout\n    def reply(value):\n        wire.write(json.dumps(value,separators=(\',\',\':\'),allow_nan=False)+\'\\n\');wire.flush()\n    actor=None;side=None;initial_hash=None\n    try:\n        with redirect_stdout(sys.stderr):\n            import numpy as np\n            import torch\n            import cv2\n            torch.set_num_threads(1);cv2.setNumThreads(1)\n            from sf2_bundle import LoadedAgent\n            from sf2_observation import Side, display_frames\n            from sf2_common import parameter_hash,runtime_versions\n            from sf2_actions import action_index\n            actor=LoadedAgent(args.agent,trusted=True,device=\'cpu\')\n            actor.model.policy.set_training_mode(False)\n            for t in actor.model.policy.state_dict().values():\n                if t.is_floating_point() and not torch.isfinite(t).all():\n                    raise ValueError(\'Non-finite model weights.\')\n            initial_hash=parameter_hash(actor.model)\n            # Warm up preprocessing and inference, not a game acceptance test.\n            probe=Side(actor.spec,1,actor.common[\'position_scale\'])\n            probe.reset(np.zeros((224,320,3),np.uint8),frame_id=0)\n            raw={\'p1_x\':100,\'p1_y\':192,\'p2_x\':200,\'p2_y\':192}\n            with torch.inference_mode():\n                pred,_=actor.model.predict(probe.observe(raw),deterministic=True)\n                action_index(pred,len(actor.spec[\'actions\']))\n            if parameter_hash(actor.model)!=initial_hash:\n                raise RuntimeError(\'Warm-up changed model parameters/buffers.\')\n        reply({\'status\':\'ready\',\'parameter_hash\':initial_hash,\'runtime_versions\':runtime_versions(),\n               \'image_shape\':list(probe.observe(raw)[\'image\'].shape) if isinstance(probe.observe(raw),dict) else list(probe.observe(raw).shape),\n               \'action_count\':len(actor.spec[\'actions\'])})\n        for _ in range(100000):\n            line=sys.stdin.buffer.readline(MAX_REQUEST+1)\n            if not line:break\n            if len(line)>MAX_REQUEST or not line.endswith(b\'\\n\'):\n                raise ValueError(\'Oversized/malformed request.\')\n            req=json.loads(line);op=req.get(\'op\');seq=req.get(\'seq\')\n            with redirect_stdout(sys.stderr):\n                if op==\'close\':\n                    reply({\'seq\':seq,\'status\':\'closed\'});break\n                if op==\'finish\':\n                    h=parameter_hash(actor.model)\n                    if h!=initial_hash:raise RuntimeError(\'Inference changed policy parameters/buffers.\')\n                    reply({\'seq\':seq,\'status\':\'unchanged\',\'parameter_hash\':h});continue\n                if op not in (\'reset\',\'act\'):raise ValueError(\'Unknown actor operation.\')\n                blob=base64.b64decode(req[\'rgb_b64\'],validate=True)\n                if len(blob)!=224*320*3:raise ValueError(\'Wrong raw RGB byte count.\')\n                rgb=np.frombuffer(blob,np.uint8).reshape(224,320,3).copy()\n                raw=req[\'coordinates\']\n                if set(raw)!={\'p1_x\',\'p1_y\',\'p2_x\',\'p2_y\'}:raise ValueError(\'Only declared coordinates are accepted.\')\n                if any(type(v) is not int or not 0<=v<=65535 for v in raw.values()):\n                    raise ValueError(\'Invalid coordinates.\')\n                frame=req[\'frame\']\n                if type(frame) is not int or frame<0:raise ValueError(\'Invalid frame ID.\')\n                if op==\'reset\':\n                    player=req[\'player\'];seed=req[\'seed\']\n                    if player not in (1,2):raise ValueError(\'Invalid controller port.\')\n                    import random\n                    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n                    side=Side(actor.spec,player,actor.common[\'position_scale\']);side.reset(rgb,frame_id=frame)\n                    save_index=0\n                    reply({\'seq\':seq,\'status\':\'reset\',\'player\':player,\'frame\':frame});continue\n                if side is None:raise RuntimeError(\'Reset the actor before requesting an action.\')\n                if frame>side.frame_ids[-1]:side.update(rgb,frame_id=frame)\n                elif frame!=side.frame_ids[-1]:raise ValueError(\'Frame IDs moved backwards.\')\n                observation=side.observe(raw)\n                # Save early post-reset and real motion-history inputs for student diagnostics.\n                if save_index in (0,3):\n                    image=observation[\'image\'] if isinstance(observation,dict) else observation\n                    label=f\'P{side.player}_{req.get("round_id",0):02d}_{save_index:02d}\'\n                    np.savez_compressed(work/(label+\'.npz\'), image=image,\n                        position=observation.get(\'position\',np.zeros(0,np.float32)) if isinstance(observation,dict) else np.zeros(0,np.float32),\n                        frame_ids=np.array(side.frame_ids))\n                    tiles=display_frames(image,actor.spec)\n                    tiles=[np.repeat(t[:,:,None],3,axis=2) if t.ndim==2 else t for t in tiles]\n                    cv2.imwrite(str(work/(label+\'.png\')),cv2.cvtColor(np.concatenate(tiles,axis=1),cv2.COLOR_RGB2BGR))\n                save_index+=1\n                with torch.inference_mode():\n                    pred,_=actor.model.predict(observation,deterministic=True)\n                    index=action_index(pred,len(actor.spec[\'actions\']))\n                reply({\'seq\':seq,\'status\':\'action\',\'action\':index,\'player\':side.player,\'frame\':frame})\n        else:raise RuntimeError(\'Actor request budget exhausted.\')\n    except BaseException as exc:\n        traceback.print_exc(file=sys.stderr)\n        try:reply({\'status\':\'error\',\'error\':type(exc).__name__+\': \'+str(exc)})\n        except Exception:pass\n        return 1\n    finally:\n        if actor is not None:actor.close()\n    return 0\n\nif __name__==\'__main__\':raise SystemExit(main())\n', 'sf2_arena_notebook.py': '"""Shared launcher used by training-notebook self-check and standalone arena."""\nfrom pathlib import Path\nimport datetime as dt\nimport json\nimport os\nimport signal\nimport shutil\nimport subprocess\nimport sys\n\nfrom sf2_submission import inspect_submission,load_rules,digest,rules_hash\n\n\ndef inspect_for_display(path,rules_path):\n    info=inspect_submission(path,load_rules(rules_path))\n    print(\'STATIC_SUBMISSION_CHECK_PASSED (no model code executed)\')\n    print(\'Team:\',info[\'metadata\'][\'team_id\'],\'| Character:\',info[\'character\'])\n    print(\'Input:\',info[\'image_shape\'],\'| Actions:\',info[\'action_count\'])\n    print(\'Submission SHA256:\',info[\'submission_sha256\'])\n    print(\'Agent SHA256:\',info[\'agent_sha256\'])\n    print(\'Training versions:\',info[\'runtime_versions_at_training\'])\n    return {k:v for k,v in info.items() if k not in (\'spec\',\'manifest\')}\n\n\ndef run_arena(root,*,submission_a,submission_b=\'\',mode=\'self_check\',trust_submissions=False,\n              rules_path=None,offsets=None,record_video=True):\n    if mode not in (\'self_check\',\'match\'):raise ValueError(\'mode must be self_check or match.\')\n    if trust_submissions is not True:\n        raise ValueError(\'Review all packages, then explicitly set TRUST_SUBMISSIONS=True. Checksums/processes are not a sandbox.\')\n    root=Path(root).resolve();code=root/\'code\'\n    rules_path=Path(rules_path or root/\'arena_rules_v1.json\').resolve();rule=load_rules(rules_path)\n    if mode==\'self_check\':\n        if submission_b and Path(submission_b).resolve()!=Path(submission_a).resolve():\n            raise ValueError(\'Self-check uses A twice. Leave submission_b empty, or choose mode="match".\')\n        submission_b=submission_a\n        if offsets is not None and list(offsets)!=rule[\'selfcheck_offsets\']:\n            raise ValueError(\'Self-check offsets must match the published practice rules.\')\n        offsets=rule[\'selfcheck_offsets\']\n    elif not submission_b:raise ValueError(\'Fill submission_b for a two-team match.\')\n    if offsets is None:offsets=rule[\'selfcheck_offsets\']\n    # Fail before creating children if either container is invalid.\n    checks=[inspect_submission(p,rule) for p in (submission_a,submission_b)]\n    stamp=dt.datetime.now(dt.timezone.utc).strftime(\'%Y%m%dT%H%M%S_%f\')\n    out=root/\'arena_runs\'/f\'{stamp}_{mode}\';out.mkdir(parents=True)\n    inputs=out/\'submission_inputs\';inputs.mkdir()\n    paths=[]\n    for name,p,info in zip(\'AB\',(submission_a,submission_b),checks):\n        target=inputs/f\'{name}_submission.zip\';shutil.copy2(p,target)\n        if digest(target.read_bytes())!=info[\'submission_sha256\']:\n            raise RuntimeError(\'Submission changed while starting the check; retry with a fixed saved ZIP.\')\n        paths.append(str(target))\n    snapshot=out/\'rules.json\';snapshot.write_text(json.dumps(rule,indent=2))\n    job={\'mode\':mode,\'out\':str(out),\'submission_a\':paths[0],\'submission_b\':paths[1],\n         \'rules_path\':str(snapshot),\'offsets\':list(offsets),\'swap_sides\':True,\'seed\':42,\n         \'record_video\':bool(record_video),\'trust_submissions\':True}\n    (out/\'job.json\').write_text(json.dumps(job,indent=2))\n    env=os.environ.copy();env.update(SF2_START_CACHE_DIR=str(root/\'cache\'/\'match_starts_v2\'),\n        OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\')\n    print(\'Arena:\',mode,\'| output:\',out,flush=True)\n    with (out/\'console.log\').open(\'w\') as log:\n        proc=subprocess.Popen([sys.executable,\'-u\',str(code/\'sf2_arena.py\'),\'--job\',str(out/\'job.json\')],\n            cwd=code,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)\n        try:\n            for line in proc.stdout:print(line,end=\'\');log.write(line);log.flush()\n            proc.wait()\n        except KeyboardInterrupt:\n            # Actor processes have their own cleanup in the arena driver\'s finally block.\n            os.killpg(proc.pid,signal.SIGINT)\n            try:proc.wait(timeout=20)\n            except subprocess.TimeoutExpired:os.killpg(proc.pid,signal.SIGKILL);proc.wait()\n            raise\n    path=out/\'report.json\'\n    if not path.is_file():\n        report={\'status\':\'ARENA_CHECK_FAILED\',\'error\':\'Driver exited without report. See console.log.\'}\n        path.write_text(json.dumps(report,indent=2))\n    else:report=json.loads(path.read_text())\n    # Repack AFTER console.log is closed so diagnostics include the complete tail.\n    from sf2_arena import make_diagnostic_zip\n    make_diagnostic_zip(out)\n    if proc.returncode or report.get(\'status\')==\'ARENA_CHECK_FAILED\':\n        print(\'FAILED: keep arena_diagnostics.zip with the unchanged submission ZIP.\')\n    return out,report\n\n\ndef show_arena_result(out,report):\n    from IPython.display import display,FileLink,Video,Image\n    out=Path(out);print(\'STATUS:\',report[\'status\'])\n    if report.get(\'summary\'):print(\'Round results:\',report[\'summary\'])\n    if report.get(\'latency\'):print(\'Latency:\',json.dumps(report[\'latency\'],indent=2))\n    if report.get(\'error\'):print(report[\'error\'])\n    for p in report.get(\'videos\',[]):\n        if p and Path(p).is_file():display(Video(p,embed=True,width=800))\n    for name in (\'report.json\',\'receipt.json\',\'arena_diagnostics.zip\'):\n        if (out/name).is_file():display(FileLink(str(out/name)))\n    print(\'No reward-based scoring, no training, no security certification. Self-check is not proof of skill.\')\n', 'sf2_arena_rpc.py': '"""Bounded JSON pipe transport for inference subprocesses (Linux/Kaggle).\nProcesses are NOT a security sandbox. No pickle objects are received from agents.\n"""\nfrom __future__ import annotations\nimport json\nimport os\nfrom pathlib import Path\nimport selectors\nimport signal\nimport subprocess\nimport sys\nimport time\n\nclass ActorProcess:\n    def __init__(self,agent,work,limits,*,command=None):\n        work=Path(work);work.mkdir(parents=True,exist_ok=True)\n        self.proc=None;self.seq=0;self.buffer=b\'\';self.times=[];self.limits=limits\n        self.log=(work/\'actor_stderr.log\').open(\'wb\')\n        env=os.environ.copy()\n        env.update(CUDA_VISIBLE_DEVICES=\'\',OMP_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\',PYTHONUNBUFFERED=\'1\')\n        cmd=command or [sys.executable,\'-u\',str(Path(__file__).with_name(\'sf2_arena_actor.py\')),\n                       \'--agent\',str(Path(agent).resolve()),\'--work\',str(work.resolve())]\n        try:\n            self.proc=subprocess.Popen(cmd,cwd=work,env=env,stdin=subprocess.PIPE,stdout=subprocess.PIPE,\n                stderr=self.log,start_new_session=True,bufsize=0)\n            os.set_blocking(self.proc.stdin.fileno(),False);os.set_blocking(self.proc.stdout.fileno(),False)\n            self.ready=self._read(time.monotonic()+float(limits[\'load_timeout_seconds\']))\n            if self.ready.get(\'status\')!=\'ready\':raise RuntimeError(\'Actor failed loading: \'+str(self.ready))\n        except BaseException:\n            self.close();raise\n\n    def _wait(self,fd,event,deadline):\n        remaining=deadline-time.monotonic()\n        if remaining<=0:raise TimeoutError(\'Actor RPC watchdog expired.\')\n        with selectors.DefaultSelector() as s:\n            s.register(fd,event)\n            if not s.select(remaining):raise TimeoutError(\'Actor RPC watchdog expired.\')\n\n    def _write(self,data,deadline):\n        view=memoryview(data)\n        while view:\n            self._wait(self.proc.stdin.fileno(),selectors.EVENT_WRITE,deadline)\n            try:n=os.write(self.proc.stdin.fileno(),view)\n            except BlockingIOError:continue\n            if n<=0:raise RuntimeError(\'Actor input pipe closed.\')\n            view=view[n:]\n\n    def _read(self,deadline):\n        while b\'\\n\' not in self.buffer:\n            self._wait(self.proc.stdout.fileno(),selectors.EVENT_READ,deadline)\n            try:b=os.read(self.proc.stdout.fileno(),65536)\n            except BlockingIOError:continue\n            if not b:raise RuntimeError(\'Actor exited before returning a result; see actor_stderr.log.\')\n            self.buffer+=b\n            if len(self.buffer)>65536:raise RuntimeError(\'Actor response exceeds 64 KiB.\')\n        line,self.buffer=self.buffer.split(b\'\\n\',1)\n        value=json.loads(line)\n        if not isinstance(value,dict):raise ValueError(\'Actor response must be a JSON object.\')\n        if value.get(\'status\')==\'error\':raise RuntimeError(value.get(\'error\',\'Actor error\'))\n        return value\n\n    def request(self,op,**payload):\n        self.seq+=1;start=time.monotonic()\n        deadline=start+float(self.limits[\'load_timeout_seconds\'] if op in (\'reset\',\'finish\') else self.limits[\'decision_timeout_seconds\'])\n        req=json.dumps({\'seq\':self.seq,\'op\':op,**payload},separators=(\',\',\':\'),allow_nan=False).encode()+b\'\\n\'\n        if len(req)>512*1024:raise ValueError(\'Actor request exceeds 512 KiB.\')\n        self._write(req,deadline);result=self._read(deadline)\n        elapsed=time.monotonic()-start\n        if result.get(\'seq\')!=self.seq:raise RuntimeError(\'Out-of-order actor response.\')\n        if op==\'act\':self.times.append(elapsed)\n        return result\n\n    def close(self):\n        proc=getattr(self,\'proc\',None)\n        if proc is not None:\n            if proc.poll() is None:\n                try:os.killpg(proc.pid,signal.SIGTERM)\n                except ProcessLookupError:pass\n                try:proc.wait(timeout=3)\n                except subprocess.TimeoutExpired:\n                    try:os.killpg(proc.pid,signal.SIGKILL)\n                    except ProcessLookupError:pass\n                    proc.wait()\n            for pipe in (proc.stdin,proc.stdout):\n                if pipe is not None:pipe.close()\n            self.proc=None\n        if getattr(self,\'log\',None):self.log.close()\n', 'sf2_boot.py': '"""Cold-boot SF2 setup. No external or hidden emulator save states.\n\nNative CPU source: GAME START (single-player arcade), never a Python bot.\nTargeted PvE is EXPERIMENTAL: an initialization-only stage selector at\n0xFF96AA (>u2) requests the opponent\'s stage. We DO NOT write p1/p2 character\nIDs to make the identity checks pass. Stop with evidence if the native game\nloads a different opponent. The user observed ChunLi vs CPU ChunLi with the prior\nmenu patch; other combinations and this edition\'s new rule-menu route need testing.\n\nMenu timing/difficulty observations and character read mappings are adapted\nfrom LEIA-qro/street_fighter (MIT), commit ea7545305f30a62dceb7c7b77b9d51466029bf7e.\nStage selector: Rune / BigBossman, GameHacking.org game 15969.\nSee SOURCES.md and THIRD_PARTY_LICENSES.txt. No ROM or model is distributed.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom collections import deque\nimport gc\nimport hashlib\nimport importlib\nimport importlib.metadata\nimport json\nimport math\nimport re\nimport time\nfrom typing import Callable\n\nimport numpy as np\n\nGAME = "StreetFighterIISpecialChampionEdition-Genesis"\nROM_SHA1 = "a5aad1d108046d9388e33247610dafb4c6516e0b"\nCHARACTERS = ("Ryu", "EHonda", "Blanka", "Guile", "Ken", "ChunLi",\n              "Zangief", "Dhalsim", "MBison", "Sagat", "Balrog", "Vega")\nCHAR_IDS = {name: i for i, name in enumerate(CHARACTERS)}\n# US character-select layout, as visible in the user\'s actual menu.\nGRID = ((0, 1, 2, 3, 10, 11), (4, 5, 6, 7, 9, 8))\nCELLS = {cid: (row, col) for row, values in enumerate(GRID) for col, cid in enumerate(values)}\nRAM_FIELDS = {\n    "p1_x": (0xFF8006, ">u2"), "p1_y": (0xFF800A, ">u2"),\n    "p2_x": (0xFF8286, ">u2"), "p2_y": (0xFF828A, ">u2"),\n    "p1_hp": (0xFF8042, ">i2"), "p2_hp": (0xFF82C2, ">i2"),\n    "p1_char": (0xFF81DB, "|u1"), "p2_char": (0xFF845B, "|u1"),\n    "p1_wins": (0xFF81DA, "|u1"), "p2_wins": (0xFF845A, "|u1"),\n    "timer_bcd": (0xFF972A, "|u1"),\n    "difficulty_menu": (0xFFFE44, "|u1"),\n    "difficulty_fight": (0xFF97B3, "|u1"),\n    "stage_selector": (0xFF96AA, ">u2"),\n    # Read-only corroboration, not treated as definitive CPU proof.\n    "p1_game_input": (0xFF81E2, "|u1"),\n    "p2_game_input": (0xFF845E, "|u1"),\n}\n\n\ndef normalize_character(value: str) -> str:\n    key = re.sub(r"[^a-z0-9]", "", str(value).lower())\n    for name in CHARACTERS:\n        if re.sub(r"[^a-z0-9]", "", name.lower()) == key:\n            return name\n    raise ValueError(f"Unknown character {value!r}. Choose one of {CHARACTERS}.")\n\n\ndef backend_module():\n    try:\n        return importlib.import_module("stable_retro")\n    except ModuleNotFoundError as exc:\n        if exc.name != "stable_retro":\n            raise\n        return importlib.import_module("retro")\n\n\ndef resolve_installed_game(backend) -> tuple[str, Path]:\n    """Resolve the actual backend ID without altering public rules or ROM bytes."""\n    mismatches = []\n    for game_id in (GAME, GAME + "-v0"):\n        for method in ("get_original_romfile_path", "get_romfile_path"):\n            fn = getattr(backend.data, method, None)\n            if fn is not None:\n                try:\n                    path = Path(fn(game_id))\n                    if path.is_file():\n                        checksum = hashlib.sha1(path.read_bytes()).hexdigest()\n                        if checksum == ROM_SHA1:\n                            return game_id, path\n                        mismatches.append(f"{game_id}: {checksum}")\n                except (OSError, RuntimeError, TypeError, ValueError):\n                    pass\n    if mismatches:\n        raise RuntimeError("Wrong ROM SHA1. Expected " + ROM_SHA1 + "; found "\n                           + "; ".join(mismatches))\n    raise FileNotFoundError("Matching ROM is not installed under " + GAME + " or "\n                            + GAME + "-v0. Run the ROM setup cell, not a .state upload.")\n\n\ndef find_installed_rom(backend) -> Path:\n    # Preserve the existing path-only API for other callers.\n    return resolve_installed_game(backend)[1]\n\n\ndef pack_buttons(buttons: list[str], p1=(), p2=()) -> np.ndarray:\n    result = np.zeros(2 * len(buttons), dtype=np.int8)\n    for port, pressed in enumerate((p1, p2)):\n        if isinstance(pressed, str):\n            pressed = (pressed,)\n        for name in pressed:\n            if name not in buttons:\n                raise ValueError(f"Unknown button {name!r}; actual buttons: {buttons}")\n            result[port * len(buttons) + buttons.index(name)] = 1\n    return result\n\n\ndef route_to_character(current_id: int, target_id: int) -> list[str]:\n    """No wrap-around is assumed. A rectangular menu is verified at runtime."""\n    if current_id not in CELLS or target_id not in CELLS:\n        raise ValueError("Character ID is outside the sourced US roster.")\n    r, c = CELLS[current_id]\n    tr, tc = CELLS[target_id]\n    vertical = ["DOWN" if tr > r else "UP"] * abs(tr-r)\n    horizontal = ["RIGHT" if tc > c else "LEFT"] * abs(tc-c)\n    return vertical + horizontal\n\n\ndef bcd_seconds(value: int) -> int | None:\n    value = int(value)\n    if value < 0 or value > 255 or (value & 15) > 9 or (value >> 4) > 9:\n        return None\n    return (value >> 4) * 10 + (value & 15)\n\n\ndef round_loaded(v: dict) -> bool:\n    """Read-only signature from the native first-round initialization."""\n    return (v["p1_hp"] == 176 and v["p2_hp"] == 176\n            and v["timer_bcd"] == 0x99\n            and v["p1_wins"] == 0 and v["p2_wins"] == 0\n            and v["p1_char"] in CELLS and v["p2_char"] in CELLS\n            and 0 < v["p1_x"] < 4096 and 0 < v["p2_x"] < 4096)\n\n\ndef validate_character_ids(v: dict, p1: str, p2: str | None) -> None:\n    if v["p1_char"] != CHAR_IDS[p1]:\n        raise RuntimeError(f"P1 identity mismatch: requested {p1}, raw ID={v[\'p1_char\']}. Inspect screenshots.")\n    if p2 is not None and v["p2_char"] != CHAR_IDS[p2]:\n        raise RuntimeError(\n            f"CPU/P2 identity mismatch: requested {p2}, raw ID={v[\'p2_char\']}. "\n            "The initialization method did NOT produce the requested opponent; no fake fallback is used.")\n\n\n@dataclass\nclass MatchConfig:\n    ruleset: str = "hyper"\n    speed_stars: int = 3\n    mode: str = "pve"\n    p1_character: str = "ChunLi"\n    p2_character: str = "ChunLi"\n    difficulty_stars: int = 4\n    target_cpu: bool = True\n    allow_initialization_stage_write: bool = True\n    video_seconds: float = 30\n    seed: int = 42\n    p1_model: str = ""\n    p2_model: str = ""\n    decision_frames: int = 6\n    position_scale: float = 1024.0\n    boot_idle_frames: int = 1740\n    menu_budget: int = 8\n    fight_load_budget: int = 1800\n    live_wait_budget: int = 900\n    confirm_p1: str = "A"\n    confirm_p2: str = "START"\n    match_cap_seconds: float = 180\n\n    def validate(self):\n        if self.ruleset not in ("champion", "hyper"):\n            raise ValueError("ruleset must be champion or hyper")\n        if type(self.speed_stars) is not int or not 0 <= self.speed_stars <= 4:\n            raise ValueError("speed_stars must be 0..4; it is not CPU difficulty")\n        if self.mode not in ("pve", "pvp"):\n            raise ValueError("mode must be pve or pvp")\n        self.p1_character = normalize_character(self.p1_character)\n        self.p2_character = normalize_character(self.p2_character)\n        if not 1 <= self.difficulty_stars <= 8:\n            raise ValueError("difficulty_stars must be 1..8 (stored menu value 0..7)")\n        if not 0 < self.video_seconds <= 300 or self.decision_frames != 6:\n            raise ValueError("video_seconds must be 0..300 and decision_frames must remain 6")\n        if not math.isfinite(self.position_scale) or self.position_scale <= 0:\n            raise ValueError("position_scale must be positive and finite")\n        if self.mode == "pve" and self.p2_model:\n            raise ValueError("PvE uses the NATIVE CPU. Leave p2_model empty; use PvP for a model opponent.")\n        if self.mode == "pve" and self.target_cpu and not self.allow_initialization_stage_write:\n            raise ValueError("Targeted CPU selection requires the experimental init-stage write. "\n                             "Set target_cpu=False only to test the natural arcade opponent.")\n        if self.boot_idle_frames < 600 or self.menu_budget < 1:\n            raise ValueError("Invalid startup timing")\n        for key in (self.confirm_p1, self.confirm_p2):\n            if key not in ("A", "B", "C", "START", "X", "Y", "Z"):\n                raise ValueError("Unsupported character confirmation button")\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")\n\n\nclass ColdBootMatch:\n    """A single real emulator; reset means close + recreate + navigate.\n\n    This intentionally does not use get_state(), set_state(), initial_state,\n    downloadable snapshots, or opaque .state bytes hidden in the notebook.\n    The cost is that every new match repeats the startup sequence.\n    """\n    def __init__(self, config: MatchConfig, output_dir: str | Path):\n        config.validate()\n        self.config = config\n        self.root = Path(output_dir).resolve()\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.backend = backend_module()\n        self.engine = None\n        self.rgb = None\n        self.values = {}\n        self.buttons = []\n        self.frame = 0\n        self.phase = "not_started"\n        self.stage_write_active = False\n        self.battle_started = False\n        self.trace = deque(maxlen=6000)\n        self.checkpoints = []\n        self.frame_callback: Callable | None = None\n        self.report = {\n            "status": "STARTED", "config": asdict(config),\n            "external_state_files_used": False, "in_memory_snapshots_used": False,\n            "training_performed": False, "pve_python_opponent_created": False,\n            "initialization_method": "native arcade menu + optional pre-fight stage selector",\n            "targeted_pve_method_experimental": True,\n            "sprite_identity_automatically_recognized": False,\n            "native_cpu_control_automatically_proven": False,\n            "ram_writes": [], "checkpoints": self.checkpoints,\n        }\n\n    def _create(self):\n        game_id, path = resolve_installed_game(self.backend)\n        checksum = hashlib.sha1(path.read_bytes()).hexdigest()\n        if checksum != ROM_SHA1:\n            raise RuntimeError(f"Wrong ROM SHA1 {checksum}; expected {ROM_SHA1}. Address map is version-specific.")\n        self.report["rom_sha1"] = checksum\n        self.report["backend_game_id"] = game_id\n        scenario = self.root / "bootstrap_scenario.json"\n        scenario.write_text("{}", encoding="utf-8")\n        self.engine = self.backend.make(game=game_id, state=self.backend.State.NONE,\n                                        players=2, use_restricted_actions=self.backend.Actions.ALL,\n                                        scenario=str(scenario), render_mode="rgb_array")\n        self.buttons = list(self.engine.buttons)\n        self.rgb, _ = self.engine.reset()\n        before = np.asarray(self.engine.get_ram()).copy()\n        for name, (address, dtype) in RAM_FIELDS.items():\n            self.engine.data.set_variable("boot_"+name, {"address": address, "type": dtype})\n        if not np.array_equal(before, np.asarray(self.engine.get_ram())):\n            raise RuntimeError("Registering read definitions unexpectedly changed RAM.")\n        self.values = self.read()\n        self.fps = float(self.engine.em.get_screen_rate())\n        if not math.isfinite(self.fps) or self.fps <= 0:\n            raise RuntimeError("Invalid native frame rate")\n        self.report.update(buttons=self.buttons, video_fps=self.fps,\n                           native_image_shape=list(self.rgb.shape),\n                           versions={})\n        for package in ("stable-retro", "numpy"):\n            try:\n                self.report["versions"][package] = importlib.metadata.version(package)\n            except importlib.metadata.PackageNotFoundError:\n                pass\n\n    def read(self):\n        return {name: int(self.engine.data.lookup_value("boot_"+name)) for name in RAM_FIELDS}\n\n    def _write_stage_before_load(self):\n        if self.battle_started:\n            raise RuntimeError("Safety check: setup writes are forbidden after battle initialization.")\n        value = CHAR_IDS[self.config.p2_character]\n        # The only RAM value this implementation ever writes. Character IDs,\n        # health, position, timer and difficulty-derived combat parameters are NOT written.\n        before = self.values["stage_selector"]\n        self.engine.data.set_value("boot_stage_selector", value)\n        if int(self.engine.data.lookup_value("boot_stage_selector")) != value:\n            raise RuntimeError("Stage-selector write did not read back correctly.")\n        self.report["ram_writes"].append({"frame": self.frame, "phase": self.phase,\n            "address": "0xFF96AA", "type": ">u2", "before": before, "value": value})\n\n    def frame_step(self, p1=(), p2=()):\n        if self.config.mode == "pve" and p2:\n            raise RuntimeError("PvE safety: never send buttons to P2; CPU is native to GAME START.")\n        if self.stage_write_active:\n            self._write_stage_before_load()\n        a = pack_buttons(self.buttons, p1, p2)\n        self.rgb, _, terminated, truncated, _ = self.engine.step(a)\n        self.frame += 1\n        self.values = self.read()\n        row = {"frame": self.frame, "phase": self.phase, "p1_buttons": list(p1),\n               "p2_buttons": list(p2), **self.values}\n        self.trace.append(row)\n        if self.frame_callback is not None:\n            self.frame_callback(self.rgb, row)\n        if terminated or truncated:\n            raise RuntimeError("Unexpected native terminal flag in bootstrap scenario.")\n        return self.values\n\n    def advance(self, count: int, p1=(), p2=()):\n        if not 0 <= int(count) <= 20000:\n            raise ValueError("Invalid native-frame advance length")\n        for _ in range(int(count)):\n            self.frame_step(p1, p2)\n\n    def tap(self, player: int, key: str, hold=8, release=8):\n        if player not in (1, 2):\n            raise ValueError("player must be 1 or 2")\n        self.advance(hold, (key,) if player == 1 else (), (key,) if player == 2 else ())\n        self.advance(release)\n\n    def shot(self, label: str):\n        from PIL import Image\n        name = f"{len(self.checkpoints):02d}_{label}.png"\n        Image.fromarray(self.rgb).save(self.root/name)\n        self.checkpoints.append({"file": name, "frame": self.frame, "phase": self.phase,\n                                 "readings": dict(self.values)})\n        print(f"[{self.phase}] {label}: char IDs {self.values[\'p1_char\']}/{self.values[\'p2_char\']}; "\n              f"HP {self.values[\'p1_hp\']}/{self.values[\'p2_hp\']}; "\n              f"timer raw {self.values[\'timer_bcd\']}", flush=True)\n\n    def _set_difficulty_via_menu(self):\n        from sf2_menu import set_difficulty_via_menu_fixed\n        return set_difficulty_via_menu_fixed(self)\n\n    def _cursor_responds(self, player: int) -> bool:\n        if round_loaded(self.values):\n            self.shot("fight_reached_before_selection")\n            raise RuntimeError("Reached a fight before choosing characters. Stop rather than patch a live fighter.")\n        key = f"p{player}_char"\n        before = self.values[key]\n        if before not in CELLS:\n            return False\n        row, col = CELLS[before]\n        direction = "RIGHT" if col < 5 else "LEFT"\n        reverse = "LEFT" if direction == "RIGHT" else "RIGHT"\n        expected = GRID[row][col+(1 if direction == "RIGHT" else -1)]\n        self.tap(player, direction)\n        after = self.values[key]\n        self.tap(player, reverse)\n        # A menu name alone is not detection. We check a reversible cursor movement.\n        return after == expected and self.values[key] == before\n\n    def _seek_player_select(self):\n        from sf2_menu import select_rules_and_game_start\n        select_rules_and_game_start(self)\n        self.phase = \'seek_native_player_select\'\n        for attempt in range(self.config.menu_budget):\n            if self._cursor_responds(1):\n                self.shot(\'p1_character_cursor_responds\')\n                return\n            self.advance(60)\n        self.shot(\'character_cursor_not_verified\')\n        raise RuntimeError(\'Character cursor not verified after GAME START; inspect screenshot, no blind START retry.\')\n\n    def _choose(self, player: int, name: str):\n        self.phase = f"choose_p{player}_{name}"\n        target = CHAR_IDS[name]\n        for key in route_to_character(self.values[f"p{player}_char"], target):\n            before = self.values[f"p{player}_char"]\n            r, c = CELLS[before]\n            nr, nc = r + (key == "DOWN") - (key == "UP"), c + (key == "RIGHT") - (key == "LEFT")\n            expected = GRID[nr][nc]\n            self.tap(player, key)\n            if self.values[f"p{player}_char"] != expected:\n                self.shot("cursor_mismatch")\n                raise RuntimeError("Actual cursor movement differs from expected grid. Recalibration required.")\n        if self.values[f"p{player}_char"] != target:\n            raise RuntimeError("Target character was not selected")\n        self.shot(f"p{player}_{name}_highlighted")\n\n    def _join_second_player(self):\n        # Native arcade challenge is deliberately used, not the V.S. BATTLE\n        # handicap/settings menu from the earlier calibration notebook.\n        self.phase = "pvp_native_second_player_join"\n        for _ in range(3):\n            if self._cursor_responds(2) and self._cursor_responds(1):\n                self.shot("both_player_cursors_respond")\n                self.report["pvp_route"] = "native arcade two-controller challenge"\n                return\n            self.tap(2, "START", release=128)\n        raise RuntimeError("P2 did not join as a selectable human player. No PvP success is claimed.")\n\n    def _confirm_and_load(self):\n        self.phase = "native_match_initialization"\n        self.stage_write_active = self.config.mode == "pve" and self.config.target_cpu\n        first = (self.config.confirm_p1,)\n        second = (self.config.confirm_p2,) if self.config.mode == "pvp" else ()\n        try:\n            for index in range(self.config.fight_load_budget):\n                # Menu confirmation held for 8 native frames; never pressed again in battle.\n                self.frame_step(first if index < 8 else (), second if index < 8 else ())\n                if round_loaded(self.values):\n                    self.stage_write_active = False\n                    self.battle_started = True\n                    self.report["first_round_initialized_frame"] = self.frame\n                    # Let native initialization settle; the source captures after eight\n                    # frames inside the walk-in window. No setup writes occur here.\n                    self.advance(8)\n                    self.shot("native_round_initialized")\n                    validate_character_ids(self.values, self.config.p1_character,\n                        self.config.p2_character if self.config.mode == "pvp" or self.config.target_cpu else None)\n                    if self.values["difficulty_menu"] != self.config.difficulty_stars-1:\n                        raise RuntimeError("Difficulty changed during initialization")\n                    self.report["difficulty_fight_readback"] = self.values["difficulty_fight"]\n                    if self.config.mode == "pve" and self.values["difficulty_fight"] != self.config.difficulty_stars-1:\n                        raise RuntimeError("Native fight difficulty copy does not match menu setting. Do not train on this setup.")\n                    return\n            self.shot("fight_initialization_timeout")\n            raise RuntimeError("No native first-round signature. The game may still be in a menu; see images and trace.")\n        finally:\n            self.stage_write_active = False\n\n    def _wait_until_live(self):\n        self.phase = "wait_for_native_countdown"\n        consecutive = 0\n        for _ in range(self.config.live_wait_budget):\n            self.frame_step()\n            validate_character_ids(self.values, self.config.p1_character,\n                self.config.p2_character if self.config.mode == "pvp" or self.config.target_cpu else None)\n            seconds = bcd_seconds(self.values["timer_bcd"])\n            if seconds is not None and 0 < seconds < 99:\n                consecutive += 1\n            else:\n                consecutive = 0\n            if consecutive >= 3:\n                if self.values["p1_hp"] < 0 or self.values["p2_hp"] < 0:\n                    raise RuntimeError("A character was KO\'d before the demo could start")\n                self.phase = "battle"\n                self.shot("battle_start")\n                self.report.update(status="BOOTSTRAP_CHECKS_PASSED_VISUAL_CHECK_REQUIRED",\n                    actual_character_ids={"p1":self.values["p1_char"],"p2":self.values["p2_char"]},\n                    initial_battle_readings=dict(self.values),\n                    all_setup_writes_stopped=True,\n                    loaded_identifiers_checked_read_only=True)\n                return\n        self.shot("countdown_not_running")\n        raise RuntimeError("Timer did not start. Do not mistake a menu/paused frame for a live fight.")\n\n    def start(self):\n        try:\n            self._create()\n            self._set_difficulty_via_menu()\n            self._seek_player_select()\n            if self.config.mode == "pvp":\n                self._join_second_player()\n            self._choose(1, self.config.p1_character)\n            if self.config.mode == "pvp":\n                self._choose(2, self.config.p2_character)\n            self._confirm_and_load()\n            self._wait_until_live()\n            return self.rgb.copy(), dict(self.values)\n        except Exception as exc:\n            self.stage_write_active = False\n            self.report.update(status="ERROR", error=repr(exc), failed_phase=self.phase)\n            if self.rgb is not None:\n                self.shot("error")\n            raise\n        finally:\n            self.flush()\n\n    def flush(self):\n        write_json(self.root/"bootstrap_report.json", self.report)\n        write_json(self.root/"bootstrap_trace.json", list(self.trace))\n\n    def close(self):\n        self.stage_write_active = False\n        if self.engine is not None:\n            self.engine.close()\n            self.engine = None\n        gc.collect()\n        self.flush()\n', 'sf2_bundle.py': '"""Portable agent = weights + exact actions + custom model source + protocol metadata.\nOnly load trusted packages: Python custom classes / SB3 checkpoints can execute code.\nChecksums detect changes; they do NOT make untrusted Python safe.\n"""\nfrom __future__ import annotations\nfrom pathlib import Path,PurePosixPath\nimport hashlib,json,shutil,tempfile,zipfile\nimport numpy as np\nfrom sf2_common import BUNDLE_SCHEMA,ENV_PROTOCOL,OBS_PROTOCOL,json_write,file_hash,source_module,rules_signature,runtime_versions\nfrom sf2_actions import signature,validate_library\n\n\ndef make_spec(agent,actions,observation_source):\n    from sf2_observation import build_descriptor\n    from sf2_boot import normalize_character\n    return {\'team_name\':str(agent[\'team_name\']),\'character\':normalize_character(agent[\'character\']),\n            \'actions\':validate_library(actions),\'action_hash\':signature(actions),\n            \'use_position\':bool(agent.get(\'use_position\',True)),\n            \'observation\':build_descriptor(observation_source)}\n\n\ndef export_agent(path,model,spec,model_source,reward_source,common,training=None):\n    from sf2_observation import observation_space,description\n    meta=description(spec)\n    if model.observation_space!=observation_space(spec.get(\'use_position\',True),common[\'position_scale\'],spec=spec):\n        raise ValueError(\'Cannot export mismatched model/observation specification.\')\n    path=Path(path).resolve();path.parent.mkdir(parents=True,exist_ok=True)\n    with tempfile.TemporaryDirectory() as td:\n        d=Path(td);model.save(d/\'policy.zip\')\n        shutil.copy2(model_source,d/\'model.py\');shutil.copy2(reward_source,d/\'reward.py\')\n        (d/\'observation.py\').write_text(spec[\'observation\'][\'source\'],encoding=\'utf-8\')\n        json_write(d/\'spec.json\',spec);json_write(d/\'common.json\',common);json_write(d/\'training.json\',training or {})\n        manifest={\'schema\':BUNDLE_SCHEMA,\'env_protocol\':ENV_PROTOCOL,\'observation_protocol\':OBS_PROTOCOL,\n                  \'observation\':meta,\n                  \'algorithm\':\'PPO\',\'team_name\':spec[\'team_name\'],\'character\':spec[\'character\'],\n                  \'action_count\':len(spec[\'actions\']),\'action_hash\':signature(spec[\'actions\']),\n                  \'model_source_sha256\':file_hash(d/\'model.py\'),\'rules\':rules_signature(common),\n                  \'input_keys\':[\'image\',\'position\'] if spec.get(\'use_position\',True) else [\'image\'],\'image_mirrored\':False,\n                  \'runtime_versions\':runtime_versions(),\n                  \'security\':\'Trusted Python only. Checksums are not a sandbox.\'}\n        json_write(d/\'manifest.json\',manifest)\n        json_write(d/\'checksums.json\',{p.name:file_hash(p) for p in sorted(d.iterdir()) if p.is_file()})\n        temp=path.with_suffix(\'.building.zip\')\n        with zipfile.ZipFile(temp,\'w\',zipfile.ZIP_DEFLATED) as z:\n            for p in sorted(d.iterdir()):z.write(p,p.name)\n        temp.replace(path)\n    return str(path)\n\n\nclass LoadedAgent:\n    def __init__(self,path,*,trusted=False,device=\'cpu\'):\n        if not trusted:raise ValueError(\'Read/review this package first, then set TRUST_AGENT_PACKAGES=True. Loading Python is not sandboxed.\')\n        self.path=Path(path).resolve();self.directory=Path(tempfile.mkdtemp(prefix=\'sf2_agent_\'));self.model=None\n        try:\n            with zipfile.ZipFile(self.path) as z:\n                infos=z.infolist()\n                allowed={\'policy.zip\',\'model.py\',\'reward.py\',\'observation.py\',\'spec.json\',\'common.json\',\'training.json\',\'manifest.json\',\'checksums.json\'}\n                if set(z.namelist())!=allowed or len(infos)!=len(allowed):raise ValueError(\'This input upgrade requires a classroom v3 package. Old v2/image-size checkpoints must use the old notebook; do not silently convert weights.\')\n                if sum(i.file_size for i in infos)>512*1024**2:raise ValueError(\'Agent package too large\')\n                for i in infos:\n                    if PurePosixPath(i.filename).name!=i.filename or (i.external_attr>>16)&0o170000==0o120000:raise ValueError(\'Unsafe archive member\')\n                    (self.directory/i.filename).write_bytes(z.read(i))\n            checks=json.loads((self.directory/\'checksums.json\').read_text())\n            if set(checks)!=allowed-{\'checksums.json\'}:raise ValueError(\'Missing package checksum\')\n            for n,h in checks.items():\n                if file_hash(self.directory/n)!=h:raise ValueError(\'Package checksum mismatch: \'+n)\n            self.manifest=json.loads((self.directory/\'manifest.json\').read_text())\n            self.spec=json.loads((self.directory/\'spec.json\').read_text());self.common=json.loads((self.directory/\'common.json\').read_text())\n            m=self.manifest\n            if m[\'schema\']!=BUNDLE_SCHEMA or m[\'env_protocol\']!=ENV_PROTOCOL or m[\'observation_protocol\']!=OBS_PROTOCOL or m[\'algorithm\']!=\'PPO\':raise ValueError(\'Unsupported agent protocol\')\n            if signature(self.spec[\'actions\'])!=m[\'action_hash\'] or self.spec.get(\'action_hash\')!=m[\'action_hash\'] or len(self.spec[\'actions\'])!=m[\'action_count\'] or self.spec[\'character\']!=m[\'character\']:raise ValueError(\'Agent/action identity mismatch\')\n            if m[\'rules\']!=rules_signature(self.common):raise ValueError(\'Rules metadata mismatch\')\n            if file_hash(self.directory/\'model.py\')!=m[\'model_source_sha256\']:raise ValueError(\'Model source mismatch\')\n            from sf2_observation import description\n            if m.get(\'observation\')!=description(self.spec):raise ValueError(\'Observation metadata mismatch\')\n            if (self.directory/\'observation.py\').read_text(encoding=\'utf-8\')!=self.spec[\'observation\'][\'source\']:\n                raise ValueError(\'Preprocessing source differs from the spec.\')\n            source_module(self.directory/\'model.py\',\'model\')\n            from stable_baselines3 import PPO\n            self.model=PPO.load(self.directory/\'policy.zip\',device=device)\n            from sf2_observation import observation_space\n            if self.manifest.get(\'input_keys\') != ([\'image\',\'position\'] if self.spec.get(\'use_position\',True) else [\'image\']):raise ValueError(\'Input metadata mismatch\')\n            if self.model.action_space.n!=len(self.spec[\'actions\']):raise ValueError(\'Policy output does not match exported action table\')\n            if self.model.observation_space!=observation_space(self.spec.get(\'use_position\',True),self.common[\'position_scale\'],spec=self.spec):raise ValueError(\'Observation specification mismatch\')\n            self.model.policy.set_training_mode(False)\n        except BaseException:\n            self.close();raise\n    def act(self,side,raw,deterministic=True):\n        if side.spec[\'action_hash\']!=self.spec[\'action_hash\']:raise ValueError(\'Wrong action table for this model\')\n        if side.spec[\'observation\'][\'source_sha256\']!=self.spec[\'observation\'][\'source_sha256\'] or side.spec.get(\'use_position\',True)!=self.spec.get(\'use_position\',True):\n            raise ValueError(\'Wrong observation definition for this policy.\')\n        if side.scale!=self.common[\'position_scale\']:raise ValueError(\'Wrong position scale for this policy.\')\n        a,_=self.model.predict(side.observe(raw),deterministic=deterministic)\n        return int(np.asarray(a).item())\n    def close(self):\n        self.model=None\n        if getattr(self,\'directory\',None):shutil.rmtree(self.directory,ignore_errors=True)\n', 'sf2_common.py': '"""Versioned contracts and isolated loading of student Python source."""\nfrom __future__ import annotations\nimport hashlib,importlib.util,json,sys\nfrom pathlib import Path\nimport numpy as np\n\nENV_PROTOCOL=\'sf2-classroom-sixframe-v2\'\nOBS_PROTOCOL=\'sf2-configurable-image4-position6-v3\'\nBUNDLE_SCHEMA=\'sf2-plain-ppo-agent-v3\'\n\ndef json_write(path,value):\n    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)\n    p.write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n\ndef file_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef source_module(path,kind):\n    """Content-addressed names prevent two teams\' custom model classes colliding."""\n    p=Path(path).resolve();name=\'sf2_custom_\'+kind+\'_\'+file_hash(p)[:24]\n    if name not in sys.modules:\n        spec=importlib.util.spec_from_file_location(name,p)\n        if spec is None or spec.loader is None:raise ImportError(p)\n        m=importlib.util.module_from_spec(spec);sys.modules[name]=m\n        try:spec.loader.exec_module(m)\n        except BaseException:\n            sys.modules.pop(name,None);raise\n    return sys.modules[name]\n\ndef parameter_hash(model):\n    h=hashlib.sha256()\n    for name,v in model.policy.state_dict().items():\n        h.update(name.encode());h.update(v.detach().cpu().contiguous().numpy().tobytes())\n    return h.hexdigest()\n\ndef compute_transition(before,after,player,winner,reason,frames):\n    if player not in (1,2):raise ValueError(\'Invalid learner player\')\n    opp=3-player\n    dealt=max(0,max(before[f\'p{opp}_hp\'],0)-max(after[f\'p{opp}_hp\'],0))\n    taken=max(0,max(before[f\'p{player}_hp\'],0)-max(after[f\'p{player}_hp\'],0))\n    result=(\'cutoff\' if reason==\'frame_cap\' else \'draw\' if winner==\'draw\' else \'win\' if winner==f\'p{player}\' else \'loss\' if winner else \'ongoing\')\n    return {\'damage_dealt\':dealt,\'damage_taken\':taken,\'result\':result,\'end_reason\':reason,\n            \'elapsed_native_frames\':frames,\'self_hp\':after[f\'p{player}_hp\'],\'opponent_hp\':after[f\'p{opp}_hp\'],\n            \'self_player\':player,\n            \'self_position\':[after[f\'p{player}_x\'],after[f\'p{player}_y\']],\n            \'opponent_position\':[after[f\'p{opp}_x\'],after[f\'p{opp}_y\']]}\n\ndef round_result(v,wins):\n    h1,h2=v[\'p1_hp\'],v[\'p2_hp\']\n    if min(h1,h2)<0:\n        return (\'draw\' if h1<0 and h2<0 else \'p2\' if h1<0 else \'p1\'),\'signed_hp_ko\'\n    a,b=v[\'p1_wins\']>wins[0],v[\'p2_wins\']>wins[1]\n    if a or b:return (\'draw\' if a and b else \'p1\' if a else \'p2\'),\'round_counter\'\n    from sf2_boot import bcd_seconds\n    if bcd_seconds(v[\'timer_bcd\'])==0:\n        return (\'p1\' if h1>h2 else \'p2\' if h2>h1 else \'draw\'),\'native_time_up\'\n    return None,None\n\ndef rules_signature(common):\n    if common[\'ruleset\'] not in (\'champion\',\'hyper\') or type(common[\'speed_stars\']) is not int or not 0<=common[\'speed_stars\']<=4:\n        raise ValueError(\'Ruleset must be champion/hyper and supported speed 0..4\')\n    if common[\'decision_frames\']!=6 or not np.isfinite(common[\'position_scale\']) or common[\'position_scale\']<=0:\n        raise ValueError(\'This protocol uses 6-frame decisions and a positive coordinate scale\')\n    if type(common[\'max_round_frames\']) is not int or not 6<=common[\'max_round_frames\']<=36000:\n        raise ValueError(\'max_round_frames must be 6..36000\')\n    return {k:common[k] for k in (\'ruleset\',\'speed_stars\',\'decision_frames\',\'position_scale\',\'max_round_frames\')}\n\n\ndef runtime_versions():\n    from importlib.metadata import version,PackageNotFoundError\n    result={}\n    for name in (\'torch\',\'numpy\',\'gymnasium\',\'stable-baselines3\',\'stable-retro\'):\n        try:result[name]=version(name)\n        except PackageNotFoundError:result[name]=None\n    return result\n', 'sf2_env.py': '"""A standard Gymnasium environment for ordinary PPO.\nSix native frames per decision; only the final transition may be shorter.\nNo action masks, ignored policy actions or long sequences. P1 native-CPU PvE.\nOptional frozen-agent training uses a true two-controller PvP game.\n"""\nfrom pathlib import Path\nimport time\nimport gymnasium as gym\nimport numpy as np\nfrom sf2_actions import compiled,action_index,QUANTUM\nfrom sf2_observation import Side,observation_space\nfrom sf2_game import Game\nfrom sf2_common import compute_transition,rules_signature,json_write,parameter_hash\n\nclass BattleEnv(gym.Env):\n    metadata={\'render_modes\':[\'rgb_array\']}\n    def __init__(self,spec,common,opponents,out,reward_fn,*,opponent_agent=None,\n                 learner_sides=(1,),opponent_deterministic=True,_game_factory=Game):\n        super().__init__();rules_signature(common)\n        self.spec=spec;self.common=dict(common);self.opponents=list(opponents);self.out=Path(out)\n        self.reward_fn=reward_fn;self.opponent_agent=opponent_agent\n        self.learner_sides=tuple(learner_sides);self.opponent_deterministic=opponent_deterministic\n        if not self.opponents:raise ValueError(\'Empty opponent schedule.\')\n        if not self.learner_sides or any(x not in (1,2) for x in self.learner_sides):\n            raise ValueError(\'learner_sides must contain 1 and/or 2.\')\n        if opponent_agent is None and self.learner_sides!=(1,):\n            raise ValueError(\'Native-CPU PvE supports learner P1 only, not a guessed native P1 CPU.\')\n        if opponent_agent is not None and rules_signature(opponent_agent.common)!=rules_signature(common):\n            raise ValueError(\'Frozen opponent arena rules differ.\')\n        self.commands=compiled(spec[\'actions\'])\n        self.other_commands=compiled(opponent_agent.spec[\'actions\']) if opponent_agent else None\n        self.action_space=gym.spaces.Discrete(len(self.commands))\n        self.observation_space=observation_space(spec.get(\'use_position\',True),common[\'position_scale\'],spec=spec)\n        self.game=_game_factory(common,out);self.render_mode=\'rgb_array\';self.sides={};self.ended=True\n        self.episode_index=0;self.video_callback=None;self.last_info={}\n        self._frozen_hash = None\n        if self.opponent_agent is not None:\n            self.opponent_agent.model.policy.set_training_mode(False)\n            for parameter in self.opponent_agent.model.policy.parameters():\n                parameter.requires_grad_(False)\n            self._frozen_hash = parameter_hash(self.opponent_agent.model)\n        self.perf={\'resets\':0,\'reset_seconds\':0.,\'step_seconds\':0.,\'decision_steps\':0,\'warm_resets\':0}\n    def reset(self,*,seed=None,options=None):\n        super().reset(seed=seed);started=time.perf_counter()\n        options=options or {}\n        opp=dict(self.opponents[int(self.np_random.integers(len(self.opponents)))])\n        self.current_opponent=opp\n        self.player=int(self.np_random.choice(self.learner_sides))\n        other_spec=self.opponent_agent.spec if self.opponent_agent else None\n        chars={self.player:self.spec[\'character\'],3-self.player:(other_spec[\'character\'] if other_spec else opp[\'character\'])}\n        mode=\'pvp\' if self.opponent_agent else \'pve\'\n        rgb,raw=self.game.open(chars[1],chars[2],mode,int(opp[\'difficulty_stars\']))\n        jitter=int(opp.get(\'neutral_jitter_max\',0))\n        offset=options.get(\'offset\',opp.get(\'neutral_offset_frames\'))\n        if not 0<=jitter<=60 or (offset is not None and (type(offset) is not int or not 0<=offset<=60)):\n            raise ValueError(\'Neutral offsets must be integers in 0..60.\')\n        self.offset_frames=int(self.np_random.integers(jitter+1)) if offset is None else offset\n        for _ in range(self.offset_frames):\n            _,_,reason=self.game.step_keys((),())\n            if reason:raise RuntimeError(\'Round ended during neutral reset offset.\')\n        self.sides={self.player:Side(self.spec,self.player,self.common[\'position_scale\'])}\n        if other_spec:self.sides[3-self.player]=Side(other_spec,3-self.player,self.common[\'position_scale\'])\n        for side in self.sides.values():side.reset(self.game.rgb,frame_id=self.game.elapsed)\n        self.game.on_frame=self.video_callback\n        self.episode_return=0.;self.episode_steps=0;self.ended=False;self.episode_index+=1\n        seconds=time.perf_counter()-started\n        self.perf[\'resets\']+=1;self.perf[\'reset_seconds\']+=seconds\n        self.perf[\'warm_resets\']=self.game.warm_resets\n        self.last_info={\'self_player\':self.player,\'opponent\':opp,\'raw\':dict(self.game.raw),\n                        \'neutral_offset_frames\':self.offset_frames,\'reset_seconds\':seconds,\n                        \'cache_status\':self.game.last_cache_status}\n        return self.sides[self.player].observe(self.game.raw),dict(self.last_info)\n    def step(self,action):\n        if self.ended:raise RuntimeError(\'Reset before stepping or after the end of a round.\')\n        start=time.perf_counter();idx=action_index(action,self.action_space.n)\n        before=dict(self.game.raw);command=self.commands[idx]\n        other=((),)*QUANTUM\n        if self.opponent_agent is not None:\n            side=self.sides[3-self.player]\n            other_idx=self.opponent_agent.act(side,before,deterministic=self.opponent_deterministic)\n            other=self.other_commands[other_idx]\n        c1,c2=(command,other) if self.player==1 else (other,command)\n        winner=reason=None;executed=0\n        for k1,k2 in zip(c1,c2):\n            raw,winner,reason=self.game.step_keys(k1,k2);executed+=1\n            if reason:break\n        for side in self.sides.values():side.update(self.game.rgb,frame_id=self.game.elapsed)\n        transition=compute_transition(before,raw,self.player,winner,reason,executed)\n        transition.update(previous_self_hp=before[f\'p{self.player}_hp\'],\n                          previous_opponent_hp=before[f\'p{3-self.player}_hp\'])\n        reward=float(self.reward_fn(dict(transition)))\n        if not np.isfinite(reward):raise ValueError(\'Reward must be a finite number.\')\n        self.episode_return+=reward;self.episode_steps+=1\n        terminated=reason is not None and reason!=\'frame_cap\';truncated=reason==\'frame_cap\'\n        self.ended=terminated or truncated\n        info={**transition,\'opponent\':dict(self.current_opponent),\'native_frames\':self.game.elapsed,\n              \'action_name\':self.spec[\'actions\'][idx][\'name\'],\'neutral_offset_frames\':self.offset_frames}\n        if self.ended:\n            info[\'episode\']={\'r\':self.episode_return,\'l\':self.episode_steps,\'t\':self.game.elapsed/self.game.fps}\n        self.last_info=info;self.perf[\'decision_steps\']+=1\n        self.perf[\'step_seconds\']+=time.perf_counter()-start\n        return self.sides[self.player].observe(raw),reward,bool(terminated),bool(truncated),info\n    def render(self):return self.game.rgb.copy()\n    def performance_stats(self):\n        result = dict(self.perf)\n        if self.opponent_agent is not None:\n            after = parameter_hash(self.opponent_agent.model)\n            result.update(frozen_parameter_sha256_before=self._frozen_hash,\n                          frozen_parameter_sha256_after=after,\n                          frozen_opponent_unchanged=(after == self._frozen_hash))\n        return result\n    def close(self):\n        self.game.close();self.ended=True\n        if self.opponent_agent is not None:\n            self.opponent_agent.close();self.opponent_agent=None\n        json_write(self.out/\'worker_performance.json\',self.perf)\n', 'sf2_game.py': '"""One reusable emulator with verified automatic start caches.\nSame native menu/character/difficulty guards as the verified one-star environment.\nOnly resetting a round restores state; no live HP/position/difficulty changes.\n"""\nfrom __future__ import annotations\nfrom collections import deque\nfrom contextlib import redirect_stdout\nfrom pathlib import Path\nimport time\nfrom sf2_boot import MatchConfig,validate_character_ids\nfrom sf2_start_cache import CachedColdBootMatch,_MEMO,_restore_exact\nfrom sf2_common import round_result,rules_signature\n\nclass Game:\n    def __init__(self,common,out):\n        rules_signature(common)\n        self.common=dict(common);self.out=Path(out);self.out.mkdir(parents=True,exist_ok=True)\n        self.boot=None;self.key=None;self.payload=None;self.on_frame=None\n        self.elapsed=0;self.last_boot_seconds=0.;self.resets=0;self.warm_resets=0\n    @property\n    def raw(self):return self.boot.values\n    @property\n    def rgb(self):return self.boot.rgb\n    @property\n    def fps(self):return self.boot.fps\n    def open(self,p1,p2,mode,stars):\n        start=time.perf_counter()\n        key=(p1,p2,mode,int(stars),self.common[\'ruleset\'],self.common[\'speed_stars\'])\n        self.mode=mode;self.on_frame=None\n        if self.boot is not None and self.key==key and self.payload is not None:\n            self.boot.frame_callback=None\n            _restore_exact(self.boot,self.payload)\n            self.warm_resets+=1\n            self.last_cache_status=\'WARM_SAME_EMULATOR_RESTORE\'\n        else:\n            self.close()\n            cfg=MatchConfig(mode=mode,p1_character=p1,p2_character=p2,\n                            difficulty_stars=int(stars),ruleset=self.common[\'ruleset\'],\n                            speed_stars=self.common[\'speed_stars\'])\n            folder=self.out/f\'start_{self.resets:04d}\'\n            folder.mkdir(parents=True,exist_ok=True)\n            self.boot=CachedColdBootMatch(cfg,folder)\n            try:\n                with (folder/\'boot.log\').open(\'w\',encoding=\'utf-8\') as log,redirect_stdout(log):\n                    self.boot.start()\n                self.payload=_MEMO.get(self.boot.cache_info[\'path\'])\n                if self.payload is None or not self.payload[\'meta\'].get(\'roundtrip_verified\'):\n                    raise RuntimeError(\'No verified start payload was produced.\')\n                self.key=key;self.last_cache_status=self.boot.cache_info[\'status\']\n            except BaseException:\n                self.close();raise\n        self.boot.trace=deque(maxlen=120)  # Small failure ring; no long routine per-frame JSON dumps.\n        self.identity=(p1,p2);self.elapsed=0\n        self.wins=(self.raw[\'p1_wins\'],self.raw[\'p2_wins\'])\n        self.write_count=len(self.boot.report[\'ram_writes\'])\n        self.last_boot_seconds=time.perf_counter()-start;self.resets+=1\n        return self.rgb.copy(),dict(self.raw)\n    def step_keys(self,k1=(),k2=()):\n        if self.boot is None:raise RuntimeError(\'Open the match first.\')\n        before=dict(self.raw)\n        if self.mode==\'pve\' and k2:raise RuntimeError(\'Native-CPU PvE must never send keys to P2.\')\n        if self.boot.stage_write_active or len(self.boot.report[\'ram_writes\'])!=self.write_count:\n            raise RuntimeError(\'Post-start RAM setup write detected.\')\n        raw=self.boot.frame_step(k1,k2);self.elapsed+=1\n        validate_character_ids(raw,*self.identity)\n        if self.mode==\'pve\' and (raw[\'difficulty_menu\']!=self.boot.config.difficulty_stars-1 or\n                               raw[\'difficulty_fight\']!=self.boot.config.difficulty_stars-1):\n            raise RuntimeError(\'CPU difficulty readback changed during battle.\')\n        winner,reason=round_result(raw,self.wins)\n        if reason is None and any(raw[f\'p{p}_hp\']>before[f\'p{p}_hp\'] for p in (1,2)):\n            raise RuntimeError(\'Unexpected health refill during a round.\')\n        if reason is None and self.elapsed>=int(self.common[\'max_round_frames\']):reason=\'frame_cap\'\n        if self.on_frame is not None:self.on_frame(self.rgb,dict(raw),tuple(k1),tuple(k2),self.elapsed)\n        return raw,winner,reason\n    def close(self):\n        if self.boot is not None:\n            self.boot.close();self.boot=None\n        self.key=None;self.payload=None\n', 'sf2_menu.py': '\n"""Repair OPTIONS detection in the existing no-state SF2 experiment.\n\nThis module uses a tiny text-mask and star-row layout measured from the user\'s\nactual 2026-09-26 startup video. It does not contain a ROM or game-state snapshot.\nNo OCR, no new dependency, no RAM value writes, no weakened readiness checks.\nIntegrated menu helper. Existing character, CPU and round guards are retained.\nRule-menu recognition below is new and requires a real Kaggle regression run.\n\nThe source of the two address candidates is the original 0xFFFE45 definition\nand its neighbouring byte. Neither is declared correct before observing\nreversible menu changes AND agreement with the visible difficulty stars.\n"""\nfrom __future__ import annotations\n\nimport base64\nimport json\nfrom pathlib import Path\nimport numpy as np\n\nVERSION = "options-visual-readback-fix-1"\n# 102 x 16 binary mask, x=76..177, y=12..27 of the native 320x224 image.\n# Captured from the visible OPTION MODE text, not a complete game image.\n_HEADER_B64 = (\n    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA"\n    "AB4+Px4eM4Aw3j4+AADMbLQwzGwAZsxsbAADMbDAwzGwAZsxsZAADMbDAwzHwAfsxsYAADMeDAwzHw"\n    "AfsxseAADMYDAwzHwAZsxsYAADMYDAwzGwAZsxsZAADMYDAwzGwAZsxsbAAB48Hh4eOYA53j4+AAAA"\n    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA"\n)\n_HEADER = np.unpackbits(np.frombuffer(base64.b64decode(_HEADER_B64), dtype=np.uint8))[:1632].reshape(16,102).astype(bool)\n_CANDIDATES = {\n    "byte_FFFE44": {"address": 0xFFFE44, "type": "|u1"},\n    "byte_FFFE45": {"address": 0xFFFE45, "type": "|u1"},\n}\n\n\ndef _colour_mask(rgb):\n    a = np.asarray(rgb, dtype=np.int16)\n    return (a.max(axis=-1) >= 105) & (a.max(axis=-1)-a.min(axis=-1) >= 55)\n\n\ndef visible_options_stars(rgb):\n    """Return 1..8 only when this exact OPTIONS header and star row are legible.\n\n    This is limited to the verified 320x224 menu layout. Unknown layouts,\n    fades, ambiguous star colours and different screens return None.\n    """\n    rgb = np.asarray(rgb)\n    if rgb.shape != (224,320,3):\n        return None\n    mask = _colour_mask(rgb[12:28,76:178])\n    union = int(np.count_nonzero(mask | _HEADER))\n    score = np.count_nonzero(mask & _HEADER) / max(1, union)\n    if score < 0.88:\n        return None\n    counts = [int(_colour_mask(rgb[48:57,136+8*i:144+8*i]).sum()) for i in range(8)]\n    if any(5 < x < 12 for x in counts):\n        return None\n    active = [x >= 12 for x in counts]\n    n = sum(active)\n    if not 1 <= n <= 8 or active != [True]*n+[False]*(8-n):\n        return None\n    return n\n\n\ndef _register_read_candidates(boot):\n    before = np.asarray(boot.engine.get_ram()).copy()\n    for name, spec in _CANDIDATES.items():\n        boot.engine.data.set_variable("options_fix_"+name, dict(spec))\n    if not np.array_equal(before, np.asarray(boot.engine.get_ram())):\n        raise RuntimeError("Registering read definitions unexpectedly changed RAM.")\n\n\ndef _wait_for_options(boot, max_frames=360):\n    """Wait for a visible, stable menu; NEVER use START as a retry here."""\n    previous = None\n    consecutive = 0\n    for i in range(max_frames+1):\n        n = visible_options_stars(boot.rgb)\n        consecutive = consecutive+1 if n is not None and n == previous else (1 if n is not None else 0)\n        previous = n\n        if consecutive >= 4:\n            return n\n        if i < max_frames:\n            boot.advance(1)\n    boot.shot("options_fix_no_legible_settings_page")\n    raise RuntimeError("OPTIONS header/stars not recognized. Kept the real screen; no blind START retry.")\n\n\ndef _sample(boot, label):\n    stars = _wait_for_options(boot)\n    candidates = {name: int(boot.engine.data.lookup_value("options_fix_"+name)) for name in _CANDIDATES}\n    raw = np.asarray(boot.engine.get_ram()).reshape(-1)\n    row = {\n        "label": label, "frame": int(boot.frame), "visible_stars": int(stars),\n        "expected_menu_value": int(stars-1), "old_or_current_menu_read": int(boot.values["difficulty_menu"]),\n        **candidates,\n        # Flat-array indices are recorded separately; never silently equated\n        # to GameData\'s bus addresses or used to select a RAM writer.\n        "flat_ram_length": int(raw.size),\n        "flat_index_FE44": int(raw[0xFE44]) if raw.size > 0xFE44 else None,\n        "flat_index_FE45": int(raw[0xFE45]) if raw.size > 0xFE45 else None,\n    }\n    boot.report["options_fix"]["readback_samples"].append(row)\n    Path(boot.root, "options_fix_readbacks.json").write_text(\n        json.dumps(boot.report["options_fix"], indent=2), encoding="utf-8")\n    boot.shot("options_fix_"+label)\n    print(f"[OPTIONS] {label}: visible={stars} stars | menu_read={row[\'old_or_current_menu_read\']} | "\n          f"FFFE44={candidates[\'byte_FFFE44\']} FFFE45={candidates[\'byte_FFFE45\']}", flush=True)\n    return row\n\n\ndef _tap_and_check(boot, key, before, label):\n    expected = max(1, before-1) if key == "LEFT" else min(8, before+1)\n    boot.tap(1, key, hold=8, release=48)\n    row = _sample(boot, label)\n    if row["visible_stars"] != expected:\n        raise RuntimeError(f"Visible difficulty did not respond to {key}: expected {expected} stars, "\n                           f"observed {row[\'visible_stars\']}. No automatic acceptance.")\n    return row\n\n\ndef set_difficulty_via_menu_fixed(boot):\n    boot.phase = "boot_and_options"\n    boot.report["options_fix"] = {"version": VERSION, "readback_samples": [],\n        "status": "STARTED", "template_source": "user startup video, OPTION MODE around 35.13 seconds",\n        "ram_value_writes": 0, "corrected_address": None}\n    # The user\'s video confirms this navigation reaches OPTIONS. The old\n    # 68-frame wait started testing during a fade, before the settings appeared.\n    boot.advance(boot.config.boot_idle_frames)\n    boot.shot("title_after_boot")\n    boot.tap(1,"START",release=128)\n    boot.tap(1,"DOWN")\n    boot.tap(1,"DOWN")\n    boot.tap(1,"START",release=240)\n    _wait_for_options(boot)\n    _register_read_candidates(boot)\n    first = _sample(boot,"baseline")\n    samples = [first]\n    n = first["visible_stars"]\n    # Reversible menu experiment; all actual changes come from gamepad keys.\n    for i,key in enumerate(("LEFT","RIGHT","RIGHT","LEFT"),1):\n        row = _tap_and_check(boot,key,n,f"calibration_{i}_{key.lower()}")\n        samples.append(row)\n        n = row["visible_stars"]\n    boot.advance(60)\n    idle = _sample(boot,"idle_stability")\n    samples.append(idle)\n    if idle["visible_stars"] != n or len({r["visible_stars"] for r in samples}) < 2:\n        raise RuntimeError("Difficulty not stable or no reversible variation was recorded.")\n    matches = [name for name in _CANDIDATES\n               if all(r[name] == r["visible_stars"]-1 for r in samples)]\n    if len(matches) != 1:\n        boot.report["options_fix"]["matching_candidates"] = matches\n        raise RuntimeError(f"Difficulty read mapping is ambiguous or absent: {matches}. "\n                           "See options_fix_readbacks.json; no guess was installed.")\n    chosen = matches[0]\n    spec = _CANDIDATES[chosen]\n    # Rebind HOW the variable is read. Do not modify any RAM contents.\n    before = np.asarray(boot.engine.get_ram()).copy()\n    boot.engine.data.set_variable("boot_difficulty_menu",dict(spec))\n    if not np.array_equal(before,np.asarray(boot.engine.get_ram())):\n        raise RuntimeError("Read-definition rebinding unexpectedly changed RAM.")\n    boot.values = boot.read()\n    boot.report["options_fix"].update(corrected_address=hex(spec["address"]),dtype=spec["type"],\n                                     chosen_candidate=chosen)\n    target = boot.config.difficulty_stars\n    for i in range(8):\n        if n == target:\n            break\n        key = "LEFT" if n > target else "RIGHT"\n        row = _tap_and_check(boot,key,n,f"set_target_{i+1}")\n        n = row["visible_stars"]\n        if boot.values["difficulty_menu"] != n-1:\n            raise RuntimeError("Selected read mapping stopped matching the visible stars.")\n    boot.advance(60)\n    final = _sample(boot,"target_set")\n    if final["visible_stars"] != target or boot.values["difficulty_menu"] != target-1:\n        raise RuntimeError("Final difficulty differs between the screen, readback and requested setting.")\n    boot.report["difficulty_set_via_menu"] = boot.values["difficulty_menu"]\n    boot.report["options_fix"]["status"] = "MENU_DIFFICULTY_VISUAL_AND_READBACK_MATCH"\n    Path(boot.root,"options_fix_readbacks.json").write_text(\n        json.dumps(boot.report["options_fix"],indent=2),encoding="utf-8")\n    boot.shot("options_difficulty_set")\n    print(f"OPTIONS FIX CHECK PASSED: {target} visible stars; read mapping {hex(spec[\'address\'])}.",flush=True)\n    # Exit only after successful visual AND numeric agreement. This is the\n    # first START sent after entering the settings screen.\n    boot.tap(1,"START",release=120)\n\n\n\n# Text masks measured from the user\'s real startup video (30.0s / 41.0s).\nMENU_TEXT_B64 = {\'rules\': \'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAHz/GMN8PHzHAAAMZmGGZmGMZmAAAMZmPH5mGMZ2AAAMB+JH5mGMZ+AAAMZmfmZ8GMZuAAAMZmQmZgGMZmAAAHz/5+fwPHzjAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAP/nfHx8AAAQEBAGZmZmJmAAAQEBAGY8ZmBmAAD+/v4H4YZnh8AAB8fHwGYYfGBmAAA4ODgGYYYGJmAAB8fHwP888Pz2AABEREQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAHx8fjx8x3wAAAAMZmmRjGZsYAAAAMZmGBjGdsAAAAAMZmGBjGfnwAAAAMZ8GBjGbgYAAAAMZgGBjGZsYAAAAHzwPDx843wAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA\', \'game\': \'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAHwa03wAfH44fH4MYaZmIAxpkYZpkN48fmAAwBg9ZngMgkfngAfRhkfBgM5+ZuAABhh+ZhgNZCZmIAxtlCZhgH7n5/wAfD3n9nwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAE4AfAAAfBh+fvBMQAxgAAZpiZmWBGwAwAAAZjwYeGBCgAfAAAfKQYGGBLgABgAAZn4a2GBFAwxjAAZkIYGGJBAwfDAA/Oc8PPzICAAIAAAAABAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAH18fO98AH2Yfn7MZmxmZmAGaYmZlN5mxmZmAGY8GBhNh9xuZmAPwkGBhM5mxm58AGZ+GppMZmxmZgAGZCGBhH72fDzwAPznfHzAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA\'}\n\n_MENU_MASKS = {k:np.unpackbits(np.frombuffer(base64.b64decode(v),dtype=np.uint8))[:48*84].reshape(48,84).astype(bool) for k,v in MENU_TEXT_B64.items()}\n\n\ndef menu_kind(rgb):\n    if np.asarray(rgb).shape != (224,320,3):\n        return None\n    actual = _colour_mask(rgb[164:212,86:170])\n    scores={k:np.count_nonzero(actual & v)/max(1,np.count_nonzero(actual | v)) for k,v in _MENU_MASKS.items()}\n    best=max(scores,key=scores.get)\n    # Exclude the HYPER star area for rule matching; speed is checked separately.\n    rule_a=actual[:,:56];rule_t=_MENU_MASKS[\'rules\'][:,:56]\n    rule_score=np.count_nonzero(rule_a & rule_t)/max(1,np.count_nonzero(rule_a | rule_t))\n    if rule_score>.82:\n        return \'rules\'\n    return \'game\' if scores[\'game\']>.82 else None\n\n\ndef cursor_row(rgb):\n    counts=[int(_colour_mask(rgb[164+16*i:180+16*i,67:84]).sum()) for i in range(3)]\n    on=[i for i,n in enumerate(counts) if n>=12]\n    return on[0] if len(on)==1 else None\n\n\ndef visible_speed_stars(rgb):\n    if menu_kind(rgb)!=\'rules\':\n        return None\n    counts=[int(_colour_mask(rgb[184:193,144+8*i:152+8*i]).sum()) for i in range(10)]\n    if any(5<n<12 for n in counts):\n        return None\n    bits=[n>=12 for n in counts];n=sum(bits)\n    return n if bits==[True]*n+[False]*(10-n) else None\n\n\ndef _seek_menu(boot, wanted, allow_start, budget=2200):\n    for i in range(budget):\n        if menu_kind(boot.rgb)==wanted and cursor_row(boot.rgb) is not None:\n            boot.advance(4)\n            if menu_kind(boot.rgb)==wanted and cursor_row(boot.rgb) is not None:\n                return\n        # START only while seeking the title/menu from intro; NEVER use directions on unknown pages.\n        if allow_start and i and i%240==0:\n            boot.tap(1,\'START\',release=60)\n        else:\n            boot.advance(1)\n    boot.shot(\'unrecognized_\'+wanted+\'_menu\')\n    raise RuntimeError(\'Expected \'+wanted+\' menu not recognized. Inspect actual startup screenshots.\')\n\n\ndef _select_row(boot,wanted,row):\n    for _ in range(4):\n        if menu_kind(boot.rgb)!=wanted:\n            raise RuntimeError(\'Menu changed before row selection.\')\n        current=cursor_row(boot.rgb)\n        if current==row:return\n        if current is None:\n            boot.advance(8);continue\n        boot.tap(1,\'DOWN\' if current<row else \'UP\',release=32)\n    raise RuntimeError(\'Cursor did not reach requested menu row.\')\n\n\ndef select_rules_and_game_start(boot):\n    """Explicitly choose rule set/speed; no character-probe keys on mode pages.\n\n    Menu templates were replay-tested against the user\'s old video. The full\n    new navigation path is NOT yet certified on a real emulator here.\n    """\n    boot.phase=\'select_rules_and_speed\'\n    _seek_menu(boot,\'rules\',allow_start=True)\n    wanted=0 if boot.config.ruleset==\'champion\' else 1\n    _select_row(boot,\'rules\',wanted)\n    if wanted==1:\n        for _ in range(8):\n            n=visible_speed_stars(boot.rgb)\n            if n is None:boot.advance(8);continue\n            if n==boot.config.speed_stars:break\n            boot.tap(1,\'LEFT\' if n>boot.config.speed_stars else \'RIGHT\',release=32)\n        if visible_speed_stars(boot.rgb)!=boot.config.speed_stars:\n            boot.shot(\'speed_not_verified\')\n            raise RuntimeError(\'HYPER speed stars not equal to requested setting.\')\n    boot.shot(\'rules_and_speed_selected\')\n    boot.report[\'ruleset_selection\']={\'ruleset\':boot.config.ruleset,\'speed_stars\':boot.config.speed_stars if wanted else None,\'method\':\'actual rule-menu text + cursor + stars\',\'visual_menu_checked\':True,\'combat_ruleset_ram_certified\':False}\n    boot.tap(1,\'START\',release=90)\n    _seek_menu(boot,\'game\',allow_start=False,budget=720)\n    _select_row(boot,\'game\',0)\n    boot.shot(\'native_game_start_selected\')\n    boot.tap(1,\'START\',release=120)\n', 'sf2_observation.py': '"""Versioned, per-agent image preprocessing plus the original six positions.\nEach exported agent carries its own exact student_observation.py source and config.\nNo control-state input, frame differences or extra geometry features.\n"""\nfrom collections import deque\nfrom pathlib import Path\nimport hashlib\nimport json\nimport sys\nimport types\nimport numpy as np\n\n_MODULES = {}\n\n\ndef _validate_config(c):\n    required = {"color", "width", "height", "stack_frames", "raw_shape", "crop_xyxy",\n                "verify_right_padding", "black_threshold", "max_nonblack_fraction"}\n    if not isinstance(c, dict) or set(c) != required:\n        raise ValueError("INPUT_CONFIG has missing or unsupported fields.")\n    if c[\'color\'] not in (\'gray\', \'rgb\') or c[\'stack_frames\'] != 4:\n        raise ValueError("This upgrade supports four gray or four RGB frames.")\n    if any(type(c[k]) is not int or not 64 <= c[k] <= 512 for k in (\'height\', \'width\')):\n        raise ValueError("Input height/width must be integer values between 64 and 512.")\n    if c[\'raw_shape\'] != [224, 320, 3]:\n        raise ValueError("This game integration expects a 224 x 320 RGB emulator buffer.")\n    box = c[\'crop_xyxy\']\n    if not isinstance(box, list) or len(box) != 4 or any(type(x) is not int for x in box):\n        raise ValueError("crop_xyxy must contain four integers.")\n    x0, y0, x1, y1 = box\n    if not (0 <= x0 < x1 <= 320 and 0 <= y0 < y1 <= 224):\n        raise ValueError("Crop is outside the raw image.")\n    if type(c[\'verify_right_padding\']) is not bool:\n        raise ValueError("verify_right_padding must be a bool.")\n    if type(c[\'black_threshold\']) is not int or not 0 <= c[\'black_threshold\'] <= 255:\n        raise ValueError("Invalid black threshold.")\n    v = c[\'max_nonblack_fraction\']\n    if not isinstance(v, (int, float)) or not np.isfinite(v) or not 0 <= v <= 0.01:\n        raise ValueError("Invalid padding tolerance (must be 0..0.01).")\n    return json.loads(json.dumps(c, allow_nan=False))\n\n\ndef _load_source(text):\n    if not isinstance(text, str) or not 0 < len(text.encode(\'utf-8\')) <= 256 * 1024:\n        raise ValueError("Invalid preprocessing source size.")\n    digest = hashlib.sha256(text.encode(\'utf-8\')).hexdigest()\n    if digest not in _MODULES:\n        name = \'sf2_observation_source_\' + digest\n        module = types.ModuleType(name)\n        module.__file__ = \'<student_observation:\' + digest + \'>\'\n        # Student files and external agent packages must already be trusted.\n        # Content hashes detect drift; they do not sandbox arbitrary Python.\n        exec(compile(text, module.__file__, \'exec\'), module.__dict__)\n        config = _validate_config(module.INPUT_CONFIG)\n        if not callable(getattr(module, \'preprocess\', None)):\n            raise TypeError("student_observation.py must define preprocess(rgb).")\n        _MODULES[digest] = (module, config)\n    return digest, _MODULES[digest]\n\n\ndef build_descriptor(source_path):\n    text = Path(source_path).read_text(encoding=\'utf-8\')\n    digest, (_, config) = _load_source(text)\n    result = {\'schema\': \'sf2-student-observation-v1\', \'source_sha256\': digest,\n              \'config\': config, \'source\': text}\n    validate_descriptor(result)\n    return result\n\n\ndef validate_descriptor(d):\n    if not isinstance(d, dict) or set(d) != {\'schema\', \'source_sha256\', \'config\', \'source\'}:\n        raise ValueError("Missing versioned observation descriptor; old input packages are not converted.")\n    if d[\'schema\'] != \'sf2-student-observation-v1\':\n        raise ValueError("Unsupported observation descriptor.")\n    digest, (module, config) = _load_source(d[\'source\'])\n    if digest != d[\'source_sha256\'] or config != d[\'config\']:\n        raise ValueError("Observation source/config/hash do not agree.")\n    return module, config\n\n\ndef image_shape(spec):\n    _, c = validate_descriptor(spec[\'observation\'])\n    channels = 1 if c[\'color\'] == \'gray\' else 3\n    return (c[\'stack_frames\'] * channels, c[\'height\'], c[\'width\'])\n\n\ndef description(spec):\n    _, c = validate_descriptor(spec[\'observation\'])\n    return {\'protocol\': \'sf2-configurable-image4-position6-v3\',\n            \'source_sha256\': spec[\'observation\'][\'source_sha256\'],\n            \'config\': c, \'image_shape\': list(image_shape(spec)),\n            \'dtype\': \'uint8\', \'normalization\': \'SB3 divides by 255 once\',\n            \'channel_order\': \'oldest to latest, then R/G/B within each RGB frame\',\n            \'decision_frames\': 6, \'position_count\': 6 if spec.get(\'use_position\', True) else 0}\n\n\ndef position(raw, player, scale=1024.0):\n    if player not in (1, 2) or not np.isfinite(scale) or scale <= 0:\n        raise ValueError(\'Invalid player or position scale.\')\n    values = np.array([raw[f\'p{player}_x\'], raw[f\'p{player}_y\'],\n                       raw[f\'p{3-player}_x\'], raw[f\'p{3-player}_y\']], np.float64)\n    if not np.isfinite(values).all() or (values < 0).any() or (values > 65535).any():\n        raise ValueError(\'Invalid coordinate readback.\')\n    x, y, ox, oy = values\n    return (np.array([x, y, ox, oy, ox-x, oy-y])/scale).astype(np.float32)\n\n\nclass Side:\n    def __init__(self, spec, player, scale=1024.0):\n        self.spec, self.player, self.scale = spec, player, scale\n        self.module, self.config = validate_descriptor(spec[\'observation\'])\n        self.channels = 1 if self.config[\'color\'] == \'gray\' else 3\n        self.frames = deque(maxlen=4)\n        self.frame_ids = deque(maxlen=4)\n        self.observed_frames_checked = 0\n\n    def _frame(self, rgb):\n        a = np.asarray(self.module.preprocess(rgb))\n        expected = (self.channels, self.config[\'height\'], self.config[\'width\'])\n        if a.dtype != np.uint8 or tuple(a.shape) != expected:\n            raise ValueError(f"Preprocessor returned {a.shape} {a.dtype}; expected {expected} uint8.")\n        self.observed_frames_checked += 1\n        return np.ascontiguousarray(a).copy()\n\n    def reset(self, rgb, frame_id=0):\n        self.frames.clear(); self.frame_ids.clear()\n        a = self._frame(rgb)\n        for _ in range(4):\n            self.frames.append(a.copy()); self.frame_ids.append(int(frame_id))\n\n    def update(self, rgb, frame_id=None):\n        if not self.frames:\n            raise RuntimeError("Initialize the frame history before updating it.")\n        n = self.frame_ids[-1]+6 if frame_id is None else int(frame_id)\n        if n <= self.frame_ids[-1]:\n            raise ValueError("Frame IDs must increase within an episode.")\n        self.frames.append(self._frame(rgb)); self.frame_ids.append(n)\n\n    def observe(self, raw):\n        if len(self.frames) != 4:\n            raise RuntimeError("Expected a complete four-frame history.")\n        image = np.concatenate(tuple(self.frames), axis=0)\n        if not self.spec.get(\'use_position\', True):\n            return image\n        return {\'image\': image, \'position\': position(raw, self.player, self.scale)}\n\n\ndef observation_space(use_position=True, scale=1024.0, *, spec):\n    from gymnasium import spaces\n    if bool(spec.get(\'use_position\', True)) != bool(use_position):\n        raise ValueError("Position flag disagrees with agent spec.")\n    if not np.isfinite(scale) or scale <= 0:\n        raise ValueError("Invalid coordinate scale.")\n    image = spaces.Box(0, 255, image_shape(spec), np.uint8)\n    if not use_position:\n        return image\n    bound = np.float32(65535/scale)\n    return spaces.Dict({\'image\': image, \'position\': spaces.Box(\n        np.array([0, 0, 0, 0, -bound, -bound], np.float32),\n        np.full(6, bound, np.float32), dtype=np.float32)})\n\n\ndef display_frames(image, spec):\n    if tuple(image.shape) != image_shape(spec) or image.dtype != np.uint8:\n        raise ValueError("Image tensor differs from its exported observation contract.")\n    channels = 1 if spec[\'observation\'][\'config\'][\'color\'] == \'gray\' else 3\n    return [image[i*channels:(i+1)*channels][0] if channels == 1\n            else image[i*channels:(i+1)*channels].transpose(1, 2, 0) for i in range(4)]\n', 'sf2_pvp_eval.py': '"""Saved-policy evaluation against an actual frozen agent, from learner ports.\nNo optimisers, no native-CPU evaluation fallback, no opponent replacement.\n"""\nfrom pathlib import Path\nimport numpy as np\nfrom sf2_bundle import LoadedAgent\nfrom sf2_common import parameter_hash, file_hash, rules_signature, source_module, json_write\nfrom sf2_env import BattleEnv\nfrom sf2_release import validate_frozen_config, validate_pvp_evaluation\n\n\ndef summarize_pvp(rows):\n    n=len(rows)\n    counts={name:sum(row[\'result\']==name for row in rows)\n            for name in (\'win\',\'loss\',\'draw\',\'cutoff\')}\n    return {**counts,\'episodes\':n,\'win_fraction\':counts[\'win\']/n if n else None,\n            \'mean_hp_margin\':float(np.mean([max(0,r[\'self_hp\'])-max(0,r[\'opponent_hp\'])\n                                           for r in rows])) if n else None,\n            \'mean_episode_reward\':float(np.mean([r[\'episode_reward\'] for r in rows])) if n else None}\n\n\ndef evaluate_frozen(job, out):\n    import torch\n    from sf2_video import Video\n    cfg=dict(job[\'evaluation\'])\n    eval_cfg=validate_pvp_evaluation({k:cfg[k] for k in\n        (\'offsets\',\'seed\',\'deterministic\',\'record_video\') if k in cfg})\n    frozen=validate_frozen_config(job[\'frozen\'])\n    if not cfg.get(\'bundle\'):raise ValueError(\'Select the learner package for PvP evaluation.\')\n    packages=[(\'trained\',cfg[\'bundle\'])]\n    if cfg.get(\'compare_initial\',False):\n        if not cfg.get(\'initial_bundle\'):raise ValueError(\'Set initial_bundle for a before/after comparison.\')\n        packages.insert(0,(\'initial\',cfg[\'initial_bundle\']))\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    opponent_sha=file_hash(frozen[\'bundle\'])\n    rows=[];videos=[];unchanged=[]\n    for label,package in packages:\n        learner=opponent=env=rec=None\n        try:\n            learner=LoadedAgent(package,trusted=job.get(\'trust_packages\',False),device=\'cpu\')\n            opponent=LoadedAgent(frozen[\'bundle\'],trusted=job.get(\'trust_packages\',False),device=\'cpu\')\n            for actor in (learner,opponent):\n                if rules_signature(actor.common)!=rules_signature(job[\'common\']):\n                    raise ValueError(\'A PvP package uses different arena rules.\')\n            start_hash=parameter_hash(learner.model)\n            reward=source_module(learner.directory/\'reward.py\',\'reward\').compute_reward\n            # difficulty_stars=1 is an internal native-menu initializer only.\n            # In mode=\'pvp\' both ports are controlled by models, never the CPU AI.\n            pair={\'character\':opponent.spec[\'character\'],\'difficulty_stars\':1,\n                  \'neutral_jitter_max\':frozen[\'neutral_jitter_max\']}\n            env=BattleEnv(learner.spec,job[\'common\'],[pair],out/label,reward,\n                opponent_agent=opponent,learner_sides=tuple(frozen[\'learner_sides\']),\n                opponent_deterministic=frozen[\'opponent_deterministic\'])\n            for player in frozen[\'learner_sides\']:\n                env.learner_sides=(player,)\n                for index,offset in enumerate(eval_cfg[\'offsets\']):\n                    # Repeated no-op offsets are scenarios, not independent CPU seeds.\n                    # Seed only this separate evaluation process; training RNGs are untouched.\n                    seed=eval_cfg[\'seed\']+player*1000+index\n                    torch.manual_seed(seed)\n                    obs,reset=env.reset(seed=seed,options={\'offset\':offset})\n                    if env.player!=player:raise RuntimeError(\'Wrong learner port in PvP evaluation.\')\n                    if eval_cfg[\'record_video\'] and index==0:\n                        rec=Video(out/f\'{label}_P{player}.mp4\',env.game.fps)\n                        def frame(rgb,raw,k1,k2,number):\n                            rec.write(rgb,[f\'PVP EVALUATION | {label} learner=P{player} | frozen=P{3-player}\',\n                                f\'P1 HP={raw["p1_hp"]} | P2 HP={raw["p2_hp"]} | offset={offset} | frame={number}\',\n                                \'No learning. The frozen opponent is not the native CPU.\'],k1,k2)\n                        env.game.on_frame=frame\n                    reward_sum=0.;count={};decisions=0\n                    while True:\n                        a=learner.act(env.sides[player],env.game.raw,\n                                      deterministic=eval_cfg[\'deterministic\'])\n                        obs,r,t,tr,info=env.step(a)\n                        reward_sum+=r;decisions+=1\n                        name=info[\'action_name\'];count[name]=count.get(name,0)+1\n                        if env.sides[1].frame_ids[-1]!=env.sides[2].frame_ids[-1]:\n                            raise RuntimeError(\'PvP actors observed different decision clocks.\')\n                        if t or tr:break\n                    row={k:info[k] for k in (\'result\',\'end_reason\',\'self_hp\',\'opponent_hp\',\'native_frames\')}\n                    row.update(controller=label,learner_player=player,opponent_player=3-player,\n                        offset=offset,seed=seed,episode_reward=reward_sum,decisions=decisions,\n                        learner_deterministic=eval_cfg[\'deterministic\'],\n                        opponent_deterministic=frozen[\'opponent_deterministic\'],\n                        initial_readings=reset[\'raw\'],action_counts=count)\n                    rows.append(row)\n                    print(f\'EVAL_PVP {label}: learner=P{player}, opponent=P{3-player}, offset={offset}, \'\n                          f\'{row["result"]}, learner/opponent HP={row["self_hp"]}/{row["opponent_hp"]}, \'\n                          f\'reward={reward_sum:+.4f}, reason={row["end_reason"]}\',flush=True)\n                    if rec:videos.append(rec.finish());rec=None\n                    env.game.on_frame=None\n            stats=env.performance_stats()\n            if parameter_hash(learner.model)!=start_hash or not stats[\'frozen_opponent_unchanged\']:\n                raise RuntimeError(\'A policy changed during no-learning PvP evaluation.\')\n            unchanged.append({\'controller\':label,\'learner_package_sha256\':file_hash(package),\n                              \'learner_unchanged\':True,\'frozen_opponent_unchanged\':True})\n        finally:\n            if rec:rec.close()\n            if env:env.close()  # owns opponent\n            elif opponent:opponent.close()\n            if learner:learner.close()\n    if file_hash(frozen[\'bundle\'])!=opponent_sha:\n        raise RuntimeError(\'Frozen package bytes changed during evaluation.\')\n    groups={label:summarize_pvp([r for r in rows if r[\'controller\']==label]) for label,_ in packages}\n    ports={label:{str(p):summarize_pvp([r for r in rows if r[\'controller\']==label and r[\'learner_player\']==p])\n                  for p in frozen[\'learner_sides\']} for label,_ in packages}\n    print(\'PVP EVALUATION SUMMARY:\',groups,flush=True)\n    result={\'status\':\'PVP_EVALUATION_COMPLETE\',\'training_performed\':False,\n        \'validation_target\':\'frozen_agent_pvp\',\'episodes\':rows,\'summary\':groups,\'per_port\':ports,\n        \'learner_sides\':frozen[\'learner_sides\'],\'opponent_sha256\':opponent_sha,\n        \'evaluation_settings\':eval_cfg,\'frozen_opponent_unchanged\':True,\'policy_checks\':unchanged,\n        \'win_fraction\':groups[\'trained\'][\'win_fraction\'],\'mean_hp_margin\':groups[\'trained\'][\'mean_hp_margin\'],\n        \'videos\':videos,\'note\':\'Every result is from the learner perspective. Fixed offsets are scenarios, not independent opponents. No general-strength claim.\'}\n    return result\n', 'sf2_release.py': '"""Small delivery helpers. No changes to the default PPO objective or game rules.\n\nA saved learner can optionally initialise a NEW training phase. Only policy\nweights are copied; the optimiser, rollout buffer and step count are new.\nThis is deliberately not advertised as an exact interrupted-run resume.\n"""\nfrom pathlib import Path\nimport hashlib\nimport json\n\n\ndef validate_frozen_config(frozen):\n    """One fixed opponent and explicit learner ports; no native-CPU difficulty."""\n    if not isinstance(frozen, dict):\n        raise TypeError(\'PvP opponent settings must be a dictionary.\')\n    permitted={\'bundle\',\'learner_sides\',\'opponent_deterministic\',\'neutral_jitter_max\'}\n    if set(frozen)-permitted:\n        raise ValueError(\'Unsupported PvP settings: \'+str(sorted(set(frozen)-permitted)))\n    bundle=frozen.get(\'bundle\',\'\')\n    if not isinstance(bundle,str) or not bundle.strip():\n        raise ValueError(\'Set the fixed opponent bundle path explicitly. It is not the learner initialisation path.\')\n    sides=frozen.get(\'learner_sides\',[2])\n    if (not isinstance(sides,(tuple,list)) or not sides\n        or any(type(p) is not int or p not in (1,2) for p in sides)\n        or len(set(sides))!=len(sides)):\n        raise ValueError(\'learner_sides must be [1], [2], [1,2] or [2,1]. P2 is a port, not a permanent screen side.\')\n    deterministic=frozen.get(\'opponent_deterministic\',True)\n    if type(deterministic) is not bool:raise ValueError(\'opponent_deterministic must be True or False.\')\n    jitter=frozen.get(\'neutral_jitter_max\',60)\n    if type(jitter) is not int or not 0<=jitter<=60:raise ValueError(\'PvP neutral_jitter_max must be 0..60.\')\n    return {\'bundle\':str(Path(bundle).expanduser().resolve()),\'learner_sides\':list(sides),\n            \'opponent_deterministic\':deterministic,\'neutral_jitter_max\':jitter}\n\n\ndef validate_pvp_evaluation(config):\n    """Same requested ports/opponent as training; seed/offsets describe scenarios."""\n    c=dict(config or {})\n    allowed={\'offsets\',\'seed\',\'deterministic\',\'record_video\'}\n    if set(c)-allowed:raise ValueError(\'Unsupported PvP validation field: \'+str(sorted(set(c)-allowed)))\n    offsets=c.get(\'offsets\',[0,12,24,36,48])\n    if (not isinstance(offsets,list) or not offsets or len(offsets)>61\n        or any(type(n) is not int or not 0<=n<=60 for n in offsets)\n        or len(offsets)!=len(set(offsets))):\n        raise ValueError(\'Use distinct integer PvP offsets in 0..60.\')\n    seed=c.get(\'seed\',2025)\n    if type(seed) is not int or not 0<=seed<2**31:raise ValueError(\'PvP evaluation seed must be an integer in 0..2**31-1.\')\n    for name in (\'deterministic\',\'record_video\'):\n        if name in c and type(c[name]) is not bool:raise ValueError(name+\' must be True or False.\')\n    return {\'offsets\':offsets,\'seed\':seed,\'deterministic\':c.get(\'deterministic\',True),\n            \'record_video\':c.get(\'record_video\',False)}\n\n\ndef frozen_check_fingerprint(root, agent, common, pve, frozen, *, initialization_bundle=\'\', train=None):\n    from lab_notebook import input_check_fingerprint\n    c=validate_frozen_config(frozen)\n    path=Path(c[\'bundle\'])\n    if not path.is_file():raise FileNotFoundError(\'Frozen opponent package does not exist: \'+str(path))\n    initial=Path(initialization_bundle).expanduser().resolve() if initialization_bundle else None\n    if initial is not None and not initial.is_file():raise FileNotFoundError(\'Initial learner package does not exist: \'+str(initial))\n    return {\'learner\':input_check_fingerprint(root,agent,common,None,profile=\'pvp\'),\n            \'opponent_sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\n            \'initialization_sha256\':hashlib.sha256(initial.read_bytes()).hexdigest() if initial else None,\n            \'learner_sides\':c[\'learner_sides\'],\'opponent_deterministic\':c[\'opponent_deterministic\'],\n            \'neutral_jitter_max\':c[\'neutral_jitter_max\'], \'train\':dict(train or {})}\n\n\ndef initialise_policy(model, spec, source, common, package, trusted=False):\n    """Optional own-agent phase initialisation, with strict input/action checks."""\n    from sf2_bundle import LoadedAgent\n    from sf2_common import rules_signature, file_hash, parameter_hash\n    if not package:\n        return {\'mode\': \'from_scratch\', \'initial_parameter_sha256\': parameter_hash(model)}\n    parent = LoadedAgent(package, trusted=trusted, device=\'cpu\')\n    try:\n        # Team display names do not change the learned control task.\n        a = {k: v for k, v in spec.items() if k != \'team_name\'}\n        b = {k: v for k, v in parent.spec.items() if k != \'team_name\'}\n        if a != b:\n            raise ValueError(\'Initial learner differs in character, actions or exact input source. Start from scratch or use matching student files.\')\n        if rules_signature(common) != rules_signature(parent.common):\n            raise ValueError(\'Initial learner uses different arena rules.\')\n        if file_hash(Path(source) / \'student_model.py\') != file_hash(parent.directory / \'model.py\'):\n            raise ValueError(\'Initial learner model source differs. Do not silently reinterpret existing weights.\')\n        expected = parameter_hash(parent.model)\n        model.policy.load_state_dict(parent.model.policy.state_dict(), strict=True)\n        actual = parameter_hash(model)\n        if actual != expected:\n            raise RuntimeError(\'Policy initialisation did not copy parameters exactly.\')\n        return {\n            \'mode\': \'saved_policy_new_optimizer\',\n            \'package\': str(Path(package).expanduser().resolve()),\n            \'package_sha256\': file_hash(package),\n            \'source_steps\': int(parent.model.num_timesteps),\n            \'initial_parameter_sha256\': actual,\n            \'optimizer_restored\': False,\n            \'note\': \'This phase starts at step 0 with a new optimizer and new environment histories. It is not an exact resume.\',\n        }\n    finally:\n        parent.close()\n\n\ndef read_package_metadata(path):\n    """Read plain JSON metadata without loading Python or deserialising weights."""\n    import zipfile\n    path = Path(path).expanduser().resolve()\n    with zipfile.ZipFile(path) as z:\n        result = {}\n        for name in (\'manifest.json\', \'spec.json\', \'training.json\', \'common.json\'):\n            info = z.getinfo(name)\n            if info.file_size > 2 * 1024 * 1024:\n                raise ValueError(\'Metadata too large: \' + name)\n            result[name] = json.loads(z.read(name))\n    return result\n\n\ndef validation_target(job):\n    return \'frozen_agent_pvp\' if job[\'kind\']==\'train_frozen\' else \'native_cpu_pve_P1\'\n\n\ndef validation_job(job, package, destination):\n    """Select the correct validation task; PvP can never fall back to CPU PvE."""\n    import copy\n    j=copy.deepcopy(job)\n    j.update(out=str(destination),trust_packages=True)\n    if job[\'kind\']==\'train_frozen\':\n        cfg=validate_pvp_evaluation(job.get(\'pvp_validation\',{}))\n        cfg[\'record_video\']=False\n        j.update(kind=\'evaluate_frozen\',pve={},\n                 evaluation={\'bundle\':str(package),\'initial_bundle\':\'\',\'compare_initial\':False,**cfg})\n        j[\'frozen\']=validate_frozen_config(job[\'frozen\'])\n    else:\n        j.update(kind=\'evaluate\',evaluation={\'bundle\':str(package),\n            \'offsets\':job.get(\'validation_offsets\',[0,12,24,36,48]),\n            \'include_random\':False,\'record_video\':False,\'deterministic\':True})\n    return j\n\n\ndef resolve_pvp_setup(setup):\n    """Explicit learner initialisation, never an implicit \'latest PvE\' fallback."""\n    required={\'learner_start\',\'learner_bundle\',\'opponent_bundle\',\'learner_sides\',\n              \'opponent_deterministic\',\'neutral_jitter_max\'}\n    if not isinstance(setup,dict) or set(setup)!=required:\n        raise ValueError(\'PVP_SETUP fields do not match the notebook template.\')\n    mode=setup[\'learner_start\'];bundle=setup[\'learner_bundle\']\n    if mode not in (\'package\',\'scratch\'):raise ValueError(\'learner_start must be package or scratch.\')\n    if not isinstance(bundle,str):raise ValueError(\'learner_bundle must be a path string.\')\n    if mode==\'package\' and not bundle.strip():\n        raise ValueError(\'Set learner_bundle to the model to continue learning, or explicitly choose learner_start="scratch".\')\n    if mode==\'scratch\' and bundle.strip():\n        raise ValueError(\'learner_start="scratch" requires learner_bundle="". Do not leave a misleading old path.\')\n    initial=str(Path(bundle).expanduser().resolve()) if bundle else \'\'\n    frozen=validate_frozen_config({\'bundle\':setup[\'opponent_bundle\'],\n        \'learner_sides\':setup[\'learner_sides\'],\'opponent_deterministic\':setup[\'opponent_deterministic\'],\n        \'neutral_jitter_max\':setup[\'neutral_jitter_max\']})\n    for role,path in [(\'initial learner\',initial),(\'fixed opponent\',frozen[\'bundle\'])]:\n        if path and not Path(path).is_file():raise FileNotFoundError(role+\' package not found: \'+path)\n    print(\'PvP learner:\',initial if initial else \'NEW RANDOM MODEL\',flush=True)\n    print(\'PvP frozen opponent:\',frozen[\'bundle\'],flush=True)\n    print(\'Learner ports:\',frozen[\'learner_sides\'],\'| opponent ports:\',[3-p for p in frozen[\'learner_sides\']],flush=True)\n    return initial,frozen\n', 'sf2_start_cache.py': '"""Verified match-start caching for the existing SF2 Student Lab.\n\nOnly emulator state is restored. Python/NumPy/Torch RNGs, model parameters,\noptimizers and the learner\'s RNG are NOT serialized or reset by this module.\nThe classroom environment applies a fresh neutral offset and resets image\nhistory, reward totals and terminal bookkeeping. No long-action queue is used.\n\nA cache is an automatically generated emulator savestate, not a new CPU seed.\nSame complete start + same input sequence may replay exactly the same fight.\nNo addresses are guessed or written to randomize the native CPU.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nfrom dataclasses import asdict\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport tempfile\nimport time\nimport zipfile\n\nimport numpy as np\nfrom sf2_boot import ColdBootMatch, GAME, ROM_SHA1, RAM_FIELDS, bcd_seconds, validate_character_ids\n\nCACHE_VERSION = \'sf2-classroom-verified-start-v2\'\nMAX_PAYLOAD_BYTES = 64 * 1024 * 1024\n_MEMO = {}\n_FAMILY = {}\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef array_digest(array):\n    a = np.ascontiguousarray(array)\n    return digest(str((a.shape, a.dtype.str)).encode() + a.tobytes())\n\n\ndef file_digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for part in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef _json(data):\n    return json.dumps(data, sort_keys=True, ensure_ascii=False, separators=(\',\', \':\'))\n\n\ndef _write_json(path, data):\n    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)\n    p.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n\n\ndef enabled():\n    return True  # Integrated runtime; no notebook monkey-patch or enable flag required.\n\n\ndef cache_root():\n    configured = os.environ.get(\'SF2_START_CACHE_DIR\')\n    if configured:\n        return Path(configured).expanduser().resolve()\n    return Path(__file__).resolve().parent.parent / \'cache\' / \'match_starts_v2\'\n\n\ndef _family(backend):\n    """Invalidate for changed emulator binaries, relevant source or ABI."""\n    key = (str(Path(backend.__file__).resolve()), CACHE_VERSION)\n    if key not in _FAMILY:\n        root = Path(backend.__file__).resolve().parent\n        paths = {Path(backend.__file__).resolve()}\n        for p in root.rglob(\'*\'):\n            if not p.is_file():\n                continue\n            if p.suffix.lower() in (\'.so\', \'.pyd\', \'.dll\', \'.dylib\'):\n                # Stable-retro\'s Python binding and the Genesis core.\n                if p.parent == root or \'genesis\' in p.name.lower():\n                    paths.add(p)\n        version = getattr(backend, \'__version__\', None)\n        try:\n            version = importlib.metadata.version(\'stable-retro\')\n        except importlib.metadata.PackageNotFoundError:\n            pass\n        if not version:\n            raise RuntimeError(\'Cannot identify stable-retro version; refusing an unversioned state cache.\')\n        src = Path(__file__).resolve().parent\n        _FAMILY[key] = {\n            \'cache_version\': CACHE_VERSION,\n            \'game\': GAME, \'rom_sha1\': ROM_SHA1,\n            \'backend_version\': str(version),\n            \'python\': platform.python_version(), \'machine\': platform.machine(),\n            \'backend_binaries\': {str(p.relative_to(root)): file_digest(p) for p in sorted(paths)},\n            \'source_hashes\': {name: file_digest(src / name) for name in\n                              (\'sf2_boot.py\', \'sf2_menu.py\', \'sf2_start_cache.py\')},\n        }\n    return _FAMILY[key]\n\n\ndef descriptor(boot):\n    # Deliberately not based on the learned network or reward function.\n    # Include the full bootstrap configuration so future setup fields cannot\n    # accidentally reuse an incompatible cached match.\n    return {\'family\': _family(boot.backend), \'match_config\': asdict(boot.config)}\n\n\ndef _cache_key(desc):\n    return digest(_json(desc).encode())\n\n\n@contextmanager\ndef _lock(path, timeout=180):\n    """Kaggle/Linux interprocess lock; no global ML random-number consumption."""\n    import fcntl\n    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)\n    with p.open(\'a+b\') as f:\n        deadline = time.monotonic() + timeout\n        while True:\n            try:\n                fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)\n                break\n            except BlockingIOError:\n                if time.monotonic() >= deadline:\n                    raise TimeoutError(f\'Timed out waiting for cache builder: {p}\')\n                time.sleep(0.1)\n        try:\n            yield\n        finally:\n            fcntl.flock(f, fcntl.LOCK_UN)\n\n\ndef _read_definitions(boot):\n    definitions = {name: {\'address\': addr, \'type\': dtype}\n                   for name, (addr, dtype) in RAM_FIELDS.items()}\n    calibrated = boot.report.get(\'options_fix\', {})\n    addr = calibrated.get(\'corrected_address\')\n    if addr is not None:\n        definitions[\'difficulty_menu\'] = {\n            \'address\': int(addr, 0) if isinstance(addr, str) else int(addr),\n            \'type\': calibrated.get(\'dtype\', \'|u1\'),\n        }\n    return definitions\n\n\ndef _snapshot(boot, desc):\n    """Capture at exactly the existing start() return point, BEFORE jitter."""\n    if not boot.battle_started or boot.stage_write_active or boot.phase != \'battle\':\n        raise RuntimeError(\'Cache capture requires the verified post-initialization battle point.\')\n    if boot.trace and (boot.trace[-1][\'p1_buttons\'] or boot.trace[-1][\'p2_buttons\']):\n        raise RuntimeError(\'Cached starts must have released controller buttons.\')\n    if getattr(boot.engine, \'movie\', None):\n        raise RuntimeError(\'BK2 recording is not supported while validating a start cache.\')\n    state = bytes(boot.engine.em.get_state())\n    if not state or len(state) > MAX_PAYLOAD_BYTES:\n        raise RuntimeError(\'Invalid emulator snapshot size.\')\n    rgb = np.asarray(boot.rgb, dtype=np.uint8).copy()\n    ram = np.asarray(boot.engine.get_ram(), dtype=np.uint8).copy()\n    meta = {\n        \'descriptor\': desc,\n        \'capture_point\': \'existing ColdBootMatch.start return, before neutral offset\',\n        \'state_sha256\': digest(state), \'rgb_sha256\': array_digest(rgb),\n        \'ram_sha256\': array_digest(ram), \'readings\': dict(boot.values),\n        \'read_definitions\': _read_definitions(boot),\n        \'frame\': int(boot.frame), \'fps\': float(boot.fps), \'buttons\': list(boot.buttons),\n        \'source_output_dir\': str(boot.root),\n        \'source_bootstrap_status\': boot.report.get(\'status\'),\n        \'source_options_check\': boot.report.get(\'options_fix\'),\n        \'native_cpu_fresh_random_seed_claimed\': False,\n        \'rng_reset_scope\': \'emulator snapshot only; NOT Python/NumPy/Torch/model RNG\',\n    }\n    _validate_start(boot)\n    return {\'state\': state, \'rgb\': rgb, \'ram\': ram, \'meta\': meta}\n\n\ndef _validate_start(boot):\n    raw = boot.values\n    validate_character_ids(raw, boot.config.p1_character, boot.config.p2_character)\n    seconds = bcd_seconds(raw[\'timer_bcd\'])\n    if seconds is None or not 0 < seconds < 99:\n        raise RuntimeError(\'Cached start must be at the original running-countdown checkpoint.\')\n    if any(not 0 <= raw[f\'p{p}_hp\'] <= 176 for p in (1, 2)):\n        raise RuntimeError(\'Cached start contains invalid/KO health.\')\n    if raw[\'difficulty_menu\'] != boot.config.difficulty_stars - 1:\n        raise RuntimeError(\'Cached menu difficulty mismatch.\')\n    if boot.config.mode == \'pve\' and raw[\'difficulty_fight\'] != boot.config.difficulty_stars - 1:\n        raise RuntimeError(\'Cached CPU difficulty mismatch.\')\n    if any(raw[f\'p{p}_wins\'] != 0 for p in (1, 2)):\n        raise RuntimeError(\'Not the original first-round starting checkpoint.\')\n\n\ndef _restore_exact(boot, payload):\n    """Restore WITHOUT RetroEnv.reset\'s extra step and without resetting any ML RNG.\n\n    The first image is the actual RGB captured with the snapshot: framebuffer\n    contents are not necessarily serialized by every emulator core. Subsequent\n    images come from actual engine.step calls; deterministic replay is verified.\n    """\n    m = payload[\'meta\']; engine = boot.engine\n    if abs(float(boot.fps) - float(m[\'fps\'])) > 1e-6:\n        raise RuntimeError(\'Emulator frame rate differs from the cached start.\')\n    if list(boot.buttons) != m[\'buttons\']:\n        raise RuntimeError(\'Controller layout differs from cached layout.\')\n    if getattr(engine, \'movie\', None):\n        raise RuntimeError(\'Cannot restore while recording a BK2 movie.\')\n    engine.em.set_state(payload[\'state\'])\n    for p in range(engine.players):\n        engine.em.set_button_mask(np.zeros(engine.num_buttons, dtype=np.uint8), p)\n    engine.data.reset()\n    for name, spec in m[\'read_definitions\'].items():\n        engine.data.set_variable(\'boot_\' + name, dict(spec))  # Definitions only, not RAM values.\n    engine.data.update_ram()\n    got_ram = np.asarray(engine.get_ram(), dtype=np.uint8)\n    if not np.array_equal(got_ram, payload[\'ram\']):\n        raise RuntimeError(\'Full RAM readback differs after restoring match-start snapshot.\')\n    boot.values = boot.read()\n    if boot.values != m[\'readings\']:\n        raise RuntimeError(\'Cached character/position/HP/timer readback mismatch.\')\n    boot.rgb = payload[\'rgb\'].copy()\n    engine.img = boot.rgb.copy()\n    engine.ram = None\n    boot.frame = int(m[\'frame\'])\n    boot.phase = \'battle\'; boot.battle_started = True; boot.stage_write_active = False\n    boot.trace.clear()\n    _validate_start(boot)\n\n\ndef _probe_trace(boot, frames=96):\n    """No model involved. Exact same nonrandom controller sequence for replay QC."""\n    old_callback = boot.frame_callback; boot.frame_callback = None\n    h = hashlib.sha256()\n    phase1 = ((), (\'RIGHT\',), (\'DOWN\', \'B\'), (), (\'UP\', \'LEFT\'), (), (\'C\',), ())\n    phase2 = ((), (\'LEFT\',), (\'Y\',), (), (\'DOWN\',), (), (\'C\',), ())\n    try:\n        for i in range(frames):\n            slot = (i // 6) % len(phase1)\n            k2 = phase2[slot] if boot.config.mode == \'pvp\' else ()\n            raw = boot.frame_step(phase1[slot], k2)\n            h.update(_json(raw).encode())\n            h.update(np.ascontiguousarray(boot.engine.get_ram()).tobytes())\n            h.update(np.ascontiguousarray(boot.rgb).tobytes())\n    finally:\n        boot.frame_callback = old_callback\n    return h.hexdigest()\n\n\ndef _check_payload(p, desc):\n    m = p[\'meta\']\n    if m.get(\'descriptor\') != desc:\n        raise RuntimeError(\'Cache identity/configuration mismatch.\')\n    if digest(p[\'state\']) != m.get(\'state_sha256\'):\n        raise RuntimeError(\'State cache checksum mismatch.\')\n    if array_digest(p[\'rgb\']) != m.get(\'rgb_sha256\') or array_digest(p[\'ram\']) != m.get(\'ram_sha256\'):\n        raise RuntimeError(\'Cached observation/RAM checksum mismatch.\')\n    if not m.get(\'roundtrip_verified\') or not m.get(\'probe_sha256\'):\n        raise RuntimeError(\'Cache was not verified by the builder.\')\n\n\ndef _save(path, p):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temp = tempfile.mkstemp(prefix=path.stem + \'.\', suffix=\'.tmp\', dir=path.parent)\n    try:\n        with os.fdopen(fd, \'wb\') as f:\n            np.savez_compressed(f, state=np.frombuffer(p[\'state\'], dtype=np.uint8),\n                                rgb=p[\'rgb\'], ram=p[\'ram\'],\n                                metadata=np.frombuffer(_json(p[\'meta\']).encode(\'utf-8\'), dtype=np.uint8))\n            f.flush(); os.fsync(f.fileno())\n        os.replace(temp, path)\n    finally:\n        if os.path.exists(temp): os.unlink(temp)\n\n\ndef _load(path, desc):\n    if path.stat().st_size > MAX_PAYLOAD_BYTES:\n        raise RuntimeError(\'Cache file exceeds safety size limit.\')\n    with zipfile.ZipFile(path) as z:\n        if sum(x.file_size for x in z.infolist()) > MAX_PAYLOAD_BYTES:\n            raise RuntimeError(\'Uncompressed state cache exceeds safety size limit.\')\n    with np.load(path, allow_pickle=False) as data:\n        if set(data.files) != {\'state\', \'rgb\', \'ram\', \'metadata\'}:\n            raise RuntimeError(\'Unexpected cache archive fields.\')\n        if any(data[k].dtype != np.uint8 for k in data.files):\n            raise RuntimeError(\'Unexpected state-cache array type.\')\n        p = {\'state\': data[\'state\'].tobytes(), \'rgb\': data[\'rgb\'].copy(),\n             \'ram\': data[\'ram\'].copy(), \'meta\': json.loads(data[\'metadata\'].tobytes().decode())}\n    _check_payload(p, desc)\n    return p\n\n\nclass CachedColdBootMatch(ColdBootMatch):\n    """Same public start interface; only replaces redundant native initialization."""\n    def start(self):\n        if not enabled():\n            t = time.perf_counter()\n            result = super().start()\n            self.cache_info = {\'status\': \'DISABLED_COLD_BOOT\', \'seconds\': time.perf_counter() - t}\n            return result\n        t = time.perf_counter(); status = \'ERROR\'; path = None\n        try:\n            desc = descriptor(self); key = _cache_key(desc)\n            path = cache_root() / (key + \'.npz\')\n            cache_root().mkdir(parents=True, exist_ok=True)\n            # Cached starts use the same implementation even for independent\n            # evaluation processes. Every new emulator verifies its replay.\n            with _lock(path.with_suffix(\'.lock\')):\n                payload = _MEMO.get(str(path))\n                if payload is None and path.exists():\n                    payload = _load(path, desc)\n                if payload is None:\n                    super().start()\n                    payload = _snapshot(self, desc)\n                    reference = _probe_trace(self)\n                    _restore_exact(self, payload)\n                    replay = _probe_trace(self)\n                    if replay != reference:\n                        raise RuntimeError(\'Snapshot restoration changed the 96-frame reference trajectory. Cache not saved.\')\n                    _restore_exact(self, payload)\n                    payload[\'meta\'].update(roundtrip_verified=True, probe_frames=96,\n                                           probe_sha256=reference)\n                    _save(path, payload)\n                    status = \'BUILT_AND_REPLAY_VERIFIED\'\n                else:\n                    _check_payload(payload, desc)\n                    self._create()  # Creates emulator and definitions, but skips the menu route.\n                    _restore_exact(self, payload)\n                    replay = _probe_trace(self, payload[\'meta\'][\'probe_frames\'])\n                    if replay != payload[\'meta\'][\'probe_sha256\']:\n                        raise RuntimeError(\'Cached start does not replay identically in this emulator. Stop; disable cache to inspect.\')\n                    _restore_exact(self, payload)\n                    status = \'CACHE_HIT_REPLAY_VERIFIED\'\n                _MEMO[str(path)] = payload\n            self.cache_info = {\n                \'status\': status, \'key\': key, \'path\': str(path),\n                \'seconds\': time.perf_counter() - t,\n                \'initial_readings\': dict(self.values),\n                \'capture_point\': payload[\'meta\'][\'capture_point\'],\n                \'snapshot_rng_restored\': \'game state only, including any serialized native PRNG state\',\n                \'learner_rng_reseeded\': False, \'new_native_cpu_seed_claimed\': False,\n                \'first_image_source\': \'actual RGB captured at original start checkpoint\',\n                \'post_restore_replay_frames\': payload[\'meta\'][\'probe_frames\'],\n            }\n            # Do not claim that a hit navigated menus or did no state restore.\n            self.report.update(status=\'CACHED_START_CHECKS_PASSED\',\n                               external_state_files_used=True, in_memory_snapshots_used=True,\n                               automatic_match_start_cache=True, cache=self.cache_info,\n                               all_setup_writes_stopped=True,\n                               source_bootstrap=payload[\'meta\'][\'source_output_dir\'],\n                               note=\'Cache restores a game start, not a new independently seeded CPU.\')\n            _write_json(self.root / \'cache_reset.json\', self.cache_info)\n            print(\'[START CACHE]\', status, f\'{self.cache_info["seconds"]:.3f}s\', flush=True)\n            return self.rgb.copy(), dict(self.values)\n        except Exception as exc:\n            self.stage_write_active = False\n            self.report.update(status=\'ERROR\', error=repr(exc), cache_file=str(path),\n                               note=\'No silent fallback. Run the explicit environment check or inspect this cache before continuing.\')\n            _write_json(self.root / \'cache_error.json\', self.report)\n            raise\n        finally:\n            self.flush()\n', 'sf2_submission.py': '"""Select one immutable v3 agent and wrap it for the public practice arena.\n\nStatic inspection NEVER imports student code or deserializes policy tensors.\nA static pass is a packaging/protocol result, NOT a security or game-skill certificate.\n"""\nfrom __future__ import annotations\nimport ast\nimport datetime as dt\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport stat\nimport tempfile\nimport zipfile\n\nfrom sf2_actions import signature, validate_library\nfrom sf2_common import BUNDLE_SCHEMA, ENV_PROTOCOL, OBS_PROTOCOL, rules_signature\nfrom sf2_observation import _validate_config\n\nAGENT_FILES = frozenset({\'policy.zip\',\'model.py\',\'reward.py\',\'observation.py\',\'spec.json\',\n                         \'common.json\',\'training.json\',\'manifest.json\',\'checksums.json\'})\nSUBMISSION_FILES = frozenset({\'agent.zip\',\'submission.json\',\'static_check.json\',\'README.txt\'})\nMAX_AGENT = 512 * 1024**2\nMAX_OUTER = MAX_AGENT + 4 * 1024**2\nJSON_LIMIT = 2 * 1024**2\nSOURCE_LIMIT = 512 * 1024\nSUBMISSION_SCHEMA = \'sye6308-submission-v1\'\n\n\ndef digest(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef canonical(value) -> bytes:\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), ensure_ascii=False,\n                      allow_nan=False).encode(\'utf-8\')\n\n\ndef json_bytes(data: bytes):\n    def no_duplicates(pairs):\n        result = {}\n        for key, value in pairs:\n            if key in result:\n                raise ValueError(\'Duplicate JSON key: \'+str(key))\n            result[key] = value\n        return result\n    def no_constant(value):\n        raise ValueError(\'Non-finite JSON number: \'+value)\n    if len(data) > JSON_LIMIT:\n        raise ValueError(\'JSON metadata exceeds the supported size.\')\n    return json.loads(data.decode(\'utf-8\'), object_pairs_hook=no_duplicates, parse_constant=no_constant)\n\n\ndef _read_archive(data: bytes, allowed, total_limit: int, *, binary_names):\n    if not data or len(data) > total_limit:\n        raise ValueError(\'Archive is empty or exceeds the supported size.\')\n    with zipfile.ZipFile(io.BytesIO(data)) as z:\n        infos = z.infolist()\n        names = [x.filename for x in infos]\n        if len(names) != len(allowed) or set(names) != set(allowed):\n            raise ValueError(\'Unexpected/missing/duplicate archive files. Expected: \'+\', \'.join(sorted(allowed)))\n        if sum(x.file_size for x in infos) > total_limit:\n            raise ValueError(\'Uncompressed archive exceeds the supported size.\')\n        result = {}\n        for x in infos:\n            if (x.filename != PurePosixPath(x.filename).name or \'\\\\\' in x.filename\n                    or x.is_dir() or stat.S_ISLNK(x.external_attr >> 16) or x.flag_bits & 1):\n                raise ValueError(\'Unsafe or encrypted archive member: \'+x.filename)\n            limit = total_limit if x.filename in binary_names else SOURCE_LIMIT if x.filename.endswith(\'.py\') else JSON_LIMIT\n            if x.file_size > limit:\n                raise ValueError(\'Archive member exceeds limit: \'+x.filename)\n            result[x.filename] = z.read(x)  # CRC checked by zipfile; never extract arbitrary paths.\n        return result\n\n\ndef load_rules(path):\n    r = json_bytes(Path(path).read_bytes())\n    if r.get(\'schema\') != \'sye6308-practice-rules-v1\':\n        raise ValueError(\'Unsupported arena rules schema.\')\n    rules_signature(r[\'common\'])\n    if r[\'inference\'][\'device\'] != \'cpu\' or r[\'inference\'][\'torch_threads\'] != 1:\n        raise ValueError(\'This practice build uses CPU inference with one torch thread per actor.\')\n    for key in (\'load_timeout_seconds\', \'decision_timeout_seconds\'):\n        if not 0 < float(r[\'inference\'][key]) <= 600:\n            raise ValueError(\'Invalid timeout rule.\')\n    if r[\'inference\'][\'deterministic\'] is not True:\n        raise ValueError(\'This practice ruleset uses deterministic inference.\')\n    if not r[\'allowed_characters\']:\n        raise ValueError(\'The rules need an explicit character list.\')\n    return r\n\n\ndef rules_hash(rules):\n    return digest(canonical(rules))\n\n\ndef inspect_agent_bytes(data: bytes, rules: dict):\n    f = _read_archive(data, AGENT_FILES, MAX_AGENT, binary_names={\'policy.zip\'})\n    hashes = json_bytes(f[\'checksums.json\'])\n    if set(hashes) != AGENT_FILES-{\'checksums.json\'}:\n        raise ValueError(\'The agent checksum list is incomplete.\')\n    for name, expected in hashes.items():\n        if digest(f[name]) != expected:\n            raise ValueError(\'Agent checksum mismatch: \'+name)\n    for name in (\'model.py\',\'observation.py\',\'reward.py\'):\n        ast.parse(f[name].decode(\'utf-8\'), filename=name)  # Syntax only; do not execute.\n    spec, common, manifest = (json_bytes(f[n]) for n in (\'spec.json\',\'common.json\',\'manifest.json\'))\n    training = json_bytes(f[\'training.json\'])\n    if not isinstance(training, dict):\n        raise ValueError(\'training.json must be an object.\')\n    if (manifest.get(\'schema\') != BUNDLE_SCHEMA or manifest.get(\'env_protocol\') != ENV_PROTOCOL\n            or manifest.get(\'observation_protocol\') != OBS_PROTOCOL or manifest.get(\'algorithm\') != \'PPO\'):\n        raise ValueError(\'Only classroom v3 ordinary-PPO agents are supported; raw .pth/.zip weights are not a submission.\')\n    validate_library(spec[\'actions\'])\n    action_hash = signature(spec[\'actions\'])\n    if spec.get(\'action_hash\') != action_hash or manifest.get(\'action_hash\') != action_hash:\n        raise ValueError(\'Action table/hash mismatch.\')\n    if manifest.get(\'action_count\') != len(spec[\'actions\']):\n        raise ValueError(\'Action count mismatch.\')\n    if spec[\'character\'] != manifest.get(\'character\') or spec[\'character\'] not in rules[\'allowed_characters\']:\n        raise ValueError(\'Character is not allowed by these rules or differs from the manifest.\')\n    if spec[\'team_name\'] != manifest.get(\'team_name\'):\n        raise ValueError(\'Stored team name differs from the agent manifest.\')\n    if not isinstance(spec.get(\'use_position\'), bool):\n        raise ValueError(\'use_position must be a bool.\')\n    if rules_signature(common) != rules_signature(rules[\'common\']) or manifest.get(\'rules\') != rules_signature(common):\n        raise ValueError(\'ARENA_RULE_MISMATCH: game speed/decision interval/coordinate scale/frame cap must match the public arena.\')\n    obs = spec[\'observation\']\n    if set(obs) != {\'schema\',\'source_sha256\',\'config\',\'source\'} or obs[\'schema\'] != \'sf2-student-observation-v1\':\n        raise ValueError(\'Unsupported observation descriptor.\')\n    c = _validate_config(obs[\'config\'])\n    if obs[\'source\'].encode(\'utf-8\') != f[\'observation.py\'] or digest(f[\'observation.py\']) != obs[\'source_sha256\']:\n        raise ValueError(\'Observation source/hash mismatch.\')\n    shape = [4 * (1 if c[\'color\'] == \'gray\' else 3), c[\'height\'], c[\'width\']]\n    expected_obs = {\'protocol\':OBS_PROTOCOL, \'source_sha256\':obs[\'source_sha256\'],\n        \'config\':c, \'image_shape\':shape, \'dtype\':\'uint8\', \'normalization\':\'SB3 divides by 255 once\',\n        \'channel_order\':\'oldest to latest, then R/G/B within each RGB frame\',\n        \'decision_frames\':6, \'position_count\':6 if spec[\'use_position\'] else 0}\n    if manifest.get(\'observation\') != expected_obs:\n        raise ValueError(\'Observation metadata mismatch.\')\n    if manifest.get(\'input_keys\') != ([\'image\',\'position\'] if spec[\'use_position\'] else [\'image\']):\n        raise ValueError(\'Input key mismatch.\')\n    if manifest.get(\'image_mirrored\') is not False:\n        raise ValueError(\'Unexpected image mirroring protocol.\')\n    if digest(f[\'model.py\']) != manifest.get(\'model_source_sha256\'):\n        raise ValueError(\'Model source/hash mismatch.\')\n    # Inspect nested SB3 container sizes/names without unpickling or torch.load.\n    with zipfile.ZipFile(io.BytesIO(f[\'policy.zip\'])) as z:\n        infos = z.infolist()\n        names = [i.filename for i in infos]\n        if (\'data\' not in names or \'policy.pth\' not in names or len(names) > 32\n                or len(set(names)) != len(names) or sum(i.file_size for i in infos) > MAX_AGENT):\n            raise ValueError(\'Malformed or oversized inner SB3 policy archive.\')\n        for i in infos:\n            if i.filename != PurePosixPath(i.filename).name or \'\\\\\' in i.filename or i.flag_bits & 1 or stat.S_ISLNK(i.external_attr >> 16):\n                raise ValueError(\'Unsafe inner policy archive member.\')\n    return {\'status\':\'STATIC_PACKAGE_CHECK_PASSED\', \'agent_sha256\':digest(data),\n        \'agent_bytes\':len(data), \'rules_id\':rules[\'rules_id\'], \'rules_sha256\':rules_hash(rules),\n        \'team_name_in_model\':spec[\'team_name\'], \'character\':spec[\'character\'],\n        \'algorithm\':\'PPO\', \'image_shape\':shape, \'input_keys\':manifest[\'input_keys\'],\n        \'action_count\':len(spec[\'actions\']), \'action_hash\':action_hash,\n        \'runtime_versions_at_training\':manifest.get(\'runtime_versions\',{}),\n        \'student_code_executed\':False, \'policy_loaded\':False,\n        \'scope\':\'Container integrity, declared contracts and syntax only. Not runtime/skill/security certification.\',\n        \'spec\':spec, \'common\':common, \'manifest\':manifest}\n\n\ndef inspect_agent(path, rules):\n    p = Path(path).expanduser().resolve()\n    if not p.is_file():\n        raise FileNotFoundError(p)\n    if p.stat().st_size > MAX_AGENT:\n        raise ValueError(\'Agent package too large.\')\n    return inspect_agent_bytes(p.read_bytes(), rules)\n\n\ndef _public_check(info):\n    return {k:v for k,v in info.items() if k not in (\'spec\',\'common\',\'manifest\')}\n\n\ndef select_agent(root, source=\'pve\', checkpoint=\'best\', bundle=\'\'):\n    if source == \'path\':\n        if not bundle:\n            raise ValueError(\'For source="path", fill the exact saved agent.zip path.\')\n        return Path(bundle).expanduser().resolve(), {\'source\':\'path\',\'checkpoint\':\'explicit\'}\n    if source not in (\'pve\',\'pvp\') or checkpoint not in (\'best\',\'last\'):\n        raise ValueError(\'source must be pve/pvp/path; checkpoint must be best/last.\')\n    if bundle:\n        raise ValueError(\'Use source="path" for an explicit path, or leave bundle empty.\')\n    p = Path(root)/f\'last_training_{source}.json\'\n    if not p.is_file():\n        raise FileNotFoundError(\'No completed \'+source+\' run here. Choose source="path" for an existing package.\')\n    report = json_bytes(p.read_bytes()) if p.stat().st_size<=JSON_LIMIT else json.loads(p.read_text())\n    selected = report.get(checkpoint+\'_agent\')\n    if not selected:\n        raise ValueError(\'No \'+checkpoint+\' package in the selected run. Choose another checkpoint explicitly; no silent fallback.\')\n    return Path(selected).resolve(), {\'source\':source,\'checkpoint\':checkpoint}\n\n\ndef prepare_submission(root, *, rules_path, team_id, display_name=\'\', source=\'pve\',\n                       checkpoint=\'best\', bundle=\'\', notes=\'\'):\n    """One click, no model execution: package ONLY the selected saved snapshot."""\n    if not isinstance(team_id,str) or not re.fullmatch(r\'[A-Za-z0-9][A-Za-z0-9_-]{0,39}\',team_id):\n        raise ValueError(\'Use a short team ID such as Group_05 (letters, digits, _ or -).\')\n    if not isinstance(display_name,str) or len(display_name)>80 or not isinstance(notes,str) or len(notes)>2000:\n        raise ValueError(\'Display name/notes too long.\')\n    rules = load_rules(rules_path)\n    selected, selection = select_agent(root,source,checkpoint,bundle)\n    if not selected.is_file():\n        raise FileNotFoundError(selected)\n    if selected.stat().st_size>MAX_AGENT:\n        raise ValueError(\'Agent package too large.\')\n    data = selected.read_bytes()  # One immutable byte snapshot. No reading current student/ files.\n    info = inspect_agent_bytes(data, rules)\n    stamp = dt.datetime.now(dt.timezone.utc).strftime(\'%Y%m%dT%H%M%S_%fZ\')\n    metadata = {\'schema\':SUBMISSION_SCHEMA, \'created_utc\':stamp, \'team_id\':team_id,\n        \'display_name\':display_name or team_id, \'notes\':notes,\n        \'agent_file\':\'agent.zip\', \'agent_sha256\':info[\'agent_sha256\'], \'character\':info[\'character\'],\n        \'algorithm\':\'PPO\', \'selection\':selection, \'rules_id\':rules[\'rules_id\'],\n        \'rules_sha256\':rules_hash(rules),\n        \'notebook_sources_used\':\'Exact selected agent snapshot; mutable current student files are NOT repackaged.\',\n        \'required_runtime_versions\':info[\'runtime_versions_at_training\'],\n        \'runtime_self_check\':\'NOT_RUN; run this exact submission in the public Arena notebook.\',\n        \'security\':\'Contains executable custom Python and serialized PyTorch/SB3 objects. Review before loading.\'}\n    folder = Path(root).resolve()/\'submissions\';folder.mkdir(parents=True,exist_ok=True)\n    target = folder/f\'{team_id}_{stamp}_submission.zip\'\n    readme = (\'SYE 6308 practice submission\\n\'\n        \'This ZIP contains one selected agent, not both best and last.\\n\'\n        \'Use the public Arena notebook to inspect and test the exact ZIP in a clean session.\\n\'\n        \'Do not run the training notebook to make this ZIP work.\\n\'\n        \'Do not replace agent.zip after self-check; the receipt binds its exact checksum.\\n\'\n        \'Technical report/course deliverables are separate. No extra ROM, .state cache, credential store or videos are collected; review your selected source for secrets.\\n\'\n        \'Static packaging is not model execution, a sandbox, or an official competition score.\\n\')\n    payload={\'agent.zip\':data,\'submission.json\':canonical(metadata),\n             \'static_check.json\':canonical(_public_check(info)),\'README.txt\':readme.encode()}\n    with tempfile.NamedTemporaryFile(dir=folder,suffix=\'.building\',delete=False) as f:\n        temp=Path(f.name)\n    try:\n        with zipfile.ZipFile(temp,\'w\',zipfile.ZIP_DEFLATED) as z:\n            for name,content in payload.items():\n                z.writestr(name,content)\n        temp.replace(target)\n    finally:\n        temp.unlink(missing_ok=True)\n    sha=digest(target.read_bytes())\n    target.with_suffix(\'.sha256\').write_text(f\'{sha}  {target.name}\\n\',encoding=\'utf-8\')\n    print(\'SUBMISSION_PACKAGED (static checks only; arena self-check still required)\')\n    print(\'Selected model:\',selected)\n    print(\'Team:\',team_id,\'| character:\',info[\'character\'],\'| actions:\',info[\'action_count\'],\'| input:\',info[\'image_shape\'])\n    print(\'Submission:\',target);print(\'Submission SHA256:\',sha)\n    return target\n\n\ndef inspect_submission(path, rules, extract_agent_to=None):\n    p=Path(path).expanduser().resolve()\n    if not p.is_file():raise FileNotFoundError(p)\n    if p.stat().st_size>MAX_OUTER:raise ValueError(\'Submission ZIP too large.\')\n    data=p.read_bytes()\n    f=_read_archive(data,SUBMISSION_FILES,MAX_OUTER,binary_names={\'agent.zip\'})\n    m=json_bytes(f[\'submission.json\'])\n    if (m.get(\'schema\')!=SUBMISSION_SCHEMA or m.get(\'agent_file\')!=\'agent.zip\'\n            or m.get(\'rules_id\')!=rules[\'rules_id\'] or m.get(\'rules_sha256\')!=rules_hash(rules)):\n        raise ValueError(\'Submission schema/rules differ from the public arena release.\')\n    if not isinstance(m.get(\'team_id\'),str) or not re.fullmatch(r\'[A-Za-z0-9][A-Za-z0-9_-]{0,39}\',m[\'team_id\']):\n        raise ValueError(\'Invalid team ID.\')\n    info=inspect_agent_bytes(f[\'agent.zip\'],rules)\n    if m.get(\'agent_sha256\')!=info[\'agent_sha256\'] or m.get(\'character\')!=info[\'character\']:\n        raise ValueError(\'Selected model does not match submission metadata.\')\n    if json_bytes(f[\'static_check.json\'])!=_public_check(info):\n        raise ValueError(\'Stored static check does not match the selected model; rebuild submission.\')\n    result={\'status\':\'STATIC_SUBMISSION_CHECK_PASSED\',\'submission_sha256\':digest(data),\n            \'submission_bytes\':len(data),\'metadata\':m,**info}\n    result[\'status\']=\'STATIC_SUBMISSION_CHECK_PASSED\'\n    if extract_agent_to is not None:\n        target=Path(extract_agent_to);target.parent.mkdir(parents=True,exist_ok=True)\n        target.write_bytes(f[\'agent.zip\'])\n        result[\'agent_path\']=str(target.resolve())\n    return result\n\n\ndef candidate_models(root, limit=40):\n    root=Path(root)\n    candidates=[]\n    for p in sorted(root.glob(\'runs/*/*/*_agent.zip\'),key=lambda p:p.stat().st_mtime,reverse=True):\n        if p.name in (\'best_agent.zip\',\'last_agent.zip\'):\n            candidates.append(p.resolve())\n    return candidates[:limit]\n\n\ndef submission_widget(root, rules_path, on_packaged=None):\n    """Optional one-click UI; the plain configuration cell is also provided."""\n    import ipywidgets as w\n    from IPython.display import display,FileLink\n    if on_packaged is not None and not callable(on_packaged):\n        raise TypeError("on_packaged must be callable or None.")\n    candidates=candidate_models(root)\n    labels=[(str(p.relative_to(Path(root).resolve())),str(p)) for p in candidates]\n    select=w.Dropdown(options=[(\'Choose a saved model...\', \'\')]+labels,description=\'Model:\',layout=w.Layout(width=\'95%\'))\n    explicit=w.Text(value=\'\',description=\'Or path:\',placeholder=\'Exact v3 agent ZIP path\',layout=w.Layout(width=\'95%\'))\n    team=w.Text(value=\'Group_01\',description=\'Team ID:\')\n    name=w.Text(value=\'\',description=\'Team name:\')\n    button=w.Button(description=\'Package selected model\',button_style=\'primary\',layout=w.Layout(width=\'220px\'))\n    output=w.Output()\n    def on_click(_):\n        output.clear_output();button.disabled=True\n        with output:\n            try:\n                chosen=explicit.value.strip() or select.value\n                if not chosen:raise ValueError(\'Select a model or enter its exact saved package path.\')\n                p=prepare_submission(root,rules_path=rules_path,team_id=team.value.strip(),\n                    display_name=name.value.strip(),source=\'path\',bundle=chosen)\n                if on_packaged is not None:\n                    on_packaged(p)\n                display(FileLink(str(p)));display(FileLink(str(p.with_suffix(\'.sha256\'))))\n            except Exception as exc:\n                print(type(exc).__name__+\': \'+str(exc))\n            finally:button.disabled=False\n    button.on_click(on_click)\n    display(w.VBox([select,explicit,team,name,button,output]))\n', 'sf2_video.py': '"""Videos show real emulator frames; synthetic test images are labelled in tests only."""\nfrom pathlib import Path\nimport shutil,subprocess\nimport cv2,numpy as np\n\nclass Video:\n    def __init__(self,path,fps):\n        self.path=Path(path);self.path.parent.mkdir(parents=True,exist_ok=True)\n        self.fps=float(fps);self.frames=0;self.writer=None\n        if not np.isfinite(self.fps) or self.fps<=0:raise ValueError(\'Invalid video FPS\')\n    def write(self,rgb,lines=(),keys1=(),keys2=()):\n        canvas=np.zeros((660,800,3),np.uint8)\n        canvas[:560]=cv2.resize(cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR),(800,560),interpolation=cv2.INTER_NEAREST)\n        for i,line in enumerate(lines[:3]):\n            text=str(line)[:115]\n            cv2.putText(canvas,text,(8,578+20*i),cv2.FONT_HERSHEY_SIMPLEX,.43,(245,245,245),1,cv2.LINE_AA)\n        for base,port,active in ((8,1,keys1),(405,2,keys2)):\n            cv2.putText(canvas,f\'P{port}\',(base,647),cv2.FONT_HERSHEY_SIMPLEX,.42,(230,230,230),1)\n            for j,k in enumerate((\'LEFT\',\'RIGHT\',\'UP\',\'DOWN\',\'X\',\'Y\',\'Z\',\'A\',\'B\',\'C\')):\n                x=base+27+j*36;on=k in active\n                cv2.rectangle(canvas,(x,630),(x+33,654),(215,215,215) if on else (40,40,40),-1)\n                cv2.putText(canvas,k,(x+2,646),cv2.FONT_HERSHEY_SIMPLEX,.29,(0,0,0) if on else (220,220,220),1)\n        if self.writer is None:\n            self.writer=cv2.VideoWriter(str(self.path),cv2.VideoWriter_fourcc(*\'mp4v\'),self.fps,(800,660))\n            if not self.writer.isOpened():raise RuntimeError(\'Video writer could not open\')\n        self.writer.write(canvas);self.frames+=1\n    def close(self):\n        if self.writer is not None:self.writer.release();self.writer=None\n    def finish(self):\n        self.close()\n        if self.frames==0:return None\n        exe=shutil.which(\'ffmpeg\')\n        if not exe:return str(self.path)\n        dst=self.path.with_name(self.path.stem+\'_playable.mp4\')\n        p=subprocess.run([exe,\'-y\',\'-loglevel\',\'error\',\'-i\',str(self.path),\'-c:v\',\'libx264\',\'-preset\',\'veryfast\',\'-crf\',\'23\',\'-pix_fmt\',\'yuv420p\',\'-movflags\',\'+faststart\',str(dst)],capture_output=True,timeout=240)\n        if p.returncode==0:return str(dst)\n        return str(self.path)\n'}

# Do not rewrite code under a running task.
if Path('/proc').exists():
    for process in Path('/proc').iterdir():
        if process.name.isdigit():
            try:
                args = (process / 'cmdline').read_bytes().split(b'\0')
            except (OSError, PermissionError):
                continue
            if any(arg.endswith((b'/run_lab.py', b'/sf2_arena.py', b'/sf2_arena_actor.py')) for arg in args):
                raise RuntimeError('Finish/stop the active SF2 task before running Setup.')

changed = [name for name, text in CORE_FILES.items()
           if (CODE_DIR/name).is_file() and (CODE_DIR/name).read_text() != text]
if changed and not REPLACE_CHANGED_PUBLIC_FILES:
    raise RuntimeError('Public files have local edits: '+str(changed)+
                       '. Back them up; only then explicitly enable replacement.')
if changed:
    import datetime, shutil
    backup = ROOT/'source_backups'/datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    backup.mkdir(parents=True)
    for name in changed:
        shutil.copy2(CODE_DIR/name, backup/name)
    print('Previous edited public files backed up to:', backup)
for name, text in CORE_FILES.items():
    compile(text, name, 'exec')
    (CODE_DIR/name).write_text(text, encoding='utf-8')
(ROOT/'release_sources.json').write_text(json.dumps({
    'release':'classroom-rc3-submission-v1',
    'files':{n:hashlib.sha256(t.encode()).hexdigest() for n,t in CORE_FILES.items()}
}, indent=2))
print('Public files ready:', len(CORE_FILES))

ARENA_RULES_TEXT = '{\n  "schema": "sye6308-practice-rules-v1",\n  "rules_id": "sye6308-practice-v1",\n  "status": "DRAFT_FOR_TA_ACCEPTANCE_NOT_OFFICIAL_COURSE_RULES",\n  "common": {"ruleset": "hyper", "speed_stars": 2, "decision_frames": 6, "position_scale": 1024.0, "max_round_frames": 7200},\n  "game": "StreetFighterIISpecialChampionEdition-Genesis",\n  "rom_sha1": "a5aad1d108046d9388e33247610dafb4c6516e0b",\n  "allowed_characters": ["Ryu", "EHonda", "Blanka", "Guile", "Ken", "ChunLi", "Zangief", "Dhalsim", "MBison", "Sagat", "Balrog", "Vega"],\n  "inference": {"device": "cpu", "torch_threads": 1, "deterministic": true, "load_timeout_seconds": 120, "decision_timeout_seconds": 2.0},\n  "selfcheck_offsets": [0, 12],\n  "selfcheck_ports": [1, 2],\n  "technical_error_handling": "Abort and retain diagnostics. A timeout/error is not a scored defeat in this practice build.",\n  "clock": "Both policies observe the same pre-decision state. Game advances only after both return. Action frame plans are exactly six native frames.",\n  "decision_timeout_note": "2 seconds is a generous engineering watchdog, NOT the previous course one-frame deadline or an approved competition latency limit.",\n  "score": "Report KO/native-timeout wins/losses/draws. A safety frame cap is a separate cutoff. Never use a student\'s reward as the match score.",\n  "security": "Separate processes and checksums are NOT a sandbox. Review custom Python/models. Formal untrusted competition needs OS/container/VM isolation, no network and no secrets.",\n  "course_approval": "The supplied old guide restricts Ryu and states a one-frame decision deadline and .pth/act(state) submission. Flexible characters, this six-frame protocol and the new ZIP format require instructor approval."\n}\n'
(ROOT/'arena_rules_v1.json').write_text(ARENA_RULES_TEXT, encoding='utf-8')
print('Public practice rules ready. This draft is not an automatic change to the course rules.')


In [ ]:
args = [sys.executable, str(CODE_DIR/'setup_sf2.py'),
        '--directory', str(ROOT/'resources'), '--sb3']
if ROM_FILE:
    args += ['--rom-file', ROM_FILE]
if ALLOW_ORIGINAL_ROM_SOURCE:
    args += ['--legacy-fetch']
subprocess.run(args, check=True)

# Use this notebook's launcher even when an older SF2 module was imported before.
for name in CORE_FILES:
    sys.modules.pop(name.removesuffix('.py'), None)
if str(CODE_DIR) in sys.path:
    sys.path.remove(str(CODE_DIR))
sys.path.insert(0, str(CODE_DIR))
from lab_notebook import run_task, run_pvp_task, show_result, last_training
from lab_review import show_training_review, make_handoff
from sf2_release import read_package_metadata, resolve_pvp_setup
os.chdir(ROOT)

subprocess.run([sys.executable, '-c',
    "import torch; from sf2_common import runtime_versions; "
    "print('Versions:', runtime_versions()); "
    "print('CUDA available:', torch.cuda.is_available()); "
    "print('Training GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU fallback')"],
    cwd=CODE_DIR, check=True)
print('Setup finished. PvE: Sections 2-7. PvP: Section 9; an existing v3 package can be used without retraining PvE.')


## 2. PvE settings — agent versus the built-in CPU
**Run this cell before Sections 3–6.** For the first experiment, keep the training defaults and change only your team name if needed.

| Setting | Meaning |
|---|---|
| `PVE_AGENT_CONFIG['character']` | Your agent's character; the template uses ChunLi. Follow the instructor-approved character rule. |
| `opponent_characters` / `difficulty_stars` | CPU matchups and difficulty, from 1 to 8 stars. The default checks ChunLi at 1 star. |
| `neutral_jitter_max` | Up to this many neutral game frames before the first observation; it is not a new independent CPU seed. |
| `total_timesteps` | Total agent transitions across all workers, not episodes or individual game frames. |
| `n_envs` / `n_steps` | Parallel games / transitions per game collected for each PPO update. Default: 8 × 512 = 4,096. |
| `batch_size` / `n_epochs` | Minibatch size / passes over each rollout during PPO training. |
| `learning_rate` / `gamma` | Update size / discount applied to future rewards. |
| `evaluate_every` / `checkpoint_every` | Requested validation/save intervals, applied at rollout boundaries. |

The 100,000-step request completes 25 whole rollouts: **102,400 transitions**. With these defaults, a 50,000-step periodic checkpoint is first due at **53,248**, not exactly 50,000.
More workers are not guaranteed to be faster. Keep the defaults for a baseline, then compare measured runtime before changing them.

**Character IDs:** `Ryu`, `EHonda`, `Blanka`, `Guile`, `Ken`, `ChunLi`,
`Zangief`, `Dhalsim`, `MBison`, `Sagat`, `Balrog`, `Vega`.
Use the exact spellings shown in the comments. This is the code/practice roster,
not proof that every matchup has been tested or permission to ignore course restrictions.

`PVE_AGENT_CONFIG['character']` selects your learner; `PVE_CONFIG['opponent_characters']` selects native CPU opponents.
Targeted CPU selection remains experimental: rerun Section 4 and inspect the actual characters before training a new matchup.


In [ ]:
# AVAILABLE CHARACTER IDs — use these exact spellings (no spaces or punctuation):
# Ryu, EHonda, Blanka, Guile, Ken, ChunLi,
# Zangief, Dhalsim, MBison, Sagat, Balrog, Vega.
# These are the implemented/public-practice IDs, not a claim that all matchups
# have passed real-game tests. Follow any stricter rules set by your instructor.
# PvE: select your learner below, then choose the native CPU opponent list.
# After changing character/opponents/difficulty, rerun Section 4 before training.
PVE_AGENT_CONFIG = {
    'team_name': 'group_demo',
    'character': 'ChunLi',  # YOUR learner's character; choose one ID from the list above.
    'seed': 42,
    'use_position': True,
}
PVE_CONFIG = {
    'opponent_characters': ['ChunLi'],  # Native CPU opponents, e.g. ['Ryu', 'Ken'].
    'difficulty_stars': [1],  # Native CPU difficulty: integer values from 1 to 8.
    'neutral_jitter_max': 60,
}
PVE_TRAIN_CONFIG = {
    'total_timesteps': 100_000,
    'n_envs': 8,
    'n_steps': 512,       # PER environment. 8*512 = 4096 per rollout.
    'batch_size': 512,
    'n_epochs': 4,
    'learning_rate': 0.00025,
    'gamma': 0.94,
    'gae_lambda': 0.95,
    'clip_range': 0.15,
    'ent_coef': 0.0,
    'vf_coef': 0.5,
    'target_kl': None,
    'device': 'auto',    # CUDA first. The task prints actual parameter devices.
    'torch_threads': 2,
    'checkpoint_every': 50_000,
    'evaluate_every': 20_480,
    'evaluate_final': True,
    'resume_from': None, # Exact optimiser resume is not implemented; see Section 5.
}
PVE_VALIDATION_OFFSETS = [0, 12, 24, 36, 48]
print('Agent:', PVE_AGENT_CONFIG)
print('CPU schedule:', PVE_CONFIG)
print('Arena rules:', COMMON_RULES)
print('Training:', PVE_TRAIN_CONFIG)


## 3. PvE student files — run all four code cells
These cells write only to `student/pve/`. They do not edit the independent PvP files.

**Important:** `%%writefile` saves the cell as a Python file; it does not start training. Edit the cell, then rerun it to save your changes. Rerunning an unchanged template overwrites the corresponding file on disk. Keep your edits in the notebook, not only in a separate file editor.

For the first run, leave all four templates unchanged. After editing one, rerun that cell and Section 4 before training again.

### 3.1 Observation — what the agent sees
The raw buffer has shape `(224, 320, 3)`. The default keeps rows `0:224` and columns `0:256`, checks that the excluded right strip is effectively black, converts to grayscale, and resizes to **height 168 × width 192**.
The agent receives four frames as `(4, 168, 192)` and, when enabled, six self-first position values.
The history updates once per decision, normally every six game frames—not on four consecutive game frames. At reset, all four entries copy the initial frame.

**Do not bypass `CROP_GUARD_FAILED` without inspecting the actual game frame.** The guard prevents silently cropping away visible content.


In [ ]:
%%writefile $PVE_STUDENT_DIR/student_observation.py
"""Student-editable image input. Default: four 192 x 168 grayscale frames.

Only transform the supplied RGB image. Never operate the game or change rewards.
The coordinate branch still uses the six original self-first position values.
SB3 divides these uint8 images by 255 once, inside its policy preprocessor.
"""
import cv2
import numpy as np

INPUT_CONFIG = {'color': 'gray', 'width': 192, 'height': 168, 'stack_frames': 4, 'raw_shape': [224, 320, 3], 'crop_xyxy': [0, 0, 256, 224], 'verify_right_padding': True, 'black_threshold': 5, 'max_nonblack_fraction': 0.005}


def preprocess(rgb):
    """One RGB uint8 frame -> (channels, height, width) uint8 array."""
    a = np.asarray(rgb)
    c = INPUT_CONFIG
    if a.dtype != np.uint8 or tuple(a.shape) != tuple(c["raw_shape"]):
        raise ValueError(f"Unexpected raw image: {a.shape} {a.dtype}; expected {c['raw_shape']} uint8.")
    x0, y0, x1, y1 = c["crop_xyxy"]
    if c["verify_right_padding"] and x1 < a.shape[1]:
        excluded = a[:, x1:, :]
        fraction = float(np.mean(np.max(excluded, axis=2) > c["black_threshold"]))
        if fraction > c["max_nonblack_fraction"]:
            raise ValueError(
                f"CROP_GUARD_FAILED: right-side nonblack fraction={fraction:.6f}. "
                "The fixed viewport is not valid for this observed frame. "
                "Stop and inspect the raw image; do not train with a silently changing crop."
            )
    view = a[y0:y1, x0:x1, :]
    if c["color"] == "gray":
        gray = cv2.cvtColor(view, cv2.COLOR_RGB2GRAY)
        result = cv2.resize(gray, (c["width"], c["height"]), interpolation=cv2.INTER_AREA)[None, :, :]
    else:
        result = cv2.resize(view, (c["width"], c["height"]), interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
    return np.ascontiguousarray(result, dtype=np.uint8)


### 3.2 Actions — what one action ID means
The default table has **15 actions**: NO_OP, eight directions, and A/B/C/X/Y/Z. The model selects one table entry per decision. Each entry specifies buttons for exactly **six game frames**.

Selecting C twice holds C continuously for twelve frames; it does not produce two press-and-release taps. Select an action without C to release it. The video button indicator shows the input sent, not whether an attack succeeded.

Leave `CUSTOM_ACTIONS=[]` for the first run. Optional added actions must also total six frames. Changing action meanings or the number of actions requires a compatible new learner; do not reinterpret an old model's output IDs.


In [ ]:
%%writefile $PVE_STUDENT_DIR/student_actions.py
"""Default: 15 basic actions, as in the old course without its two Ryu specials.
A basic key is held for six emulator frames; consecutive C means continuous C.
Use NO_OP (or another action) to release it. This is not automatic repeated tapping.
Students may append short combinations/sequences totaling exactly six frames.
No complete fighting strategy, auto-facing or long charge macro is provided.
"""
CUSTOM_ACTIONS = []

# Optional student exercise (OFF by default): a short, explicit press/release.
# CUSTOM_ACTIONS = [
#     {'name': 'tap_C', 'segments': [
#         {'keys': ['C'], 'frames': 3}, {'keys': [], 'frames': 3}]},
# ]


### 3.3 Reward — feedback for training
The default is:

`reward = 0.5 × (damage_dealt − damage_taken) / 176 + outcome_bonus`

The result bonus is +1 for a natural win, −1 for a natural loss, and 0 for a draw or a safety cutoff. Final-step damage is included. Both KO and the game's own time-up result use the win/loss bonus. A safety frame cap is different from a natural time-up and is not automatically a loss.

Edit `compute_reward` to test a reward design, but preserve finite numeric output. Changing this reward does not change the arena referee or its match winner. Training return is not validation win rate.


In [ ]:
%%writefile $PVE_STUDENT_DIR/student_reward.py
"""Simple classroom reward: small net-damage feedback + clear match outcome.

Compatible with SF2_Flexible_Classroom_PPO (ordinary PPO, six-frame steps).
The arena supplies damage_dealt/damage_taken with negative KO HP treated as 0.
A natural timeout uses the arena's win/loss/draw result, just like a KO.
A safety cutoff is not a loss. No distance, survival, action or time bonus.

With <=176 HP per fighter and no in-round healing, the sum of damage rewards
is bounded by [-0.5, 0.5]. A normal completed win therefore has positive raw
episode return, and a loss negative raw episode return. This is NOT a claim
about discounted returns, win rate, or empirically verified learning success.
"""
import math

REWARD_VERSION = "net_damage_outcome_v1"
FULL_HP = 176.0
DAMAGE_WEIGHT = 0.5
OUTCOME_BONUS = {
    "ongoing": 0.0,
    "win": 1.0,
    "loss": -1.0,
    "draw": 0.0,
    "cutoff": 0.0,
}


def compute_reward(t: dict) -> float:
    dealt = float(t["damage_dealt"])
    taken = float(t["damage_taken"])
    result = t["result"]

    # Do not silently replace invalid readings with a plausible reward.
    if not (math.isfinite(dealt) and math.isfinite(taken)):
        raise ValueError("Damage must be finite.")
    if not (0.0 <= dealt <= FULL_HP and 0.0 <= taken <= FULL_HP):
        raise ValueError("Damage is outside the expected 0..176 range.")
    if result not in OUTCOME_BONUS:
        raise ValueError(f"Unknown match result: {result!r}")

    # Count this step's damage even on the final step, then add outcome once.
    damage_reward = DAMAGE_WEIGHT * (dealt - taken) / FULL_HP
    return float(damage_reward + OUTCOME_BONUS[result])


### 3.4 Model — how observations become decisions
The default uses ordinary **PPO**. Three convolution layers produce 512 image features. With `use_position=True`, a separate 6 → 64 branch processes positions; the combined features feed the policy and value outputs. With `use_position=False`, the builder selects the standard image-only `CnnPolicy`.

For your first run, do not edit this cell. Later, change one model choice at a time and retrain a compatible learner. SB3 normalizes the uint8 image by 255; do not divide it a second time in this feature extractor.

**Existing-model compatibility:** the loader checks the exact observation and model source, including comments and whitespace, when initializing a new learning phase from a package. Put explanatory notes in Markdown instead of changing those source files when you need to keep using existing weights. Saved-model evaluation uses the source stored inside the agent package.


In [ ]:
%%writefile $PVE_STUDENT_DIR/student_model.py
"""Ordinary SB3 PPO + Nature-style CNN + a minimal optional position branch.
No MaskablePPO, control vector, frame-difference images or hand-crafted geometry.
With use_position=False, use the old standard CnnPolicy (NatureCNN) directly.
"""
import torch
from torch import nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

MODEL_CONFIG = {'image_features':512, 'position_features':64}

class ImagePositionExtractor(BaseFeaturesExtractor):
    def __init__(self,observation_space,image_features=512,position_features=64):
        super().__init__(observation_space,features_dim=image_features+position_features)
        # The three convolution sizes/strides match SB3 NatureCNN.
        cnn=nn.Sequential(
            nn.Conv2d(observation_space['image'].shape[0],32,8,4),nn.ReLU(),
            nn.Conv2d(32,64,4,2),nn.ReLU(),
            nn.Conv2d(64,64,3,1),nn.ReLU(),nn.Flatten())
        with torch.no_grad():
            flattened=cnn(torch.zeros(1,*observation_space['image'].shape)).shape[1]
        self.image_net=nn.Sequential(cnn,nn.Linear(flattened,image_features),nn.ReLU())
        self.position_net=nn.Sequential(nn.Linear(6,position_features),nn.ReLU())
    def forward(self,obs):
        # SB3 normalizes the uint8 image once; do not divide by 255 again here.
        return torch.cat((self.image_net(obs['image']),self.position_net(obs['position'])),dim=1)

def build_model(env,config,seed,device='auto'):
    has_position=hasattr(env.observation_space,'spaces')
    policy='MultiInputPolicy' if has_position else 'CnnPolicy'
    kwargs={}
    if has_position:
        kwargs={'features_extractor_class':ImagePositionExtractor,
                'features_extractor_kwargs':dict(MODEL_CONFIG),
                'net_arch':dict(pi=[],vf=[])}
    return PPO(policy,env,
        learning_rate=float(config['learning_rate']),n_steps=int(config['n_steps']),
        batch_size=int(config['batch_size']),n_epochs=int(config['n_epochs']),
        gamma=float(config['gamma']),gae_lambda=float(config.get('gae_lambda',0.95)),
        clip_range=float(config['clip_range']),ent_coef=float(config.get('ent_coef',0.0)),
        vf_coef=float(config.get('vf_coef',0.5)),max_grad_norm=0.5,
        target_kl=config.get('target_kl'),seed=int(seed),device=device,verbose=1,
        policy_kwargs=kwargs)


## 4. PvE environment check — required before training
Leave `RUN_ENVIRONMENT_CHECK=True` and run this cell after Sections 1–3.

**Inspect:** the raw reset frame, the four actual model-input frames, the stepped history, and the preview video. The default image tensor is `(4, 168, 192)`; four equal reset frames are expected. A successful check prints `ENVIRONMENT_CHECK_PASSED`.

Do not start Section 5 after an error. After edits to PvE source, character, CPU schedule or shared code/rules, rerun the check. It tests the **first configured matchup**, not every character/difficulty combination and not the strength of a trained model. PvP has its own independent check in Section 9.3.


In [ ]:
RUN_ENVIRONMENT_CHECK = True
if RUN_ENVIRONMENT_CHECK:
    LAST_CHECK, CHECK_REPORT = run_task(ROOT, 'check', PVE_AGENT_CONFIG, COMMON_RULES,
                                      PVE_TRAIN_CONFIG, PVE_CONFIG)
    show_result(LAST_CHECK, CHECK_REPORT)
else:
    print('Environment check skipped. Check the current sources/settings before training.')


### 4.1 Optional button test — no model learning
Skip this on the first training run. Enable `RUN_ACTION_CHECK` to compare holding C with a press-and-release pattern.

`C_hold` holds the key. `C_tap` presses for six frames, releases for 24, then repeats. The CPU still plays normally. These videos test controller input, not a trained policy or a move-success detector.


In [ ]:
RUN_ACTION_CHECK = False
if RUN_ACTION_CHECK:
    LAST_ACTION_CHECK, ACTION_REPORT = run_task(ROOT, 'actions', PVE_AGENT_CONFIG, COMMON_RULES,
        PVE_TRAIN_CONFIG, PVE_CONFIG, actions={'names': ['C_hold', 'C_tap', 'NO_OP']})
    show_result(LAST_ACTION_CHECK, ACTION_REPORT)
else:
    print('Optional action check is OFF.')


## 5. PvE training — this cell updates the learner
**Before running:** Section 4 must have passed for the current files and settings. Set `RUN_PVE_TRAINING=True`; leave `START_FROM_BUNDLE=''` for a new randomly initialized model.

**During training:** game workers collect transitions on CPU; the learner uses the selected device. Periodic validation is a separate no-learning task. Read the printed output directory and actual device rather than assuming every process uses the GPU.

**On success:** expect `TRAINING_COMPLETE`, a `best_agent.zip` path selected by CPU validation, and a `last_agent.zip` path for the final model. They need not contain identical weights. Results are under `runs/pve/`.

A supplied matching package initializes a **new phase with a new optimizer**, not an exact interrupted-run resume. Do not enable the trust switch for an unreviewed package.

**Stopping early:** cleanup may save `interrupted_policy.zip`, but this is a raw SB3 checkpoint, not a complete classroom agent. Do not pass it to Sections 6–9. Preserve any already completed `checkpoint_*_agent.zip` or `evaluation_*_agent.zip` files and the run folder. The automatic “latest run” selector tracks completed runs only and can still point to an older successful run after an interruption.


In [ ]:
RUN_PVE_TRAINING = False
START_FROM_BUNDLE = ''
TRUST_START_PACKAGE = False

if RUN_PVE_TRAINING:
    LAST_TRAINING, TRAIN_REPORT = run_task(ROOT, 'train', PVE_AGENT_CONFIG, COMMON_RULES,
        PVE_TRAIN_CONFIG, PVE_CONFIG, validation_offsets=PVE_VALIDATION_OFFSETS,
        initialization_bundle=START_FROM_BUNDLE, trust_packages=TRUST_START_PACKAGE)
    print('Best validation-selected model:', TRAIN_REPORT.get('best_agent'))
    print('Final model:', TRAIN_REPORT['last_agent'])
else:
    print('PvE training is OFF. Set RUN_PVE_TRAINING=True after Section 4.')


### 5.1 Review PvE results — no learning
Enable `RUN_TRAINING_REVIEW` after a completed run. Leave `REVIEW_RUN=''` for the latest completed PvE run in this project, or paste another run-directory path.

Compare the validation win fraction with the training-return curve. A rising reward curve alone does not show stronger play. The review also displays recent training outcomes and the best-checkpoint selection record.


In [ ]:
RUN_TRAINING_REVIEW = False
REVIEW_RUN = ''
if RUN_TRAINING_REVIEW:
    directory = REVIEW_RUN or last_training(ROOT, kind='pve')['directory']
    show_training_review(directory)
else:
    print('Run review is OFF.')


## 6. Evaluate an agent against the CPU — no learning
Set `RUN_EVALUATION=True`. An empty `EVALUATION_BUNDLE` selects the latest completed PvE run's **best** agent; in the current implementation it falls back to **last** when no best exists. To avoid ambiguity, paste an exact agent path when comparing experiments.

For an explicit package path, review the package and enable `TRUST_EVALUATION_PACKAGE`. The package must be a classroom **v3 agent ZIP**—not raw `.pth`, raw SB3 `policy.zip`, an outer submission ZIP, or an old v1/v2 agent.

The default includes random actions and, for the automatically selected local run, its initial policy. `deterministic=True` chooses the policy's preferred action; `compare_sampling=True` adds a sampled-policy comparison. None of these evaluations updates weights.

**Expected output:** `EVALUATION_COMPLETE`, win/loss/draw/cutoff counts, HP margins and videos. The ten offsets are selected starting scenarios within the training jitter range, not ten independent opponents or a held-out generalization test.


In [ ]:
RUN_EVALUATION = False
EVALUATION_BUNDLE = ''
TRUST_EVALUATION_PACKAGE = False
EVALUATION_PVE = dict(PVE_CONFIG)
EVALUATION_CONFIG = {
    'offsets': [1, 7, 13, 19, 25, 31, 37, 43, 49, 55],
    'deterministic': True,
    'compare_sampling': False,
    'include_random': True,
    'record_video': True,
}
if RUN_EVALUATION:
    evaluation = dict(EVALUATION_CONFIG)
    if EVALUATION_BUNDLE:
        evaluation['bundle'] = EVALUATION_BUNDLE
        trusted = TRUST_EVALUATION_PACKAGE
    else:
        saved = last_training(ROOT, kind='pve')
        evaluation['bundle'] = saved.get('best_agent') or saved['last_agent']
        evaluation['initial_bundle'] = saved.get('initial_agent')
        trusted = True  # Own packages produced by this notebook.
    LAST_EVALUATION, EVALUATION_REPORT = run_task(ROOT, 'evaluate', PVE_AGENT_CONFIG,
        COMMON_RULES, PVE_TRAIN_CONFIG, EVALUATION_PVE, evaluation=evaluation, trust_packages=trusted)
    show_result(LAST_EVALUATION, EVALUATION_REPORT)
else:
    print('Independent CPU evaluation is OFF.')


## 7. Package ONE selected agent for the practice arena
Choose one saved model, enter your `team_id`, and set `RUN_PACKAGE_SUBMISSION=True`.

`source='pve'` or `'pvp'` selects that profile's latest completed run; `checkpoint='best'` or `'last'` chooses the file explicitly. `source='path'` uses the exact `bundle` path. Unlike Section 6, this packager does **not** silently fall back when best is missing.

**Output:** a `..._submission.zip` containing one inner `agent.zip`, metadata and static-check results, plus an external `.sha256` file. `SUBMISSION_PACKAGED` means the container passed static checks, not that the model has played a game. Continue to Section 7.2.

| File | Where it belongs |
|---|---|
| `best_agent.zip`, `last_agent.zip`, or another complete v3 agent | Evaluation, raw-agent match, or learner/opponent package fields |
| `..._submission.zip` | Section 7.2 or the standalone public practice arena |
| `..._handoff.zip` | Personal experiment backup; not a match input |
| `interrupted_policy.zip` / raw `.pth` | Developer recovery only; not accepted as a classroom agent |

Follow the instructor's confirmed submission requirements; this practice format does not replace them automatically.


In [ ]:
RUN_PACKAGE_SUBMISSION = False
SUBMISSION_CONFIG = {
    'team_id': 'Group_01',
    'display_name': '',
    'source': 'pve',       # 'pve', 'pvp', or 'path'
    'checkpoint': 'best',  # 'best' or 'last' for pve/pvp
    'bundle': '',          # Exact saved v3 agent ZIP when source='path'
    'notes': '',
}
if RUN_PACKAGE_SUBMISSION:
    from sf2_submission import prepare_submission
    from IPython.display import display, FileLink
    LAST_SUBMISSION = prepare_submission(ROOT,
        rules_path=ROOT/'arena_rules_v1.json', **SUBMISSION_CONFIG)
    display(FileLink(str(LAST_SUBMISSION)))
    display(FileLink(str(LAST_SUBMISSION.with_suffix('.sha256'))))
else:
    print('Submission packaging is OFF. Choose one saved model, then enable.')


### 7.1 Optional model picker — an alternative to Section 7
Enable `SHOW_SUBMISSION_PICKER`, select a saved model (or enter its exact v3 agent path), enter your team ID, and click **Package selected model**. You do not need to package it a second time in Section 7.

In this guide revision, a successful button click updates `LAST_SUBMISSION`, so Section 7.2 selects the newly packaged ZIP. A failed click does not count as packaging. Always check the printed filename and the self-check target before proceeding.


In [ ]:
SHOW_SUBMISSION_PICKER = False
if SHOW_SUBMISSION_PICKER:
    from sf2_submission import submission_widget

    def _remember_packaged_submission(path):
        # Both packaging routes now update the same self-check selection.
        global LAST_SUBMISSION
        LAST_SUBMISSION = Path(path).resolve()
        print('Selected for Section 7.2:', LAST_SUBMISSION)

    submission_widget(ROOT, ROOT/'arena_rules_v1.json',
                      on_packaged=_remember_packaged_submission)
else:
    print('Optional submission picker is OFF.')


### 7.2 Self-check the packaged ZIP — no learning
Set `RUN_SUBMISSION_SELF_CHECK=True`. Review the executable package before setting `TRUST_SUBMISSIONS=True`.

Leave `SUBMISSION_TO_CHECK=''` to use the last successful Section 7 / 7.1 package in this session. For a specific experiment or an uploaded file, enter its exact **outer submission ZIP** path. Check the printed `Self-check target` before reading the results.

The self-check runs two copies of the model, tests both controller ports and writes an external receipt and diagnostics. `ARENA_SELF_CHECK_PASSED` is an interface result, not evidence of skill or security. Keep the exact ZIP together with its receipt.

For the final compatibility check, use the separate public arena notebook in a clean Kaggle session with the game resource and this same ZIP. Process isolation and hashes do not make arbitrary submitted Python safe.


In [ ]:
RUN_SUBMISSION_SELF_CHECK = False
TRUST_SUBMISSIONS = False
SUBMISSION_TO_CHECK = ''  # Exact submission ZIP; empty uses LAST_SUBMISSION from Section 7.
if RUN_SUBMISSION_SELF_CHECK:
    from sf2_arena_notebook import run_arena, show_arena_result
    selected = SUBMISSION_TO_CHECK or globals().get('LAST_SUBMISSION', '')
    if not selected:
        raise ValueError('Package one model first or fill SUBMISSION_TO_CHECK.')
    print('Self-check target:', Path(selected).expanduser().resolve())
    LAST_ARENA_CHECK, ARENA_CHECK_REPORT = run_arena(ROOT,
        submission_a=str(selected), mode='self_check', trust_submissions=TRUST_SUBMISSIONS)
    show_arena_result(LAST_ARENA_CHECK, ARENA_CHECK_REPORT)
else:
    print('Submission self-check is OFF.')


### 7.3 Optional backup — keep your experiment, not just one entry
Set `RUN_EXPORT=True` and choose `EXPORT_KIND='pve'` or `'pvp'`. This exports the selected profile's latest completed run: best/last agent links and a handoff archive with source snapshots and reports.

Files are created under `/kaggle/working/sf2_classroom_rc3/`. Download and retain the desired artifacts before discarding your session. A filesystem path is not itself a browser download URL; when an inline link does not work, locate the same file in Kaggle's **Output** pane. Save and check your output artifacts explicitly.

**Do not select Save & Run All merely to download a model:** that option starts a clean top-to-bottom run and can launch enabled training tasks again. Keep a record of the exact package used for each evaluation.


In [ ]:
RUN_EXPORT = False
EXPORT_KIND = 'pve'  # 'pve' or 'pvp'; these have separate latest-run records.
if RUN_EXPORT:
    from IPython.display import display, FileLink
    saved = last_training(ROOT, kind=EXPORT_KIND)
    for name in ('best_agent', 'last_agent'):
        path = saved.get(name)
        if path and Path(path).is_file():
            print(name + ':', path)
            display(FileLink(path))
    handoff = make_handoff(saved['directory'])
    display(FileLink(str(handoff)))
else:
    print('Export is OFF.')


## 8. Match two saved agents — NO training
Run Section 1, fill **both** `agent_a` and `agent_b` with complete v3 agent ZIP paths, review the packages, then enable `TRUST_MATCH_PACKAGES` and `RUN_MATCH`.

`swap_sides=True` runs both controller-port assignments. `rounds_per_side=1` gives one round per assignment. The same package may be entered twice for a self-match. Different agents may use different networks, observations and action tables, but their shared game rules must agree.

**This section never improves either model.** It does not invoke PPO learning, and its swap switch does not affect Section 9. A repeated deterministic match from the same cached start may replay identically; more repeats are not automatically independent evidence.

This is the raw-agent playground. Use Section 7.2 / the public arena for testing an outer submission ZIP.


In [ ]:
RUN_MATCH = False
TRUST_MATCH_PACKAGES = False
MATCH_CONFIG = {
    'agent_a': '',  # Actual v3 package path.
    'agent_b': '',  # Same as A is allowed for an interface check.
    'swap_sides': True,
    'rounds_per_side': 1,
    'deterministic': True,
}
if RUN_MATCH:
    if not MATCH_CONFIG['agent_a'] or not MATCH_CONFIG['agent_b']:
        raise ValueError('Fill both agent_a and agent_b. Use the same path twice for a self-match.')
    LAST_MATCH, MATCH_REPORT = run_task(ROOT, 'pvp', {}, COMMON_RULES,
        {'torch_threads': 2}, pvp=MATCH_CONFIG, trust_packages=TRUST_MATCH_PACKAGES)
    show_result(LAST_MATCH, MATCH_REPORT)
else:
    print('Arena match is OFF. This section never updates a model.')


## 9. Optional PvP training — one learner, one fixed opponent
Here the learner plays against a **saved model**, not the game's CPU. Only the learner is updated. The opponent stays fixed throughout this training phase; this is not automatic self-play with an evolving opponent.

You may run this route directly after Section 1 with existing reviewed v3 packages; no new PvE run is required. Complete 9.1, all four 9.2 cells, and 9.3 before enabling 9.4.

### 9.1 Select the learner and opponent
| Field | Meaning |
|---|---|
| `learner_start='package'` | Start a new learning phase from your matching saved agent; fill `learner_bundle`. |
| `learner_start='scratch'` | Start a new random learner; leave `learner_bundle=''`. |
| `opponent_bundle` | The fixed opponent package. Required in both modes. |
| `learner_sides=[2]` | Learn on controller port P2. P2 does not always remain on the screen's right. |
| `learner_sides=[1,2]` | Randomly choose a learner port each training episode; validation tests both. |

To train your agent against its own frozen snapshot, explicitly enter the same saved package in both bundle fields. They are loaded as separate copies; only the learner changes.

The default configuration intentionally leaves package paths empty and will not run until you fill them. `PVP_TRAIN_CONFIG` is independent of PvE. Its **8,192-step** default is a small wiring/learning smoke run, not evidence of adequate training or guaranteed improvement.

**Character IDs:** `Ryu`, `EHonda`, `Blanka`, `Guile`, `Ken`, `ChunLi`,
`Zangief`, `Dhalsim`, `MBison`, `Sagat`, `Balrog`, `Vega`.
Use the exact spellings shown in the comments. This is the code/practice roster,
not proof that every matchup has been tested or permission to ignore course restrictions.

**Saved models keep their character identity.** When starting from a package, the learner character must match that package. The fixed opponent uses its own package character. To choose a different learner character, use scratch mode with an empty learner bundle, then repeat the PvP check.


In [ ]:
# AVAILABLE CHARACTER IDs — use these exact spellings (no spaces or punctuation):
# Ryu, EHonda, Blanka, Guile, Ken, ChunLi,
# Zangief, Dhalsim, MBison, Sagat, Balrog, Vega.
# These are the implemented/public-practice IDs, not a claim that all matchups
# have passed real-game tests. Follow any stricter rules set by your instructor.
# PvP: this character belongs to the LEARNER, not the fixed opponent.
# In package mode it must match learner_bundle; choose another character in scratch mode.
PVP_AGENT_CONFIG = {
    'team_name': 'group_demo_pvp',
    'character': 'ChunLi',  # LEARNER character; must match learner_bundle in package mode.
    'seed': 42,
    'use_position': True,
}
PVP_SETUP = {
    'learner_start': 'package',  # 'package' or 'scratch'
    'learner_bundle': '',       # Model to continue learning. Required in package mode.
    'opponent_bundle': '',      # Fixed opponent; its character comes from this package.
    'learner_sides': [2],       # Always P2. [1,2] chooses a port each episode.
    'opponent_deterministic': True,
    'neutral_jitter_max': 60,
}
TRUST_PVP_TRAINING_PACKAGES = False  # Enable only for reviewed, trusted packages.

# Independent literal settings, NOT dict(PVE_TRAIN_CONFIG, ...).
PVP_TRAIN_CONFIG = {
    'total_timesteps': 8_192,
    'n_envs': 2,
    'n_steps': 512,
    'batch_size': 512,
    'n_epochs': 4,
    'learning_rate': 0.00025,
    'gamma': 0.94,
    'gae_lambda': 0.95,
    'clip_range': 0.15,
    'ent_coef': 0.0,
    'vf_coef': 0.5,
    'target_kl': None,
    'device': 'auto',
    'torch_threads': 2,
    'checkpoint_every': 4_096,
    'evaluate_every': 4_096,
    'evaluate_initial': True,
    'evaluate_final': True,
    'resume_from': None,
}
PVP_VALIDATION_CONFIG = {
    'offsets': [0, 12, 24, 36, 48],
    'seed': 2025,
    'deterministic': True,  # Learner selection during validation only.
    'record_video': False,
}
print('PvP learner:', PVP_AGENT_CONFIG)
print('PvP setup:', PVP_SETUP)
print('PvP PPO settings:', PVP_TRAIN_CONFIG)
print('Validation: selected frozen model, selected learner ports. No native-CPU fallback.')


### 9.2 PvP student files — run all four cells
These write to `student/pvp/`, not `student/pve/`. Changes to the PvE editing cells do not automatically appear here. `%%writefile` overwrites the target file, so keep your intended edits in these notebook cells.

For `learner_start='package'`, preserve matching observation/model source and action definitions. Even comment-only source edits can fail the exact-source compatibility check. The fixed opponent always uses its own packaged definitions.

Change the PvP reward in 9.2c and the network/PPO construction in 9.2d. A new architecture, input or action design normally requires `learner_start='scratch'`. This release supports ordinary PPO packages, not a generic DQN/SAC loader.

#### 9.2a Observation — independent PvP input definition
The default remains four grayscale decision-time frames plus optional self-first positions. See Section 3.1 for shape/crop semantics; running Section 3 is not required for this route.


In [ ]:
%%writefile $PVP_STUDENT_DIR/student_observation.py
"""Student-editable image input. Default: four 192 x 168 grayscale frames.

Only transform the supplied RGB image. Never operate the game or change rewards.
The coordinate branch still uses the six original self-first position values.
SB3 divides these uint8 images by 255 once, inside its policy preprocessor.
"""
import cv2
import numpy as np

INPUT_CONFIG = {'color': 'gray', 'width': 192, 'height': 168, 'stack_frames': 4, 'raw_shape': [224, 320, 3], 'crop_xyxy': [0, 0, 256, 224], 'verify_right_padding': True, 'black_threshold': 5, 'max_nonblack_fraction': 0.005}


def preprocess(rgb):
    """One RGB uint8 frame -> (channels, height, width) uint8 array."""
    a = np.asarray(rgb)
    c = INPUT_CONFIG
    if a.dtype != np.uint8 or tuple(a.shape) != tuple(c["raw_shape"]):
        raise ValueError(f"Unexpected raw image: {a.shape} {a.dtype}; expected {c['raw_shape']} uint8.")
    x0, y0, x1, y1 = c["crop_xyxy"]
    if c["verify_right_padding"] and x1 < a.shape[1]:
        excluded = a[:, x1:, :]
        fraction = float(np.mean(np.max(excluded, axis=2) > c["black_threshold"]))
        if fraction > c["max_nonblack_fraction"]:
            raise ValueError(
                f"CROP_GUARD_FAILED: right-side nonblack fraction={fraction:.6f}. "
                "The fixed viewport is not valid for this observed frame. "
                "Stop and inspect the raw image; do not train with a silently changing crop."
            )
    view = a[y0:y1, x0:x1, :]
    if c["color"] == "gray":
        gray = cv2.cvtColor(view, cv2.COLOR_RGB2GRAY)
        result = cv2.resize(gray, (c["width"], c["height"]), interpolation=cv2.INTER_AREA)[None, :, :]
    else:
        result = cv2.resize(view, (c["width"], c["height"]), interpolation=cv2.INTER_AREA).transpose(2, 0, 1)
    return np.ascontiguousarray(result, dtype=np.uint8)


#### 9.2b Actions — independent PvP action table
One table entry controls exactly six game frames. Consecutive identical button actions hold the key continuously. Keep the table compatible with a loaded learner; the opponent has its own separate table.


In [ ]:
%%writefile $PVP_STUDENT_DIR/student_actions.py
"""Default: 15 basic actions, as in the old course without its two Ryu specials.
A basic key is held for six emulator frames; consecutive C means continuous C.
Use NO_OP (or another action) to release it. This is not automatic repeated tapping.
Students may append short combinations/sequences totaling exactly six frames.
No complete fighting strategy, auto-facing or long charge macro is provided.
"""
CUSTOM_ACTIONS = []

# Optional student exercise (OFF by default): a short, explicit press/release.
# CUSTOM_ACTIONS = [
#     {'name': 'tap_C', 'segments': [
#         {'keys': ['C'], 'frames': 3}, {'keys': [], 'frames': 3}]},
# ]


#### 9.2c Reward — measured from the learner's perspective
The same reward template is used, but `damage_dealt`, `damage_taken`, win and loss refer to whichever port the learner controls. Do not hard-code P1 as “self” when training P2.


In [ ]:
%%writefile $PVP_STUDENT_DIR/student_reward.py
"""Simple classroom reward: small net-damage feedback + clear match outcome.

Compatible with SF2_Flexible_Classroom_PPO (ordinary PPO, six-frame steps).
The arena supplies damage_dealt/damage_taken with negative KO HP treated as 0.
A natural timeout uses the arena's win/loss/draw result, just like a KO.
A safety cutoff is not a loss. No distance, survival, action or time bonus.

With <=176 HP per fighter and no in-round healing, the sum of damage rewards
is bounded by [-0.5, 0.5]. A normal completed win therefore has positive raw
episode return, and a loss negative raw episode return. This is NOT a claim
about discounted returns, win rate, or empirically verified learning success.
"""
import math

REWARD_VERSION = "net_damage_outcome_v1"
FULL_HP = 176.0
DAMAGE_WEIGHT = 0.5
OUTCOME_BONUS = {
    "ongoing": 0.0,
    "win": 1.0,
    "loss": -1.0,
    "draw": 0.0,
    "cutoff": 0.0,
}


def compute_reward(t: dict) -> float:
    dealt = float(t["damage_dealt"])
    taken = float(t["damage_taken"])
    result = t["result"]

    # Do not silently replace invalid readings with a plausible reward.
    if not (math.isfinite(dealt) and math.isfinite(taken)):
        raise ValueError("Damage must be finite.")
    if not (0.0 <= dealt <= FULL_HP and 0.0 <= taken <= FULL_HP):
        raise ValueError("Damage is outside the expected 0..176 range.")
    if result not in OUTCOME_BONUS:
        raise ValueError(f"Unknown match result: {result!r}")

    # Count this step's damage even on the final step, then add outcome once.
    damage_reward = DAMAGE_WEIGHT * (dealt - taken) / FULL_HP
    return float(damage_reward + OUTCOME_BONUS[result])


#### 9.2d Model — independent PvP learner
This is the learner's PPO/model definition, not the opponent's. Leave it unchanged for a matching saved learner. The template does not update the opponent or share a trainable policy between the two players.


In [ ]:
%%writefile $PVP_STUDENT_DIR/student_model.py
"""Ordinary SB3 PPO + Nature-style CNN + a minimal optional position branch.
No MaskablePPO, control vector, frame-difference images or hand-crafted geometry.
With use_position=False, use the old standard CnnPolicy (NatureCNN) directly.
"""
import torch
from torch import nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

MODEL_CONFIG = {'image_features':512, 'position_features':64}

class ImagePositionExtractor(BaseFeaturesExtractor):
    def __init__(self,observation_space,image_features=512,position_features=64):
        super().__init__(observation_space,features_dim=image_features+position_features)
        # The three convolution sizes/strides match SB3 NatureCNN.
        cnn=nn.Sequential(
            nn.Conv2d(observation_space['image'].shape[0],32,8,4),nn.ReLU(),
            nn.Conv2d(32,64,4,2),nn.ReLU(),
            nn.Conv2d(64,64,3,1),nn.ReLU(),nn.Flatten())
        with torch.no_grad():
            flattened=cnn(torch.zeros(1,*observation_space['image'].shape)).shape[1]
        self.image_net=nn.Sequential(cnn,nn.Linear(flattened,image_features),nn.ReLU())
        self.position_net=nn.Sequential(nn.Linear(6,position_features),nn.ReLU())
    def forward(self,obs):
        # SB3 normalizes the uint8 image once; do not divide by 255 again here.
        return torch.cat((self.image_net(obs['image']),self.position_net(obs['position'])),dim=1)

def build_model(env,config,seed,device='auto'):
    has_position=hasattr(env.observation_space,'spaces')
    policy='MultiInputPolicy' if has_position else 'CnnPolicy'
    kwargs={}
    if has_position:
        kwargs={'features_extractor_class':ImagePositionExtractor,
                'features_extractor_kwargs':dict(MODEL_CONFIG),
                'net_arch':dict(pi=[],vf=[])}
    return PPO(policy,env,
        learning_rate=float(config['learning_rate']),n_steps=int(config['n_steps']),
        batch_size=int(config['batch_size']),n_epochs=int(config['n_epochs']),
        gamma=float(config['gamma']),gae_lambda=float(config.get('gae_lambda',0.95)),
        clip_range=float(config['clip_range']),ent_coef=float(config.get('ent_coef',0.0)),
        vf_coef=float(config.get('vf_coef',0.5)),max_grad_norm=0.5,
        target_kl=config.get('target_kl'),seed=int(seed),device=device,verbose=1,
        policy_kwargs=kwargs)


### 9.3 PvP environment check — required before PvP training
Fill the paths in 9.1, run all four 9.2 cells, review the packages, enable `TRUST_PVP_TRAINING_PACKAGES`, then set `RUN_PVP_CHECK=True`.

Expect `FROZEN_ENVIRONMENT_CHECK_PASSED`. Inspect each requested port, the actual model inputs and videos. This check verifies wiring and unchanged weights, not successful learning.

Repeat it after changes to PvP source, packages, learner ports, PPO settings or shared code/rules. A passed PvE check cannot replace this check.


In [ ]:
RUN_PVP_CHECK = False
if RUN_PVP_CHECK:
    start, opponent = resolve_pvp_setup(PVP_SETUP)
    LAST_PVP_CHECK, PVP_CHECK_REPORT = run_pvp_task(
        ROOT, 'check', PVP_AGENT_CONFIG, COMMON_RULES, PVP_TRAIN_CONFIG, opponent,
        initialization_bundle=start, trust_packages=TRUST_PVP_TRAINING_PACKAGES)
    show_result(LAST_PVP_CHECK, PVP_CHECK_REPORT)
else:
    print('PvP check is OFF. Fill the two paths, review trust, then enable RUN_PVP_CHECK.')


### 9.4 PvP training — only the learner changes
After 9.3 passes, set `RUN_PVP_TRAINING=True`. The learner uses the selected device; each CPU game worker holds a frozen opponent copy. PvP can be slower than native-CPU PvE because the opponent also runs neural-network inference.

Initial, periodic and final validation use the same selected frozen opponent and learner ports. Best is selected by **PvP win fraction**, then learner HP margin—not by CPU performance or raw training reward.

With `evaluate_initial=True`, step zero can remain best when later training does not improve it. Inspect both `best_agent.zip` and `last_agent.zip`, and confirm the report says the opponent stayed unchanged. Interruption/package restrictions are the same as Section 5.


In [ ]:
RUN_PVP_TRAINING = False
if RUN_PVP_TRAINING:
    start, opponent = resolve_pvp_setup(PVP_SETUP)
    LAST_PVP_TRAINING, PVP_TRAIN_REPORT = run_pvp_task(
        ROOT, 'train', PVP_AGENT_CONFIG, COMMON_RULES, PVP_TRAIN_CONFIG, opponent,
        initialization_bundle=start, trust_packages=TRUST_PVP_TRAINING_PACKAGES,
        pvp_validation=PVP_VALIDATION_CONFIG)
    print('PvP best:', PVP_TRAIN_REPORT.get('best_agent'))
    print('PvP last:', PVP_TRAIN_REPORT['last_agent'])
    print('Selection target:', PVP_TRAIN_REPORT['validation_target'])
    print('Learner ports:', PVP_TRAIN_REPORT['learner_sides'])
    print('Opponent unchanged:', PVP_TRAIN_REPORT['frozen_opponent_unchanged'])
else:
    print('PvP training is OFF. It has its own model files and does not run PvE validation.')


### 9.5 Evaluate and review PvP training — no learning
Set `RUN_PVP_EVALUATION=True` to compare the phase-start learner with the final learner against the currently selected opponent. The empty bundle field selects the latest completed **PvP last** agent, deliberately not best, so a step-zero best cannot conceal regression.

For an explicit learner path, enter a matching phase-start reference only when you have one. Results are reported separately for each requested learner port. Keep the opponent, rules and scenarios fixed when interpreting before/after results.

`RUN_PVP_REVIEW` displays the training/validation history. To assess retained CPU performance, run Section 6 separately with the PvP agent path; that test does not determine the PvP best checkpoint.


In [ ]:
RUN_PVP_EVALUATION = False
PVP_EVALUATION_BUNDLE = ''  # Empty = this project's latest PvP LAST model, never latest PvE.
PVP_REFERENCE_BUNDLE = ''   # For a supplied external model, optionally supply its phase-start model.
if RUN_PVP_EVALUATION:
    # Opponent selection is independent of how the learner was originally initialized.
    from sf2_release import validate_frozen_config
    opponent = validate_frozen_config({
        'bundle': PVP_SETUP['opponent_bundle'],
        'learner_sides': PVP_SETUP['learner_sides'],
        'opponent_deterministic': PVP_SETUP['opponent_deterministic'],
        'neutral_jitter_max': PVP_SETUP['neutral_jitter_max'],
    })
    evaluation = dict(PVP_VALIDATION_CONFIG, offsets=[1, 13, 25, 37, 49], record_video=True)
    if PVP_EVALUATION_BUNDLE:
        evaluation.update(bundle=PVP_EVALUATION_BUNDLE,
            initial_bundle=PVP_REFERENCE_BUNDLE, compare_initial=bool(PVP_REFERENCE_BUNDLE))
    else:
        saved = last_training(ROOT, kind='pvp')
        evaluation.update(bundle=saved['last_agent'], initial_bundle=saved['initial_agent'], compare_initial=True)
    LAST_PVP_EVALUATION, PVP_EVALUATION_REPORT = run_pvp_task(
        ROOT, 'evaluate', PVP_AGENT_CONFIG, COMMON_RULES, PVP_TRAIN_CONFIG, opponent,
        evaluation=evaluation, trust_packages=TRUST_PVP_TRAINING_PACKAGES)
    show_result(LAST_PVP_EVALUATION, PVP_EVALUATION_REPORT)
    print('Per learner port:', json.dumps(PVP_EVALUATION_REPORT['per_port'], indent=2))
else:
    print('PvP evaluation is OFF.')


In [ ]:
RUN_PVP_REVIEW = False
PVP_REVIEW_RUN = ''
if RUN_PVP_REVIEW:
    directory = PVP_REVIEW_RUN or last_training(ROOT, kind='pvp')['directory']
    show_training_review(directory)
else:
    print('PvP run review is OFF.')


## 10. Troubleshooting and instructor checks

| Message or symptom | What to do |
|---|---|
| `NameError` for `ROOT`, configs or helper functions | Run the required setup/configuration cells in the current session. |
| A task prints `OFF` | Its `RUN_*` switch is False. Enable only the task you intend to run. |
| `Run Section 4` / `Run Section 9.3` | Rerun the appropriate check after saving your current source/configuration. |
| No completed PvE/PvP training | The automatic selector has no successful run for that profile; use an explicit complete v3 package path. |
| Missing learner/opponent path | Fill the required 9.1 paths. In scratch mode only the learner path stays empty. |
| Trust-switch error | Inspect the code/weights package and enable the corresponding trust switch only when appropriate. |
| Source or observation mismatch | Use the exact matching files for that learner, or start a compatible new learner. A comment edit can change the source hash. |
| Unsupported ZIP / missing files | Distinguish the inner agent, outer submission, handoff backup and raw SB3 checkpoint. |
| `CROP_GUARD_FAILED` | Inspect the game image. Do not silently disable the guard to hide cropped content. |
| CUDA out of memory or very slow training | Record the actual device/configuration. Try a smaller valid batch or fewer workers in a separate experiment; rerun the relevant check. |
| Inline download does not start | Locate the exact artifact in Kaggle Output. Do not rerun training just to regenerate a link. |
| Training finished but the agent plays poorly | Review evaluation counts, videos and action use; completion is not a strength benchmark. |

### Instructor release checklist
Confirm the authoritative character/timing/submission rules and freeze the tested dependency environment. Test a fresh Kaggle session with the intended game resource, the default PvE route, and every additional matchup students must use. Section 4 checks the first PvE condition only; targeted CPU character setup is experimental in the underlying framework.

Test PvP on both required ports, verify learner updates/opponent immutability, and reload an exported agent. Test the exact outer submission in a fresh public arena session, then test another independently produced compatible agent. Keep the receipt, diagnostics and actual runtime versions.

**Scope of this guide revision:** English usage notes, a model-picker/self-check handoff fix, and a printed self-check target. The eight student-source cells, PPO training code, reward, game rules and learning defaults are unchanged. Local syntax/logic tests do not replace real-emulator, GPU-training or Kaggle-browser acceptance.

Reference documentation: Kaggle Notebooks; IPython `%%writefile`; Stable Baselines3 PPO, policy networks and vector environments. The original RC3 source attribution, draft-rule status and license obligations remain applicable.

**Python 3.13 installation failure from R1:** do not delete the version guard or install 0.9.5 with `--ignore-requires-python`. Use this R2 setup, or install `stable-retro==1.0.1` in the current interpreter before rerunning the original setup cell. This is a backend dependency change, not a training result or an all-characters acceptance test.
